In [2]:
# Libérer le cache avant tout
!pip cache purge
!rm -rf /root/.cache/huggingface/hub
!rm -rf /tmp/*
!df -h  # Vérifier l'espace disponible

Files removed: 0
Filesystem                                                                Size  Used Avail Use% Mounted on
overlay                                                                   7.9T  6.7T  1.2T  85% /
tmpfs                                                                      64M     0   64M   0% /dev
shm                                                                        14G     0   14G   0% /dev/shm
/dev/loop1                                                                 20G  352K   20G   1% /kaggle/lib
192.168.16.2:/data/kagglesdsdata/datasets/9758289/15253674/di2cm37vhum6    60T   43T   18T  71% /kaggle/input/datasets/chaymadallel/mitre-attack-stix
192.168.16.2:/data/kagglesdsdata/datasets/9750971/15241381/di2cm37p7xjm    60T   43T   18T  71% /kaggle/input/private-dataset
192.168.16.2:/data/kagglesdsdata/datasets/10224363/15943555/di37nsmrojwk   60T   43T   18T  71% /kaggle/input/datasets/chaymadallel/cti-faiss-index
/dev/sda1                               

In [3]:
%%capture
!pip install https://download.pytorch.org/whl/cu117/torch-2.0.1%2Bcu117-cp310-cp310-linux_x86_64.whl

In [4]:
!pip install langsmith --quiet

In [5]:
!pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 93.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: openai
    Found 

In [6]:
import os
from kaggle_secrets import UserSecretsClient
from langsmith import Client

fresh_key = UserSecretsClient().get_secret("LANGCHAIN_API_KEY")

os.environ["LANGSMITH_API_KEY"]  = fresh_key
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "cti_rag"

client = Client(api_key=fresh_key)
project = client.read_project(project_name="cti_rag")
print(f"✅ Connected to project: {project.id}")


✅ Connected to project: eb1500e3-380e-4e80-bbff-4349dd678465


In [7]:
!pip install -q transformers langchain langchain-community langchain-huggingface \
              langchain-text-splitters faiss-cpu sentence-transformers bitsandbytes accelerate
print("all is good")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
all is good


In [8]:
!pip install langchain-core

In [9]:
%%writefile config.py
import os

# ── Paths to data ───────────────────────────────────
FAISS_INDEX_PATH = "/kaggle/input/datasets/chaymadallel/cti-faiss-index"
JSONL_DATA_PATH  = "/kaggle/input/datasets/chaymadallel/dataa-1/darkgram_cti_final.jsonl"

# ── LLM & Embedding ──────────────────────────────────
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
TEMPERATURE = 0.1

# RAG & Hybrid Logic
RAG_TOP_K = 3
WEIGHTS = {
    "static":      0.5,   
    "llm":         0.35,   
    "statistical": 0.15,   
}

Writing config.py


In [10]:
%%writefile llm_helper.py
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from config import LLM_MODEL

# ── Singletons: loaded once, reused on every call ────────────────────────────
_llm_instance = None
_tokenizer_instance = None

def get_llm():
    global _llm_instance, _tokenizer_instance

    if _llm_instance is None:  # Only load on first call — expensive operation

        # ── Tokenizer ────────────────────────────────────────────────────────
        _tokenizer_instance = AutoTokenizer.from_pretrained(
            LLM_MODEL,
            trust_remote_code=True  # Qwen has custom code in its repo; this allows it to run
        )

        # ── 4-bit Quantization Config ─────────────────────────────────────────
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,               # Compress weights to 4-bit → ~4GB VRAM instead of 14GB
            bnb_4bit_compute_dtype=torch.bfloat16,  # Do the math in bfloat16 internally for stability
                                                     # (weights stay 4-bit, compute is higher precision)
            bnb_4bit_use_double_quant=True,  # Quantize the quantization constants too → saves ~0.4 extra bits/param
            bnb_4bit_quant_type="nf4"        # NormalFloat4: best quality 4-bit format for LLMs,
                                             # designed specifically for normally-distributed neural net weights
        )

        # ── Model Loading ─────────────────────────────────────────────────────
        model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            quantization_config=quantization_config,  # Apply the 4-bit config above
            device_map="balanced",   # Split layers evenly across both T4 GPUs (vs "auto" which fills GPU 0 first)
            trust_remote_code=True,  # Same as tokenizer — needed for Qwen custom architecture
            low_cpu_mem_usage=True,  # Stream weights from disk instead of loading all into RAM first
                                     # Critical for 7B models to avoid CPU RAM spike during load
        )

        # ── Inference Pipeline ────────────────────────────────────────────────
        _llm_instance = pipeline(
            "text-generation",
            model=model,
            tokenizer=_tokenizer_instance,
            max_new_tokens=2048,       # Max tokens the model can generate per call
                                      # (doesn't count the input prompt, only the output)
            return_full_text=False,   # Return only the generated reply, not prompt + reply
            do_sample=False,          # Greedy decoding: always pick the highest-probability token
                                      # → deterministic, best for structured JSON output
            pad_token_id=_tokenizer_instance.eos_token_id,  # Tells the model what token to use for padding
                                                             # Qwen doesn't define a pad token by default → this silences the warning
            repetition_penalty=1.1,   # Penalize tokens that already appeared in the output
                                      # 1.0 = no penalty, >1.0 = discourages repetition
                                      # 1.1 is a safe value — prevents Qwen 7B from looping
        )

    return _llm_instance, _tokenizer_instance


def extract_json(text: str) -> str:
    # Find the first {...} block in the model output using regex
    # re.DOTALL makes '.' match newlines too — needed for multi-line JSON
    match = re.search(r"(\{.*\})", text, re.DOTALL)
    return match.group(1) if match else text  # Return JSON block if found, else return raw text


def llm_analyze(system_prompt: str, user_content: str) -> dict:
    llm, tokenizer = get_llm()

    # Build the chat message structure Qwen expects
    messages = [
        {"role": "system", "content": system_prompt},  # Instructions / persona for the model
        {"role": "user", "content": user_content}       # The actual input to analyze
    ]

    # Convert messages to a single string using Qwen's chat template
    # add_generation_prompt=True appends the "<|im_start|>assistant" token
    # so the model knows it should start generating a reply
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        results = llm(prompt)
        output = results[0]["generated_text"].strip()

        print(f"\n--- QWEN RESPONSE ---\n{output}\n-------------------")

        # Extract JSON from output and parse it into a Python dict
        return json.loads(extract_json(output))

    except Exception as e:
        # If anything fails (bad JSON, model error, etc.), return a safe fallback dict
        return {"error": str(e), "llm_failed": True}


Writing llm_helper.py


In [11]:
# from llm_helper import llm_analyze

# print("Checking LLM Life...")
# res = llm_analyze("Respond with a JSON containing the reponse of 'hello who are you and what's your name ? Are you expert in CyberSecurity? Can you be the best llm for this project Multi agnet cti with rag and help in all the pipline!'", "test")
# print("Result:", res)

In [12]:
%%writefile extract_entities.py
"""
TOOL 1 — Entity Extraction (HYBRID)
====================================
Static + LLM IOC extraction pipeline (Kaggle-safe version)
"""

import re
from typing import Any
from langchain_core.tools import tool
from llm_helper import llm_analyze


# =============================================================================
# REGEX PATTERNS
# =============================================================================

IP_PATTERN = re.compile(r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}(?!\.\d)\b")
# The (?<![\.,;:!]) ensures the match doesn't end with a period, comma, colon, etc.
URL_PATTERN = re.compile(r"https?://[^\s<>\"'\)]+(?<![\.,;:!])")

DOMAIN_PATTERN = re.compile(
    r"\b(?:[a-zA-Z0-9-]+\.)+(?:com|net|org|ru|io|xyz|top|cc|"
    r"tk|ml|ga|cf|info|biz|onion|gov|edu|co|me)\b"
)

BTC_PATTERN = re.compile(r"\b(?:1|3)[A-HJ-NP-Za-km-z1-9]{25,39}\b|bc1[a-zA-HJ-NP-Z0-9]{25,87}\b")
ETH_PATTERN = re.compile(r"\b0x[a-fA-F0-9]{40}\b")
XMR_PATTERN = re.compile(r"\b4[0-9AB][1-9A-HJ-NP-Za-km-z]{93}\b")

SHA256_PATTERN = re.compile(r"\b[a-fA-F0-9]{64}\b")
SHA1_PATTERN   = re.compile(r"\b[a-fA-F0-9]{40}\b")
MD5_PATTERN    = re.compile(r"\b[a-fA-F0-9]{32}\b")

EMAIL_PATTERN = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")
CVE_PATTERN   = re.compile(r"\bCVE-\d{4}-\d{4,}\b", re.IGNORECASE)


# =============================================================================
# THREAT SEEDS
# =============================================================================

THREAT_INTEL_SEEDS = {
    "malware_families": {"emotet", "trickbot", "qakbot", "redline", "lumma stealer"},
    "ransomware_groups": {"lockbit", "conti", "play", "clop"},
    "apt_groups": {"apt28", "apt29", "lazarus", "apt41"},
    "abused_tools": {"mimikatz", "cobalt strike", "metasploit", "rclone"},
}

TOOL_BLACKLIST = {"powershell", "cmd", "mshta", "rundll32", "vssadmin"}


# =============================================================================
# HELPERS
# =============================================================================

def _deduplicate(items: list) -> list:
    seen = set()
    out = []
    for i in items:
        if not i:
            continue
        k = i.strip().lower()
        if k not in seen:
            seen.add(k)
            out.append(i.strip())
    return out


def _find_seeds(seeds: set, text: str) -> list:
    return [s for s in seeds if re.search(rf"\b{re.escape(s)}\b", text)]


def _is_valid_ipv4(ip: str) -> bool:
    parts = ip.split(".")
    return len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts)


# =============================================================================
# STATIC EXTRACTION
# =============================================================================

def _static_extraction(text: str) -> dict:
    sha256 = _deduplicate(SHA256_PATTERN.findall(text))
    sha1   = _deduplicate([h for h in SHA1_PATTERN.findall(text) if h not in sha256])
    md5    = _deduplicate([h for h in MD5_PATTERN.findall(text) if h not in sha256 + sha1])

    urls    = _deduplicate(URL_PATTERN.findall(text))
    domains = _deduplicate(DOMAIN_PATTERN.findall(text))
    domains = [d for d in domains if not any(d in u for u in urls)]

    low = text.lower()

    return {
        "ips": _deduplicate(IP_PATTERN.findall(text)),
        "urls": urls,
        "domains": domains,

        "crypto_wallets": {
            "btc": _deduplicate(BTC_PATTERN.findall(text)),
            "eth": _deduplicate(ETH_PATTERN.findall(text)),
            "xmr": _deduplicate(XMR_PATTERN.findall(text)),
        },

        "hashes": {
            "md5": md5,
            "sha1": sha1,
            "sha256": sha256,
        },

        "malware_names": _deduplicate(_find_seeds(THREAT_INTEL_SEEDS["malware_families"], low)),
        "ransomware_groups": _deduplicate(_find_seeds(THREAT_INTEL_SEEDS["ransomware_groups"], low)),
        "apt_groups": _deduplicate(_find_seeds(THREAT_INTEL_SEEDS["apt_groups"], low)),
        "tools_abused_static": _deduplicate(_find_seeds(THREAT_INTEL_SEEDS["abused_tools"], low)),

        "emails": _deduplicate(EMAIL_PATTERN.findall(text)),
        "cves": _deduplicate(CVE_PATTERN.findall(text)),
    }


# =============================================================================
# LLM EXTRACTION
# =============================================================================

LLM_PROMPT = """You are a Cyber Threat Intelligence (CTI) entity extractor.
Analyze the text and extract the following entities. Return ONLY valid JSON with
exactly these keys (use empty lists/string when nothing is found):

{
  "new_malware":        ["malware family names NOT already well-known (novel strains)"],
  "threat_actors":      ["APT group names, ransomware operator aliases, threat actor handles"],
  "defanged_iocs":      ["defanged IPs like 1.2.3[.]4, defanged URLs like hxxps://evil[.]com"],
  "campaign_names":     ["named operation or campaign identifiers, e.g. Operation ShadowHammer"],
  "targeted_sectors":   ["industry verticals targeted, e.g. healthcare, finance, energy"],
  "targeted_countries": ["country or region names explicitly mentioned as targets"],
  "tools_abused":       ["legitimate tools abused by attackers, e.g. PsExec, AnyDesk, ngrok"],
  "additional_cves":    ["any CVE IDs not already captured by regex, e.g. CVE-2024-1234"],
  "context_notes":      "one sentence summarising the most important context not captured above"
}

Return ONLY the JSON object. No markdown, no explanation, no preamble."""


def _llm_extraction(text: str) -> dict:
    res = llm_analyze(LLM_PROMPT, text)

    if res.get("llm_failed") or res.get("parse_error"):
        return {
            "new_malware": [],
            "threat_actors": [],
            "defanged_iocs": [],
            "campaign_names": [],
            "targeted_sectors": [],
            "targeted_countries": [],
            "tools_abused": [],
            "additional_cves": [],
            "context_notes": "",
        }

    return res


# =============================================================================
# MERGE ENGINE
# =============================================================================

def _merge_results(static: dict, llm: dict) -> dict:

    malware = _deduplicate(static.get("malware_names", []) + llm.get("new_malware", []))
    ransomware = static.get("ransomware_groups", [])
    apt = static.get("apt_groups", [])

    tools = _deduplicate(
        static.get("tools_abused_static", []) + llm.get("tools_abused", [])
    )
    tools = [t for t in tools if t.lower() not in TOOL_BLACKLIST]

    extra_ips, extra_urls, extra_domains = [], [], []

    for ioc in llm.get("defanged_iocs", []):

        if isinstance(ioc, dict):
            items = [str(v) for v in ioc.values()]
        elif isinstance(ioc, str):
            items = [ioc]
        else:
            continue

        for item in items:
            clean = (
                item.replace("[.]", ".")
                    .replace("hxxp://", "http://")
                    .replace("hxxps://", "https://")
            )

            ip = IP_PATTERN.search(clean)
            if ip and _is_valid_ipv4(ip.group()):
                extra_ips.append(ip.group())

            elif clean.startswith("http"):
                extra_urls.append(clean)

            elif DOMAIN_PATTERN.search(clean):
                extra_domains.append(clean)

    cves = _deduplicate(static.get("cves", []) + llm.get("additional_cves", []))

    merged = {
        "ips": _deduplicate(static.get("ips", []) + extra_ips),
        "urls": _deduplicate(static.get("urls", []) + extra_urls),
        "domains": _deduplicate(static.get("domains", []) + extra_domains),

        "crypto_wallets": static.get("crypto_wallets", {}),
        "hashes": static.get("hashes", {}),

        "malware_names": malware,
        "ransomware_groups": ransomware,
        "apt_groups": apt,
        "tools_abused": tools,

        "emails": static.get("emails", []),
        "cves": cves,

        "threat_actors": _deduplicate(llm.get("threat_actors", [])),
        "campaign_names": _deduplicate(llm.get("campaign_names", [])),
        "targeted_sectors": llm.get("targeted_sectors", []),
        "targeted_countries": llm.get("targeted_countries", []),
        "context_notes": llm.get("context_notes", ""),
    }

    # ========================
    # IOC COUNTERS
    # ========================

    merged["ioc_count_network"] = (
        len(merged["ips"]) + len(merged["urls"]) + len(merged["domains"]) +
        len(merged["emails"]) + len(merged["cves"]) +
        sum(len(v) for v in merged["crypto_wallets"].values()) +
        sum(len(v) for v in merged["hashes"].values())
    )

    merged["ioc_count"] = merged["ioc_count_network"] + (
        len(merged["malware_names"]) +
        len(merged["ransomware_groups"]) +
        len(merged["apt_groups"]) +
        len(merged["tools_abused"])
    )

    return merged


# =============================================================================
# TOOL ENTRY
# =============================================================================

@tool
def extract_entities(message: str) -> dict[str, Any]:
    """Extract IOCs, malware names, threat actors, CVEs, and all CTI entities
    from a raw threat intelligence message using hybrid static + LLM analysis."""
    static = _static_extraction(message)
    llm = _llm_extraction(message)
    return _merge_results(static, llm)

Writing extract_entities.py


In [13]:
%%writefile rag_search.py
"""
TOOL 2 — RAG Search
====================
Searches a pre-built FAISS vector index of historical Telegram CTI messages
to find past threats similar to the current message being analyzed.

"""

import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
import config

INDEX_PATH = config.FAISS_INDEX_PATH
MODEL_NAME = config.EMBEDDING_MODEL

# Module-level cache so the vectorstore is only loaded once per process
_vectorstore = None


def _get_vectorstore() -> FAISS:
    """
    Lazily load the FAISS vectorstore from disk.
    Subsequent calls return the cached instance without reloading.
    The model runs on CPU to preserve GPU memory for LLM inference.
    """
    global _vectorstore
    if _vectorstore is None:
        embeddings = HuggingFaceEmbeddings(
            model_name=MODEL_NAME,
            model_kwargs={"device": "cpu"},
        )
        _vectorstore = FAISS.load_local(
            INDEX_PATH,
            embeddings,
            allow_dangerous_deserialization=True,
        )
    return _vectorstore


@tool
def rag_search(query: str, k: int = 3) -> str:
    """
    Search the Telegram CTI database for messages similar to the given query.

    Uses L2 (Euclidean) distance — a LOWER score means a CLOSER (more similar) match.

    Args:
        query: Natural-language description or raw IOC text to search for
        k    : Number of top results to return (default: 3)

    Returns:
        JSON string with the query and a list of matched documents,
        each including its similarity score, content snippet, channel, and date.
    """
    vs = _get_vectorstore()

    # similarity_search_with_score returns (Document, float) pairs
    # For FAISS with L2 distance: lower score = more similar
    results_with_scores = vs.similarity_search_with_score(query, k=k)

    formatted_results = []
    for doc, score in results_with_scores:
        formatted_results.append({
            "score":   round(float(score), 4),
            "content": doc.page_content[:500],           # truncate to avoid token overload
            "channel": doc.metadata.get("channel_name", "unknown"),
            "date":    doc.metadata.get("date", ""),
        })

    return json.dumps({"query": query, "results": formatted_results})

Writing rag_search.py


In [14]:
import config
print(f"Current Index Path: {config.FAISS_INDEX_PATH}")

Current Index Path: /kaggle/input/datasets/chaymadallel/cti-faiss-index


In [15]:
%%writefile detect_patterns.py
"""
TOOL 3 — Pattern Detection (HYBRID)
=====================================
Analyzes a batch of CTI messages to identify dominant themes, statistical
anomalies, and semantic threat classifications.

Three layers:
  Layer 1 (Static)      : Frequency counting of known CTI vocabulary
  Layer 2 (Statistical) : Z-score anomaly detection + Shannon entropy
  Layer 3 (LLM)         : Semantic threat classification and emerging terms

No bugs were found in the original — this version adds English comments only.
"""

import re
import math
from collections import Counter
from langchain_core.tools import tool
from llm_helper import llm_analyze


CTI_SHORT_TOKENS = {
    "c2", "c&c", "ip", "vm", "av", "os", "id", "db", "xss", "sql",
    "rce", "lfi", "rfi", "poc", "apt", "rat", "bot", "dns", "tcp",
    "udp", "ssh", "rdp", "smb", "ftp", "vpn", "tor", "ioc", "ttps",
    "edr", "ids", "ips", "waf", "dll", "exe", "cmd", "ps1", "bat",
}


def _tokenize_smart(text: str) -> list[str]:
    """
    Tokenize text into meaningful terms for frequency analysis.

    Keeps compound tokens (e.g. 'cobalt.strike', 'c&c') and CTI short tokens.
    Discards short pure-digit sequences (port numbers, random IDs) that add noise.
    """
    raw_tokens = re.findall(r"[a-zA-Z0-9&][\w&.-]*", text.lower())

    filtered = []
    for token in raw_tokens:
        if token in CTI_SHORT_TOKENS:
            # Always keep known CTI abbreviations regardless of length
            filtered.append(token)
        elif token.isdigit() and len(token) < 4:
            # Skip short numeric strings (e.g. port "80", year "22")
            continue
        else:
            filtered.append(token)

    return filtered


# =============================================================================
# LAYER 1 — STATIC FREQUENCY ANALYSIS
# =============================================================================

CTI_TERMS_SEEDS = {
    "ransomware", "phishing", "exploit", "vulnerability", "cve",
    "botnet", "c2", "payload", "dropper", "loader", "stealer",
    "keylogger", "backdoor", "rootkit", "zero-day", "0day",
    "breach", "leak", "dump", "credentials", "brute", "ddos",
    "injection", "xss", "sqli", "rce", "privilege", "escalation",
    "lateral", "exfiltration", "encryption", "ransom", "bitcoin",
    "monero", "tor", "onion", "obfuscation", "persistence",
    "evasion", "sandbox", "rat", "apt", "campaign", "malware",
    "trojan", "worm", "spyware", "adware", "cryptominer",
}


def _static_frequency_analysis(tokens: list[str]) -> dict:
    """
    Count token frequencies and identify known CTI terms.

    Returns:
        word_counts    : Full Counter of all tokens
        top_keywords   : Top-30 tokens with CTI-term flag
        known_cti_terms: Sorted list of recognized CTI vocabulary hits
    """
    word_counts = Counter(tokens)
    top_30      = word_counts.most_common(30)

    top_keywords = [
        {
            "word":              word,
            "count":             count,
            "is_known_cti_term": word in CTI_TERMS_SEEDS,
        }
        for word, count in top_30
    ]

    # Sort known CTI terms by frequency (most frequent first)
    known_cti_found = sorted(
        [w for w in word_counts if w in CTI_TERMS_SEEDS],
        key=lambda w: word_counts[w],
        reverse=True,
    )

    return {
        "word_counts":     word_counts,
        "top_keywords":    top_keywords,
        "known_cti_terms": known_cti_found,
    }


# =============================================================================
# LAYER 2 — STATISTICAL ANOMALY DETECTION
# =============================================================================

def _statistical_anomaly_detection(word_counts: Counter) -> dict:
    """
    Detect statistically unusual tokens and measure vocabulary diversity.

    Methods:
      Z-score          : Flags tokens whose frequency is > 2 standard deviations
                         above the mean — strong signal of a dominant theme.
      Shannon entropy  : Low entropy → very repetitive (campaign/botnet).
                         High entropy → diverse vocabulary (mixed traffic/noise).
      Concentration    : % of total word mass in the top-10 tokens.
                         High concentration → dominant single theme.

    Returns:
        anomalies          : Up to 10 high-Z-score tokens with interpretation
        entropy            : Normalized Shannon entropy value
        diversity_ratio    : entropy / max_possible_entropy (0.0–1.0)
        concentration_ratio: Share of vocabulary in top-10 tokens
        stats              : Raw statistical parameters
        interpretation     : Human-readable summary
    """
    if not word_counts:
        return {"anomalies": [], "entropy": 0, "concentration_ratio": 0}

    counts = list(word_counts.values())
    total  = sum(counts)
    n      = len(counts)

    # Compute mean and standard deviation for Z-score
    mean     = total / n
    variance = sum((c - mean) ** 2 for c in counts) / n
    std_dev  = math.sqrt(variance) if variance > 0 else 1  # avoid division by zero

    # Flag tokens with Z-score > 2 (statistically unusual frequency)
    anomalies = []
    for word, count in word_counts.items():
        z_score = (count - mean) / std_dev
        if z_score > 2.0:
            anomalies.append({
                "word":           word,
                "count":          count,
                "z_score":        round(z_score, 2),
                "interpretation": (
                    f"'{word}' appears {count}x "
                    f"(mean={mean:.1f}, z={z_score:.1f}) "
                    f"— abnormally high frequency"
                ),
            })
    anomalies.sort(key=lambda a: a["z_score"], reverse=True)

    # Shannon entropy: H = -∑ p(x) log2(p(x))
    entropy = 0.0
    for count in counts:
        if count > 0:
            p = count / total
            entropy -= p * math.log2(p)

    max_entropy     = math.log2(n) if n > 1 else 0
    diversity_ratio = entropy / max_entropy if max_entropy > 0 else 0

    # Concentration: what fraction of total tokens are the top-10?
    top_10_sum  = sum(count for _, count in word_counts.most_common(10))
    concentration = top_10_sum / total if total > 0 else 0

    return {
        "anomalies":          anomalies[:10],
        "entropy":            round(entropy, 3),
        "max_entropy":        round(max_entropy, 3),
        "diversity_ratio":    round(diversity_ratio, 3),
        "concentration_ratio": round(concentration, 3),
        "stats": {
            "mean_frequency": round(mean, 2),
            "std_deviation":  round(std_dev, 2),
            "unique_tokens":  n,
            "total_tokens":   total,
        },
        "interpretation": _interpret_stats(diversity_ratio, concentration),
    }


def _interpret_stats(diversity: float, concentration: float) -> str:
    """Generate a human-readable interpretation of the statistical metrics."""
    parts = []

    if diversity < 0.3:
        parts.append(
            "Very concentrated vocabulary → likely targeted campaign "
            "or highly similar messages (spam/botnet pattern)"
        )
    elif diversity < 0.6:
        parts.append(
            "Moderately diverse vocabulary → possible themed messages "
            "related to the same actor or technique"
        )
    else:
        parts.append(
            "Very diverse vocabulary → varied messages, "
            "no obvious dominant campaign or theme"
        )

    if concentration > 0.5:
        parts.append(
            f"The top-10 most frequent words represent "
            f"{concentration:.0%} of total tokens → strong dominant theme"
        )

    return " | ".join(parts)


# =============================================================================
# LAYER 3 — LLM SEMANTIC CLASSIFICATION
# =============================================================================

LLM_PATTERN_PROMPT = """You are a literal-minded Cyber Threat Intelligence (CTI) Auditor.

### CRITICAL NEUTRALITY MANDATE:
You must distinguish between routine IT work and actual cyber attacks.
- Do NOT search for 'hidden' malicious meanings in standard corporate English.
- Words like 'view', 'scheduled', 'new', 'update', or 'internal' are NOT suspicious alone.
- If the text describes a scheduled reboot, a patch, or an HR portal, it is NOT an attack.

### CLASSIFICATION CATEGORIES (Choose EXACTLY ONE):
1. BENIGN: Internal company news, holiday calendars, general greetings, or HR links.
2. ADMINISTRATIVE: Scheduled reboots, system patching, IT maintenance, or routine tickets.
3. MALICIOUS: ONLY if there is explicit evidence of C2, Exploit payloads, Ransomware,
   or Unauthorized Data Leaks.

### YOUR TASKS:
1. CLASSIFY the primary theme using the categories above.
2. If BENIGN or ADMINISTRATIVE: Set 'is_malicious' to false, 'confidence' to 1.0,
   and 'predicted_next_steps' to ['None' or 'Routine Maintenance'].
3. If MALICIOUS: Identify specific patterns and predict the attacker's next steps.

STRICT JSON SCHEMA — use exact keys 'keywords' and 'attack_pattern'
inside 'correlated_patterns'. Do not use 'pattern' or 'description'.

Return ONLY valid JSON:
{
    "threat_classification": "BENIGN | ADMINISTRATIVE | RANSOMWARE | PHISHING | APT | etc.",
    "is_malicious": false,
    "confidence": 1.0,
    "emerging_terms": [],
    "correlated_patterns": [],
    "predicted_next_steps": ["Step 1", "Step 2"],
    "flagged_jargon": [],
    "semantic_summary": "Literal explanation. If safe, state why."
}"""


def _llm_pattern_analysis(
    top_keywords:    list[dict],
    anomalies:       list[dict],
    sample_messages: list[str],
) -> dict:
    """
    Ask the LLM to semantically classify the dominant threat pattern.

    Provides the LLM with:
      - Top-20 keywords by frequency
      - Statistical anomalies
      - First 3 sample messages (up to 300 chars each)

    Returns the parsed LLM response, or a safe fallback on failure.
    """
    context_parts = [
        "## Top Keywords (by frequency):",
        *[f"  - {kw['word']}: {kw['count']}x" for kw in top_keywords[:20]],
        "",
        "## Statistical Anomalies:",
        *[f"  - {a['interpretation']}" for a in anomalies[:5]],
        "",
        "## Sample Messages (first 3):",
        *[f"  ---\n  {msg[:300]}" for msg in sample_messages[:3]],
    ]
    context = "\n".join(context_parts)
    result  = llm_analyze(LLM_PATTERN_PROMPT, context)

    # Return a neutral fallback if the LLM call or JSON parsing failed
    if "llm_failed" in result or "parse_error" in result:
        return {
            "threat_classification": "unknown",
            "confidence":            0.0,
            "emerging_terms":        [],
            "correlated_patterns":   [],
            "predicted_next_steps":  [],
            "semantic_summary":      "LLM analysis unavailable",
        }

    return result


# =============================================================================
# LANGCHAIN TOOL ENTRY POINT
# =============================================================================

@tool
def detect_patterns(messages: list[str]) -> dict:
    """
    Hybrid pattern analysis across a batch of CTI messages.

    Layer 1 (Static)      : Known CTI term frequencies
    Layer 2 (Statistical) : Z-score anomaly detection + entropy measurement
    Layer 3 (LLM)         : Semantic threat classification and emerging term detection

    Args:
        messages: List of raw CTI message strings to analyze together

    Returns:
        Combined analysis with token statistics, anomalies, and LLM classification
    """
    if not messages:
        return {"error": "No messages provided for analysis."}

    # Step 1: Tokenize all messages together into one corpus
    all_tokens = []
    for msg in messages:
        all_tokens.extend(_tokenize_smart(msg))

    # Step 2: Layer 1 — static frequency analysis
    static = _static_frequency_analysis(all_tokens)

    # Step 3: Layer 2 — statistical anomaly detection
    stats = _statistical_anomaly_detection(static["word_counts"])

    # Step 4: Layer 3 — LLM semantic classification
    llm_analysis = _llm_pattern_analysis(
        top_keywords    = static["top_keywords"],
        anomalies       = stats["anomalies"],
        sample_messages = messages,
    )

    # -------------------------------------------------------------------------
    # SAFETY GATE
    # The LLM can over-classify benign content as MALICIOUS.
    # We only trust a MALICIOUS verdict if there is hard evidence
    # (known CTI terms found) OR at least a statistical anomaly.
    # -------------------------------------------------------------------------
    has_hard_evidence       = len(static["known_cti_terms"]) > 0
    has_statistical_anomaly = len(stats["anomalies"]) > 0
    has_evidence            = has_hard_evidence or has_statistical_anomaly

    classification = llm_analysis.get("threat_classification", "UNKNOWN")
    confidence     = llm_analysis.get("confidence", 0.0)
    is_malicious   = llm_analysis.get("is_malicious", False)

    if not has_hard_evidence and classification == "MALICIOUS":
        if has_statistical_anomaly:
            # Downgrade: statistical signal but no confirmed CTI terms
            classification = "POTENTIALLY SUSPICIOUS (Unconfirmed)"
            confidence    *= 0.7
        else:
            # No evidence at all — override the LLM verdict
            classification = "BENIGN / INFORMATIONAL (No Evidence)"
            confidence    *= 0.5
            is_malicious   = False

    # Only mark as malicious if we have at least some corroborating evidence
    final_is_malicious = is_malicious if has_evidence else False

    return {
        "total_messages": len(messages),
        "is_malicious":   final_is_malicious,
        "unique_tokens":  stats["stats"]["unique_tokens"],

        # Layer 1 results
        "top_keywords":    static["top_keywords"],
        "known_cti_terms": static["known_cti_terms"],

        # Layer 2 results
        "statistical_anomalies": stats["anomalies"],
        "entropy":               stats["entropy"],
        "diversity_ratio":       stats["diversity_ratio"],
        "concentration_ratio":   stats["concentration_ratio"],
        "stats_interpretation":  stats["interpretation"],

        # Layer 3 results
        "threat_classification":     classification,
        "classification_confidence": round(confidence, 2),
        "emerging_terms":            llm_analysis.get("emerging_terms", []),
        "correlated_patterns":       llm_analysis.get("correlated_patterns", []),
        "predicted_next_steps":      llm_analysis.get("predicted_next_steps", []),
        "flagged_jargon":            llm_analysis.get("flagged_jargon", []),
        "semantic_summary":          llm_analysis.get("semantic_summary", ""),
    }


Writing detect_patterns.py


In [16]:
%%writefile detect_behaviors.py
"""
TOOL 4 — Dynamic Behavior Detection (HYBRID)
=============================================
Detects malicious behaviors in raw CTI text using three layers:
  Layer 1 (Static)  : Regex signatures for known MITRE ATT&CK techniques
  Layer 2 (LLM)     : Contextual detection of novel or implied techniques
  Layer 3 (Scoring) : Weighted merge with confidence thresholding

FIXES vs original:
  1. SEVERITY_SCORES was defined twice (module level + inside detect_behaviors()).
     The duplicate inside the function has been removed.
  2. LLM_CATEGORY_MAP was rebuilt on every _merge_behaviors() call.
     Moved to module level as a constant.
  3. filtered_merged intermediate variable simplified — merged directly.
"""

import re
from langchain_core.tools import tool
from llm_helper import llm_analyze
from config import WEIGHTS


# =============================================================================
# LAYER 1 — STATIC BEHAVIOR SIGNATURES (Regex)
# =============================================================================

BEHAVIOR_SIGNATURES = {
    "process_creation": {
        "description": "Suspicious process creation or injection",
        "patterns": [
            r"(?i)process\s*(creation|injection|hollowing)",
            r"(?i)(spawn|execute|launch|run)\s*(cmd|powershell|wscript|mshta|rundll)",
            r"(?i)(CreateProcess|NtCreateThread|WriteProcessMemory)",
            r"(?i)shellcode\s*(inject|execution|load)",
            r"(?i)(?:cmd|powershell|bash)\.exe",
            r"(?i)invoke[- ]expression",
        ],
        "severity": "high",
    },
    "registry_modification": {
        "description": "Windows registry modification for persistence",
        "patterns": [
            r"(?i)reg(istry)?\s*(key|value|modif|add|set|delete|write)",
            r"(?i)(HKLM|HKCU|HKEY_LOCAL_MACHINE|HKEY_CURRENT_USER)",
            r"(?i)\\CurrentVersion\\Run",
            r"(?i)(autorun|startup|persistence)\s*(registry|reg)",
        ],
        "severity": "high",
    },
    "network_communication": {
        "description": "Suspicious network communications (C2, data exfiltration)",
        "patterns": [
            r"(?i)(c2|c&c|command\s*and\s*control)\s*(server|communication|beacon)",
            r"(?i)(beacon|callback|phone\s*home|heartbeat)",
            r"(?i)(exfiltrat|data\s*theft|upload\s*stolen)",
            r"(?i)(reverse\s*shell|bind\s*shell|netcat)",
            r"(?i)tor\s*(network|proxy|hidden|onion)",
        ],
        "severity": "critical",
    },
    "file_system_activity": {
        "description": "Suspicious file activity (encryption, deletion)",
        "patterns": [
            r"(?i)(encrypt|decrypt|cipher)\s*(file|document|data)",
            r"(?i)(ransom\s*note|\.locked|\.encrypted|\.crypt)",
            r"(?i)(drop|write|create)\s*(file|payload|executable|dll)",
            r"(?i)(delete|wipe|shred)\s*(log|shadow|backup)",
            r"(?i)vssadmin\s*delete\s*shadows",
        ],
        "severity": "high",
    },
    "credential_access": {
        "description": "Theft or access of credentials",
        "patterns": [
            r"(?i)(steal|dump|harvest|extract)\s*(credential|password|token|cookie)",
            r"(?i)(mimikatz|lsass\s*dump|hashdump|sam\s*dump)",
            r"(?i)(keylog|keystroke|clipboard)",
            r"(?i)(brute\s*force|password\s*spray|credential\s*stuff)",
        ],
        "severity": "critical",
    },
    "defense_evasion": {
        "description": "Evasion and anti-analysis techniques",
        "patterns": [
            r"(?i)(obfuscat|pack|crypt|encod)\s*(code|payload|script|binary)",
            r"(?i)(anti[- ]?(debug|vm|sandbox|analysis))",
            r"(?i)(disable|bypass|tamper)\s*(antivirus|defender|edr|amsi)",
            r"(?i)(fileless|living\s*off\s*the\s*land|lolbin)",
        ],
        "severity": "medium",
    },
}


# =============================================================================
# MODULE-LEVEL CONSTANTS (FIX: moved out of functions to avoid re-creation)
# =============================================================================


SEVERITY_SCORES = {
    "critical": 4,
    "high":     3,
    "medium":   2,
    "low":      1,
    "none":     0,
}

LLM_CATEGORY_MAP = {
    "process manipulation":   "process_creation",
    "persistence mechanisms": "registry_modification",
    "network activity":       "network_communication",
    "file system operations": "file_system_activity",
    "defense evasion":        "defense_evasion",
    "discovery":              "discovery",
    "lateral movement":       "lateral_movement",
    "impact":                 "file_system_activity",
}

ALERT_THRESHOLD = 0.5


# =============================================================================
# LAYER 1 — STATIC REGEX DETECTION
# =============================================================================

def _static_behavior_detection(text: str) -> list[dict]:
    """
    Scan the text with all BEHAVIOR_SIGNATURES regex patterns.

    For each matching category, records up to 10 matched indicator strings.
    Static matches are assigned a high base confidence (0.95) because regex
    patterns require an exact or near-exact textual match.

    Returns:
        List of detected behavior dicts, each with category, severity,
        matched indicators, and detection metadata.
    """
    behaviors = []

    for category, config in BEHAVIOR_SIGNATURES.items():
        matched = []
        for pattern in config["patterns"]:
            matches = re.findall(pattern, text)
            if matches:
                for m in matches:
                    indicator = m if isinstance(m, str) else m[0]
                    if indicator and indicator not in matched:
                        matched.append(indicator)

        if matched:
            behaviors.append({
                "category":          category,
                "description":       config["description"],
                "severity":          config["severity"],
                "matched_indicators": matched[:10],  # cap at 10 to avoid overload
                "match_count":       len(matched),
                "detection_source":  "static_regex",
                "confidence":        0.95,           # high confidence for exact regex match
            })

    return behaviors


# =============================================================================
# LAYER 2 — LLM BEHAVIOR DETECTION
# =============================================================================

LLM_BEHAVIOR_PROMPT = """You are a literal-minded Malware Behavior Analyst.

### CRITICAL NEUTRALITY MANDATE:
- Do NOT search for hidden malicious intent in routine IT tasks.
- Routine reboots, patching, or HR links are NOT "Persistence" or "Phishing".
- If the text describes legitimate activity, return an EMPTY behaviors list.

### TASK:
Analyze the message and identify explicit malicious behaviors from categories such as:
1. Process manipulation (injection, hollowing)
2. Persistence (registry, scheduled tasks)
3. Network activity (C2, tunneling)
4. File system operations (encryption, dropper)
5. Credential theft (dumping, keylogging)
6. Defense evasion (obfuscation, AV bypass)
7. Discovery (recon, enumeration)
8. Lateral movement (pass-the-hash, RDP abuse)

Return ONLY valid JSON:
{
    "behaviors": [
        {
            "category": "category_name",
            "description": "Literal description of what was observed",
            "severity": "critical|high|medium|low",
            "confidence": 0.0,
            "indicators": ["excerpt from text"],
            "is_novel": false,
            "notes": "Additional context"
        }
    ],
    "attack_chain_summary": "Brief narrative connecting the behaviors",
    "novel_techniques": ["Description of any never-seen-before techniques"]
}"""


def _llm_behavior_detection(text: str) -> dict:
    """
    Ask the LLM to identify malicious behaviors that regex patterns may miss.

    Returns the parsed LLM result, or a safe empty fallback on failure.
    """
    result = llm_analyze(LLM_BEHAVIOR_PROMPT, text)

    if result.get("llm_failed") or result.get("parse_error"):
        return {
            "behaviors":             [],
            "attack_chain_summary":  "",
            "novel_techniques":      [],
        }

    return result


# =============================================================================
# LAYER 3 — MERGE AND SCORE
# =============================================================================

def _merge_behaviors(
    static_behaviors: list[dict],
    llm_result:       dict,
) -> list[dict]:
    """
    Merge behaviors detected by static regex and LLM into a unified list.

    Merge strategy:
    - Both layers agree on a category  → boost confidence (+0.05), mark "static+llm"
    - Only static detected it          → keep as-is at 0.95 confidence
    - Only LLM detected it             → include if weighted confidence >= ALERT_THRESHOLD
    - Novel LLM behaviors              → flagged with is_novel=True for reporting

    LLM category names are normalized via LLM_CATEGORY_MAP before comparison.
    """
    merged = []
    static_categories = {b["category"] for b in static_behaviors}
    llm_behaviors     = llm_result.get("behaviors", [])

    # --- Process static behaviors first ---
    for sb in static_behaviors:
        cat = sb["category"]

        # Check if the LLM also detected this same category
        llm_match = next(
            (lb for lb in llm_behaviors if lb.get("category") == cat),
            None,
        )

        if llm_match:
            # Both layers agree → boost confidence and enrich indicators
            sb["confidence"]       = min(0.99, sb["confidence"] + 0.05)
            sb["detection_source"] = "static+llm"
            sb["llm_notes"]        = llm_match.get("notes", "")

            # Merge LLM indicators without duplicating existing ones
            llm_indicators        = llm_match.get("indicators", [])
            combined_indicators   = sb["matched_indicators"] + [
                i for i in llm_indicators if i not in sb["matched_indicators"]
            ]
            sb["matched_indicators"] = combined_indicators[:15]

        merged.append(sb)

    # --- Process LLM-only behaviors ---
    for lb in llm_behaviors:
        raw_cat   = lb.get("category", "unknown").lower()
        # Normalize to a canonical category key using the module-level map
        cat       = LLM_CATEGORY_MAP.get(raw_cat, raw_cat)
        lb["category"] = cat

        # Only add if this category was NOT already found by static regex
        if cat not in static_categories:
            merged.append({
                "category":          cat,
                "description":       lb.get("description", ""),
                "severity":          lb.get("severity", "medium"),
                "confidence":        lb.get("confidence", 0.7) * WEIGHTS["llm"],
                "matched_indicators": lb.get("indicators", []),
                "match_count":       len(lb.get("indicators", [])),
                "detection_source":  "llm_only",
                "is_novel":          lb.get("is_novel", False),
                "llm_notes":         lb.get("notes", ""),
            })

    # Sort by severity then confidence (highest first)
    merged.sort(
        key=lambda b: (
            SEVERITY_SCORES.get(b["severity"], 0),
            b.get("confidence", 0),
        ),
        reverse=True,
    )

    return merged


# =============================================================================
# LANGCHAIN TOOL ENTRY POINT
# =============================================================================

@tool
def detect_behaviors(message: str) -> dict:
    """
    Hybrid detection of malicious behaviors in a single CTI message.

    Layer 1 (Static)  : Regex signatures for known ATT&CK techniques
    Layer 2 (LLM)     : Contextual analysis for novel or implied techniques
    Layer 3 (Scoring) : Confidence-weighted merge with safety gate

    Safety Gate Logic:
    - Static regex detections are always included (high confidence).
    - LLM-only detections are included only if their weighted confidence
      is >= ALERT_THRESHOLD (0.5) to suppress hallucinations.

    Args:
        message: Raw text describing a threat or malware behavior

    Returns:
        Complete behavioral analysis with source attribution and severity
    """
    static_behaviors = _static_behavior_detection(message)
    llm_result       = _llm_behavior_detection(message)
    merged           = _merge_behaviors(static_behaviors, llm_result)

    # -------------------------------------------------------------------------
    # SAFETY GATE — filter out low-confidence LLM-only detections
    # A behavior passes if:
    #   A) Static regex found it (high certainty, no threshold needed)
    #   B) Both static AND LLM confirmed it (verified by two sources)
    #   C) LLM found it alone but weighted confidence >= ALERT_THRESHOLD
    # -------------------------------------------------------------------------
    merged = [
        b for b in merged
        if b.get("detection_source") in ("static_regex", "static+llm")
        or b.get("confidence", 0) >= ALERT_THRESHOLD
    ]

    # Determine the most severe behavior detected
    max_severity = "none"
    if merged:
        max_severity = max(
            merged,
            key=lambda b: SEVERITY_SCORES.get(b["severity"], 0),
        )["severity"]

    # Count detections per source type for the summary
    source_counts = {
        "static_regex":   sum(1 for b in merged if b["detection_source"] == "static_regex"),
        "static_plus_llm": sum(1 for b in merged if b["detection_source"] == "static+llm"),
        "llm_only":       sum(1 for b in merged if b["detection_source"] == "llm_only"),
    }

    # Collect novel techniques for the report
    novel = [b for b in merged if b.get("is_novel")]

    # Build a human-readable behavior summary
    summary_parts = [f"{len(merged)} behavior(s) detected."]
    if source_counts["llm_only"] > 0:
        summary_parts.append(
            f"{source_counts['llm_only']} detected ONLY by LLM "
            f"(not covered by static signatures)."
        )
    if novel:
        summary_parts.append(
            f"{len(novel)} potentially novel technique(s) identified."
        )
    summary_parts.append(f"Max severity: {max_severity}.")

    return {
        "behaviors_detected": merged,
        "total_behaviors":    len(merged),
        "max_severity":       max_severity,
        "detection_sources":  source_counts,
        "novel_techniques":   llm_result.get("novel_techniques", []),
        "attack_chain":       llm_result.get("attack_chain_summary", ""),
        "behavior_summary":   " ".join(summary_parts),
    }


Writing detect_behaviors.py


In [17]:
# from detect_behaviors import detect_behaviors as detect_behaviors_tool
# import json

# sample_messages = [
#     "User account attempted multiple failed logins",
#     "Suspicious PowerShell execution detected",
#     "Routine backup completed successfully",
#     "Unusual outbound traffic to unknown IP",
#     "Scheduled maintenance window for database servers"
# ]

# results = []
# for msg in sample_messages:
#     res = detect_behaviors_tool.func(msg)   
#     results.append(res)

# print(json.dumps(results, indent=2))


In [55]:
%%writefile threat_brief.py
"""
CTI FUSION ENGINE — Threat Brief Builder (v3)
===============================================
Redesigned from a passive summarizer (v1) and made fully dynamic (v3).

v2 → v3 changes: eliminated all hardcoded static content lists:
  ✗ Hardcoded malware/country regex in RAG normalization
  ✗ Keyword-based kill chain detection (_KILL_CHAIN_MAP with word lists)
  ✗ Static actor TTP database (_ACTOR_FINGERPRINTS)

Replaced with dynamic approaches:
  ✓ RAG entity search uses entities already extracted by extract_entities (Tool 1)
  ✓ Kill chain derived from behavior .category fields (ATT&CK ontology aliasing)
  ✓ Actor profiling delegated to LLM — enriched with RAG intelligence

Architecture (6 stages):
  Stage 0: Signal Normalization  — parse RAG using already-known entities
  Stage 1: Cross-Signal Correlation — entity×behavior, entity×RAG, CVE×RAG
  Stage 2: Contradiction Resolution  — RAG > regex > llm_only > patterns
  Stage 3: Hypothesis Building       — objective, kill chain, LLM actor profile
  Stage 4: LLM Fusion Analysis       — validate hypotheses, write narrative, T-codes
  Stage 5: Output Assembly           — structured for MITRE Agent + report

Source confidence hierarchy:
  RAG (0.90) > static_regex (0.85) > static+llm (0.87) > llm_only (0.70) > patterns (0.60)
"""

import json
import re
from typing import Any

from llm_helper import llm_analyze


# =============================================================================
# CONSTANTS — weights and ontology mappings only (no content lists)
# =============================================================================

SOURCE_CONFIDENCE = {
    "rag":          0.90,
    "static_regex": 0.85,
    "static+llm":   0.87,
    "llm_only":     0.70,
    "patterns":     0.60,
}

RAG_SCORE_THRESHOLD = 150.0

# Minimum confidence for a correlation link to enter the evidence graph
CORRELATION_MIN_CONFIDENCE = 0.50

_ATTACKCAT_TO_STAGE = {
    # detect_behaviors canonical regex categories
    "process_creation":       "execution",
    "registry_modification":  "persistence",
    "network_communication":  "command_and_control",
    "file_system_activity":   "impact",
    "credential_access":      "credential_access",
    "defense_evasion":        "defense_evasion",
    # LLM-detected categories (from detect_behaviors LLM_CATEGORY_MAP)
    "process_manipulation":   "execution",
    "persistence_mechanisms": "persistence",
    "network_activity":       "command_and_control",
    "file_system_operations": "impact",
    "discovery":              "discovery",
    "lateral_movement":       "lateral_movement",
    "collection":             "collection",
    "exfiltration":           "exfiltration",
    "initial_access":         "initial_access",
    "privilege_escalation":   "privilege_escalation",
    "impact":                 "impact",
}

# Canonical kill chain order (ordering only — no keywords)
_KILL_CHAIN_ORDER = [
    "initial_access", "execution", "persistence", "privilege_escalation",
    "defense_evasion", "credential_access", "discovery", "lateral_movement",
    "collection", "exfiltration", "command_and_control", "impact",
]

# Stage → urgency rating (no keyword matching — stage names are categorical)
_STAGE_URGENCY = {
    "impact":               ("CRITICAL", "Attack in final impact stage — immediate response required"),
    "exfiltration":         ("HIGH",     "Data exfiltration in progress — contain now"),
    "command_and_control":  ("HIGH",     "Active C2 — attacker has established foothold"),
    "collection":           ("HIGH",     "Data being staged for exfiltration"),
    "lateral_movement":     ("MEDIUM",   "Attacker moving laterally — expanding access"),
    "credential_access":    ("MEDIUM",   "Credentials being harvested — escalation likely"),
    "privilege_escalation": ("MEDIUM",   "Privilege escalation observed"),
    "persistence":          ("MEDIUM",   "Persistence established — attacker intends long-term access"),
    "defense_evasion":      ("LOW",      "Evasion active — detection may be impaired"),
    "execution":            ("LOW",      "Payload executing — initial compromise confirmed"),
    "initial_access":       ("LOW",      "Initial access phase — early stage, contain before escalation"),
    "discovery":            ("LOW",      "Reconnaissance in progress"),
}

# Confirmed-event keywords — indicate LE action, patch releases, or attribution
_CONFIRMED_EVENT_KEYWORDS = {
    "arrested", "disrupted", "seized", "taken down", "indicted",
    "confirmed", "attributed", "decryptor", "patch released",
    "infrastructure dismantled", "convicted", "charged", "sanctions",
}

# LE disruption keywords — subset of confirmed-event, reduces threat urgency
_LE_DISRUPTION_KEYWORDS = {
    "arrested", "seized", "disrupted", "taken down", "indicted",
    "infrastructure dismantled", "convicted", "charged",
}


# =============================================================================
# STAGE 0 — SIGNAL NORMALIZATION
# =============================================================================

def _normalize_rag_results(rag_context: dict, entities: dict) -> list[dict]:
    """
    Parse RAG results into structured intelligence objects.

    Dynamic entity matching:
      The search vocabulary is built entirely from entities already extracted
      by extract_entities (Tool 1). No hardcoded actor/country/tool lists.
      This means RAG normalization automatically adapts to whatever is relevant
      in the current message — new actors, unknown malware, emerging threats.

    CVE extraction uses a format-based regex (CVE-YEAR-ID universal standard),
    which is the one legitimate exception: it matches a format, not a content list.

    Args:
        rag_context : Raw rag_search tool output
        entities    : extract_entities tool output — provides dynamic vocabulary

    Returns:
        Score-filtered list of structured RAG intelligence objects
    """
    # Build dynamic search vocabulary from what extract_entities already found
    known_actors: set[str] = set()
    for field in ("malware_names", "ransomware_groups", "apt_groups", "threat_actors"):
        for item in entities.get(field, []):
            if isinstance(item, str) and item.strip():
                known_actors.add(item.strip().lower())

    known_countries: set[str] = {
        c.strip().lower()
        for c in entities.get("targeted_countries", [])
        if isinstance(c, str) and c.strip()
    }

    known_tools: set[str] = {
        t.strip().lower()
        for t in entities.get("tools_abused", [])
        if isinstance(t, str) and t.strip()
    }

    normalized = []
    for r in rag_context.get("results", []):
        score   = r.get("score", 999.0)
        content = r.get("content", "")

        if score > RAG_SCORE_THRESHOLD:
            continue  # too dissimilar to the query — discard

        content_lower = content.lower()

        # CVE: format-based (matches universal CVE-YEAR-ID specification)
        cves_found = [c.upper() for c in re.findall(r"CVE-\d{4}-\d{4,}", content, re.IGNORECASE)]

        # All other entity checks are dynamic — driven by what extract_entities found
        actors_found    = [a for a in known_actors    if a in content_lower]
        countries_found = [c for c in known_countries if c in content_lower]
        tools_found     = [t for t in known_tools     if t in content_lower]

        # Classify this RAG item: confirmed event vs unverified report
        is_confirmed_event = any(kw in content_lower for kw in _CONFIRMED_EVENT_KEYWORDS)
        has_le_action      = any(kw in content_lower for kw in _LE_DISRUPTION_KEYWORDS)

        normalized.append({
            "similarity_score":   round(score, 4),
            "confidence":         SOURCE_CONFIDENCE["rag"],
            "content":            content,
            "channel":            r.get("channel", "unknown"),
            "date":               r.get("date", ""),
            # Dynamically matched entities
            "actors_found":       actors_found,
            "countries_found":    countries_found,
            "tools_found":        tools_found,
            "cves_found":         cves_found,
            # Event classification flags
            "is_confirmed_event": is_confirmed_event,
            "has_le_action":      has_le_action,
        })

    # Most similar first (ascending L2 distance = ascending score)
    return sorted(normalized, key=lambda r: r["similarity_score"])


def _extract_behavioral_iocs(entities: dict) -> list[str]:
    """
    Extract IOCs relevant for MITRE ATT&CK technique mapping.
    Excludes raw network IOCs (IPs, hashes, emails) — no ATT&CK signal.
    """
    relevant = []
    for key in (
        "malware_names", "ransomware_groups", "apt_groups",
        "tools_abused", "cves", "campaign_names", "threat_actors",
    ):
        for item in entities.get(key, []):
            if isinstance(item, str) and item.strip():
                relevant.append(item.strip())

    urls = entities.get("urls", [])
    if len(urls) <= 5:
        relevant.extend(urls)

    return list(dict.fromkeys(relevant))


# =============================================================================
# STAGE 1 — CROSS-SIGNAL CORRELATION ENGINE
# =============================================================================

def _correlate_signals(
    entities:       dict,
    behaviors:      list[dict],
    patterns:       dict,
    rag_normalized: list[dict],
) -> dict:
    """
    Build an evidence graph by correlating signals across all four tools.

    Correlation pairs:
      entity × behavior : Actor type (ransomware/APT/malware) matched against
                          behavior category and severity — no name lookup tables
      entity × RAG      : Dynamic search: which of our extracted actors appear
                          in RAG content? Boosts confidence; flags LE actions
      behavior × pattern: Coherence check between behavior severity and pattern
                          classification; detects BENIGN vs MALICIOUS contradictions
      RAG × pattern     : Independent theme corroboration via pattern CTI term overlap
      CVE × RAG         : Same CVE in current message AND historical RAG = strong signal
    """
    links              = []
    rag_confirmations  = []
    rag_contradictions = []
    entity_scores      = {}

    all_actors = list(dict.fromkeys(
        entities.get("malware_names",     []) +
        entities.get("ransomware_groups", []) +
        entities.get("apt_groups",        []) +
        entities.get("threat_actors",     [])
    ))

    behavior_categories = {b.get("category", "") for b in behaviors}
    behavior_severities  = {b.get("severity",  "") for b in behaviors}

    # ── Entity × Behavior Correlation ────────────────────────────────────────
    # Coherence is determined by ACTOR TYPE (from entity dict labels),
    # not by a hardcoded name→behavior lookup table.
    ransomware_behaviors = {"file_system_activity", "network_communication", "credential_access"}
    apt_behaviors        = {"lateral_movement", "credential_access", "defense_evasion",
                            "network_communication", "discovery"}
    generic_behaviors    = {"process_creation", "network_communication"}

    for actor in all_actors:
        if actor in entities.get("ransomware_groups", []):
            actor_type = "ransomware"
            expected   = ransomware_behaviors
        elif actor in entities.get("apt_groups", []):
            actor_type = "apt"
            expected   = apt_behaviors
        else:
            actor_type = "malware"
            expected   = generic_behaviors

        confirmed = expected & behavior_categories
        base_conf = SOURCE_CONFIDENCE["llm_only"]
        conf      = min(0.95, base_conf + 0.08 * len(confirmed)) if confirmed else base_conf

        entity_scores[actor] = conf
        if confirmed:
            links.append({
                "type":               "entity_x_behavior",
                "entity":             actor,
                "actor_type":         actor_type,
                "confirmed_behaviors": list(confirmed),
                "confidence":         conf,
                "interpretation": (
                    f"'{actor}' ({actor_type}) corroborated by "
                    f"{len(confirmed)} matching behavior(s): {', '.join(confirmed)}"
                ),
            })

    # ── Entity × RAG Correlation ──────────────────────────────────────────────
    for rag_item in rag_normalized:
        for actor in all_actors:
            if actor.lower() in rag_item["actors_found"]:
                boost    = 0.15 if rag_item["is_confirmed_event"] else 0.08
                new_conf = min(0.98, entity_scores.get(actor, 0.70) + boost)

                rag_confirmations.append({
                    "type":               "rag_confirms_entity",
                    "entity":             actor,
                    "rag_channel":        rag_item["channel"],
                    "rag_date":           rag_item["date"],
                    "is_confirmed_event": rag_item["is_confirmed_event"],
                    "confidence_before":  entity_scores.get(actor, 0.70),
                    "confidence_after":   new_conf,
                    "interpretation": (
                        f"RAG independently references '{actor}' "
                        f"({'confirmed event' if rag_item['is_confirmed_event'] else 'related report'}) "
                        f"— confidence {entity_scores.get(actor, 0.70):.2f} → {new_conf:.2f}"
                    ),
                })
                entity_scores[actor] = new_conf

                if rag_item["has_le_action"]:
                    rag_contradictions.append({
                        "type":        "rag_le_action",
                        "entity":      actor,
                        "rag_channel": rag_item["channel"],
                        "note": (
                            f"RAG intelligence indicates law enforcement action against '{actor}' "
                            f"— verify whether this actor's infrastructure is still active"
                        ),
                    })

    # ── Behavior × Pattern Coherence ─────────────────────────────────────────
    classification  = patterns.get("threat_classification", "").upper()
    cti_terms_found = patterns.get("known_cti_terms", [])
    has_critical    = "critical" in behavior_severities
    has_high        = "high"     in behavior_severities

    if has_critical and any(kw in classification for kw in ("RANSOMWARE", "APT", "MALICIOUS")):
        coherence_score = 0.88
        coherence_note  = "Critical behaviors and pattern classification are coherent."
    elif has_high and classification not in ("BENIGN", "ADMINISTRATIVE"):
        coherence_score = 0.75
        coherence_note  = "High-severity behaviors partially corroborate pattern classification."
    elif classification in ("BENIGN", "ADMINISTRATIVE") and (has_critical or has_high):
        coherence_score = 0.35
        coherence_note  = (
            "⚠ CONTRADICTION: Pattern engine says BENIGN/ADMINISTRATIVE but "
            "critical/high-severity behaviors detected. "
            "Behaviors (static_regex, conf=0.85) override patterns (conf=0.60)."
        )
    else:
        coherence_score = SOURCE_CONFIDENCE["patterns"]
        coherence_note  = "Pattern classification has limited behavioral corroboration."

    links.append({
        "type":             "behavior_x_pattern",
        "classification":   classification,
        "behavior_severity": "critical" if has_critical else ("high" if has_high else "medium"),
        "coherence_score":  coherence_score,
        "interpretation":   coherence_note,
    })

    # ── RAG × Pattern Correlation ─────────────────────────────────────────────
    all_rag_text     = " ".join(r["content"].lower() for r in rag_normalized)
    rag_term_overlap = [t for t in cti_terms_found if t in all_rag_text]
    if rag_term_overlap:
        links.append({
            "type":         "rag_x_pattern",
            "shared_terms": rag_term_overlap,
            "confidence":   SOURCE_CONFIDENCE["rag"],
            "interpretation": (
                f"RAG and pattern engine independently identify "
                f"{len(rag_term_overlap)} shared CTI term(s): "
                f"{', '.join(rag_term_overlap[:5])} — independent theme confirmation"
            ),
        })

    # ── CVE × RAG ─────────────────────────────────────────────────────────────
    entity_cves = set(entities.get("cves", []))
    rag_cves    = set(cve for r in rag_normalized for cve in r["cves_found"])
    cve_overlap = entity_cves & rag_cves
    if cve_overlap:
        links.append({
            "type":        "cve_x_rag",
            "shared_cves": list(cve_overlap),
            "confidence":  SOURCE_CONFIDENCE["rag"],
            "interpretation": (
                f"CVE(s) {', '.join(cve_overlap)} appear in BOTH the current message "
                f"AND historical RAG intelligence — active exploitation confirmed"
            ),
        })

    return {
        "links":              links,
        "rag_confirmations":  rag_confirmations,
        "contradictions":     rag_contradictions,
        "entity_scores":      entity_scores,
        "cve_overlap":        list(cve_overlap),
        "rag_term_overlap":   rag_term_overlap,
        "total_correlations": len(links) + len(rag_confirmations),
    }


# =============================================================================
# STAGE 2 — CONTRADICTION RESOLUTION
# =============================================================================

def _resolve_contradictions(
    correlation_graph: dict,
    entities:          dict,
    patterns:          dict,
    rag_normalized:    list[dict],
) -> dict:
    """
    Apply source confidence priority chain to resolve conflicting signals.
    Priority: RAG (0.90) > static_regex (0.85) > llm_only (0.70) > patterns (0.60)
    """
    notes = []

    # A — Behavior vs Pattern contradiction
    bp_link = next(
        (l for l in correlation_graph["links"] if l["type"] == "behavior_x_pattern"), None
    )
    if bp_link and bp_link["coherence_score"] < 0.50:
        final_classification = "MALICIOUS (Behavior-Confirmed, Pattern Overridden)"
        final_confidence     = SOURCE_CONFIDENCE["static_regex"]
        notes.append(
            f"Pattern overridden: static_regex (conf={SOURCE_CONFIDENCE['static_regex']}) "
            f"> patterns (conf={SOURCE_CONFIDENCE['patterns']})"
        )
    else:
        raw_class        = patterns.get("threat_classification", "UNKNOWN")
        raw_conf         = patterns.get("classification_confidence", 0.50)
        rag_boost        = 0.10 if correlation_graph.get("rag_term_overlap") else 0.0
        final_classification = raw_class
        final_confidence     = min(0.95, raw_conf + rag_boost)

    # B — Entity confidence flags
    entity_scores = correlation_graph.get("entity_scores", {})
    entity_flags  = {}
    for entity, score in entity_scores.items():
        if score < CORRELATION_MIN_CONFIDENCE:
            entity_flags[entity] = f"unconfirmed (conf={score:.2f})"
        elif score >= SOURCE_CONFIDENCE["rag"]:
            entity_flags[entity] = f"high-confidence (conf={score:.2f}) — RAG+behavior corroborated"
        else:
            entity_flags[entity] = f"moderate-confidence (conf={score:.2f})"

    # C — LE disruption flags
    le_flags = [c for c in correlation_graph.get("contradictions", []) if c["type"] == "rag_le_action"]
    if le_flags:
        notes.append(
            f"RAG references possible LE action against: "
            f"{', '.join(f['entity'] for f in le_flags)}. "
            f"Assess whether threat actor infrastructure remains active."
        )

    return {
        "final_classification": final_classification,
        "final_confidence":     round(final_confidence, 2),
        "resolution_notes":     notes,
        "entity_flags":         entity_flags,
        "high_conf_actors":     [e for e, f in entity_flags.items() if "high-confidence" in f],
        "unconfirmed_actors":   [e for e, f in entity_flags.items() if "unconfirmed"     in f],
        "le_action_flags":      le_flags,
    }


# =============================================================================
# STAGE 3 — HYPOTHESIS BUILDING
# =============================================================================

def _derive_kill_chain(behaviors: list[dict]) -> tuple[list[str], str, str]:
    """
    Derive kill chain stages from behavior .category fields via ATT&CK ontology.

    No keyword scanning. detect_behaviors already maps raw text → ATT&CK categories.
    We alias those category names to Unified Kill Chain stage names via
    _ATTACKCAT_TO_STAGE (a principled ontology mapping, not a word list).

    Returns:
        (ordered_stages, current_stage, urgency_level)
    """
    detected: set[str] = set()
    for b in behaviors:
        cat    = b.get("category", "").lower()
        mapped = _ATTACKCAT_TO_STAGE.get(cat)
        if mapped:
            detected.add(mapped)
        # If category IS already a canonical stage name, include directly
        if cat in _STAGE_URGENCY:
            detected.add(cat)

    ordered       = [s for s in _KILL_CHAIN_ORDER if s in detected]
    current_stage = ordered[-1] if ordered else "unknown"
    urgency, _    = _STAGE_URGENCY.get(current_stage, ("UNKNOWN", ""))

    return ordered, current_stage, urgency


# ── LLM Actor Profiling ───────────────────────────────────────────────────────

_ACTOR_PROFILE_PROMPT = """You are a Cyber Threat Intelligence actor attribution analyst.

You receive:
- Observed tools, behaviors, and entities from a current CTI message
- RAG intelligence (highest confidence source — historical, vetted data)
- Pre-computed kill chain stages and entity confidence scores

YOUR TASK: Profile the most likely threat actor from what is actually observed.
Use your knowledge of known APT groups, ransomware operators, and cybercrime actors.
Do NOT invent actors. If evidence is insufficient, output "unknown".

SOURCE HIERARCHY: RAG (0.90) > behaviors (0.85) > entities (0.70)

STRICT RULES:
- Unconfirmed actors (no RAG / no behavior corroboration) → confidence < 0.50
- If RAG shows law enforcement action against an actor, flag disruption_risk
- typical_next_moves must reflect this actor's KNOWN playbook, not generic steps
- All fields required; null for unknown
- IMPORTANT: Distinguish between MALWARE FAMILY (the tool) and THREAT ACTOR (the operator).
  "LockBit 3.0" is a malware family; the operator may be an affiliate group such as ALPHV/BlackCat.
  If both are present, report the OPERATOR as likely_actor and the malware as evidence.
- If multiple actor names appear, report the one with strongest behavioral corroboration,
  not necessarily the one mentioned most often.

Return ONLY valid JSON:
{
  "likely_actor": "actor name | 'unknown opportunistic' | 'unknown targeted APT'",
  "confidence": 0.0,
  "actor_maturity": "script_kiddie | opportunistic | targeted | apt_level",
  "actor_type": "ransomware | apt | stealer | ddos | broker | unknown",
  "double_extortion": false,
  "typical_next_moves": ["concrete next steps from this actor's known playbook"],
  "known_campaigns": ["campaigns matching this profile from your knowledge base"],
  "target_sectors": ["sectors this actor is known to target"],
  "target_geographies": ["countries/regions typically targeted"],
  "infrastructure_notes": "What is known about this actor's C2 / hosting infrastructure",
  "disruption_risk": "active | possibly_disrupted | disrupted | unknown",
  "evidence_summary": ["specific evidence items driving this attribution"],
  "attribution_gaps": ["what information is missing to raise confidence further"]
}"""


def _llm_actor_profile(
    entities:          dict,
    behaviors:         list[dict],
    rag_normalized:    list[dict],
    kill_chain_stages: list[str],
    resolution:        dict,
) -> dict:
    """
    Dynamically profile the threat actor via LLM + RAG intelligence.

    Replaces the static _ACTOR_FINGERPRINTS dictionary entirely.
    The LLM has actor TTP knowledge from training; RAG provides mission-specific
    historical intelligence. Together they replace any hardcoded lookup table
    while covering far more actors and staying current via RAG.
    """
    context = {
        "high_confidence_actors": resolution.get("high_conf_actors", []),
        "unconfirmed_actors":     resolution.get("unconfirmed_actors", []),
        "le_disruption_flags":    [f["entity"] for f in resolution.get("le_action_flags", [])],
        "observed_tools":         entities.get("tools_abused",      []),
        "observed_malware":       entities.get("malware_names",     []),
        "ransomware_groups":      entities.get("ransomware_groups", []),
        "apt_groups":             entities.get("apt_groups",        []),
        "detected_behaviors": [
            {
                "category":  b.get("category"),
                "severity":  b.get("severity"),
                "source":    b.get("detection_source"),
                "indicators": b.get("matched_indicators", [])[:3],
            }
            for b in behaviors
        ],
        "kill_chain_stages_observed": kill_chain_stages,
        "targeted_sectors":           entities.get("targeted_sectors",   []),
        "targeted_countries":         entities.get("targeted_countries", []),
        # RAG as ground truth — full structured intelligence
        "rag_intelligence": [
            {
                "content":     r["content"][:350],
                "channel":     r["channel"],
                "date":        r["date"],
                "actors_found": r["actors_found"],
                "is_confirmed_event": r["is_confirmed_event"],
                "has_le_action":      r["has_le_action"],
            }
            for r in rag_normalized
        ],
    }

    result = llm_analyze(_ACTOR_PROFILE_PROMPT, json.dumps(context, default=str))

    if result.get("llm_failed") or result.get("parse_error"):
        candidates = (
            resolution.get("high_conf_actors") or
            entities.get("ransomware_groups") or
            entities.get("apt_groups") or
            entities.get("threat_actors") or
            ["unknown"]
        )
        actor = candidates[0] if candidates else "unknown"
        return {
            "likely_actor":       actor,
            "confidence":         0.50 if actor != "unknown" else 0.10,
            "actor_maturity":     "unknown",
            "actor_type":         "unknown",
            "double_extortion":   False,
            "typical_next_moves": [],
            "known_campaigns":    [],
            "target_sectors":     entities.get("targeted_sectors", []),
            "target_geographies": entities.get("targeted_countries", []),
            "infrastructure_notes": "",
            "disruption_risk":    "unknown",
            "evidence_summary":   resolution.get("high_conf_actors", []),
            "attribution_gaps":   ["LLM actor profiling unavailable — fallback used"],
        }

    return result


def _build_hypotheses(
    entities:          dict,
    behaviors:         list[dict],
    patterns:          dict,
    rag_normalized:    list[dict],
    correlation_graph: dict,
    resolution:        dict,
) -> dict:
    """
    Build pre-computed intelligence hypotheses before the main LLM fusion call.

    H1 — Attack Objective : from entity type labels + behavior categories (no hardcoded names)
    H2 — Operational Stage: from behavior categories via ATT&CK ontology alias
    H3 — Actor Profile    : fully dynamic LLM call enriched with RAG intelligence
    """
    # H1 — Objective: inferred from entity TYPE (dict key) + behavior CATEGORY
    is_ransomware = (
        bool(entities.get("ransomware_groups")) or
        any(b.get("category") == "file_system_activity" and
            b.get("severity") in ("high", "critical") for b in behaviors)
    )
    is_espionage = (
        bool(entities.get("apt_groups")) or
        any(b.get("category") in ("exfiltration", "collection", "discovery") for b in behaviors)
    )
    is_financial = bool(any(entities.get("crypto_wallets", {}).values()))

    if is_ransomware:
        objective    = "ransomware_extortion"
        obj_conf     = min(0.95, 0.65 + 0.08 * len(entities.get("ransomware_groups", [])))
        obj_evidence = entities.get("ransomware_groups", []) + (
            ["file encryption behavior confirmed"] if any(
                b.get("category") == "file_system_activity" for b in behaviors
            ) else []
        )
    elif is_espionage:
        objective    = "data_exfiltration_espionage"
        obj_conf     = min(0.90, 0.60 + 0.08 * len(entities.get("apt_groups", [])))
        obj_evidence = entities.get("apt_groups", []) + ["exfiltration/collection behaviors"]
    elif is_financial:
        objective    = "financial_theft"
        obj_conf     = 0.70
        obj_evidence = ["crypto wallet indicators present"]
    else:
        objective    = "unknown"
        obj_conf     = 0.30
        obj_evidence = []

    h1 = {
        "hypothesis": "H1 — Attack Objective",
        "assessment": objective,
        "confidence": round(obj_conf, 2),
        "evidence":   obj_evidence,
    }

    # H2 — Kill Chain / Operational Stage: from behavior categories (ontology)
    ordered_stages, current_stage, urgency_level = _derive_kill_chain(behaviors)
    urgency_note = _STAGE_URGENCY.get(current_stage, ("UNKNOWN", ""))[1]

    h2 = {
        "hypothesis":       "H2 — Operational Stage",
        "all_stages":       ordered_stages,
        "current_stage":    current_stage,
        "urgency":          urgency_level,
        "urgency_note":     urgency_note,
        "stages_completed": len(ordered_stages),
    }

    # H3 — Actor Profile: fully dynamic LLM call
    actor_profile = _llm_actor_profile(
        entities          = entities,
        behaviors         = behaviors,
        rag_normalized    = rag_normalized,
        kill_chain_stages = ordered_stages,
        resolution        = resolution,
    )

    h3 = {
        "hypothesis":    "H3 — Actor Profile (LLM-Dynamic)",
        "actor_profile": actor_profile,
        "best_match":    actor_profile.get("likely_actor", "unknown"),
        "best_score":    actor_profile.get("confidence",   0.0),
        "note": (
            f"LLM attribution: '{actor_profile.get('likely_actor', 'unknown')}' "
            f"(conf={actor_profile.get('confidence', 0.0):.2f})"
        ),
    }

    return {
        "h1_objective":         h1,
        "h2_operational_stage": h2,
        "h3_actor_profile":     h3,
        "kill_chain_stages":    ordered_stages,
        "urgency":              urgency_level,
    }


# =============================================================================
# STAGE 4 — LLM FUSION ANALYST
# =============================================================================

_FUSION_PROMPT = """You are a Cyber Threat Intelligence (CTI) Fusion Analyst.

### YOUR ROLE
You do NOT summarize tool outputs independently.
You receive pre-computed correlation results and static hypotheses.
Your job: VALIDATE, ENRICH, and CONNECT them into one unified intelligence picture.

### WHAT IS ALREADY DONE
1. Cross-signal correlation graph (entity×behavior, entity×RAG, CVE×RAG)
2. Contradiction resolution using source confidence hierarchy
3. Attack objective, kill chain, and actor profile hypothesized

### YOUR TASKS
1. Validate the pre-built hypotheses — confirm or adjust with reasoning
2. Write the unified attack NARRATIVE (one connected paragraph, not a list)
3. Identify signal relationships the static engine missed
4. Assign ATT&CK T-codes to each behavior for the MITRE Agent
5. List intelligence gaps — tell the MITRE Agent what NOT to hallucinate

### SOURCE PRIORITY (use when signals conflict)
  RAG (0.90) > static_regex (0.85) > llm_entity (0.70) > patterns (0.60)

### STRICT RULES
- Do NOT assert unconfirmed_actors with confidence ≥ 0.50
- Do NOT invent CVEs, tools, or actor names not present in the input
- Do NOT copy RAG content verbatim — interpret and cite it
- attack_narrative must be one connected paragraph, not bullet points
- Every mitre_relevant_behavior MUST include an attack_technique_hint.
Use a real MITRE ATT&CK T-code ONLY if you are certain it matches the observed behavior.
If uncertain, write 'T-code: unknown — [describe the technique in plain English]'.
NEVER invent a T-code. An honest 'unknown' is required over a hallucinated T-code.
- If RAG shows LE disruption, reduce threat urgency and note it explicitly

Return ONLY valid JSON — no markdown, no preamble:
{
  "attack_narrative": "Single connected paragraph describing the full correlated attack chain",
  "actor_assessment": {
    "validated_actor": "actor name or unknown",
    "confidence_adjustment": "raised | unchanged | lowered",
    "adjustment_reason": "why confidence was adjusted from the H3 hypothesis",
    "validated_maturity": "opportunistic | targeted | apt_level",
    "rag_confirmed": true
  },
  "environment_context": {
    "target_os": "Windows | Linux | macOS | unknown",
    "target_sector": ["list"],
    "infrastructure_criticality": "high | medium | low | unknown",
    "victim_profile": "brief description of likely victim org type"
  },
  "attack_maturity": "opportunistic | targeted | apt_level",
  "operational_stage": "current kill chain stage name + urgency context",
  "kill_chain_narrative": "Causal paragraph connecting each observed stage to the next",
  "intelligence_gaps": ["Specific unknowns that matter for MITRE technique mapping"],
  "mitre_relevant_behaviors": [
    {
      "category": "credential_access",
      "description": "LSASS dumped via Mimikatz following UAC bypass",
      "attack_technique_hint": "T1003.001 — OS Credential Dumping: LSASS Memory",
      "confidence": 0.92,
      "evidence": ["mimikatz entity (RAG-confirmed)", "credential_access behavior (static_regex)"],
      "key_indicators": ["mimikatz", "lsass", "hashdump"]
    }
  ],
  "campaign_assessment": {
    "is_known_campaign": true,
    "campaign_name_or_type": "string",
    "double_extortion": true,
    "threat_still_active": true,
    "disruption_note": null
  },
  "noise_filtered": [
    {"item": "192.168.1.1", "reason": "raw network IOC — no ATT&CK technique mapping"}
  ]
}"""


def _llm_fusion_analyst(
    correlation_graph: dict,
    resolution:        dict,
    hypotheses:        dict,
    entities:          dict,
    behaviors:         list[dict],
    patterns:          dict,
    rag_normalized:    list[dict],
    behavioral_iocs:   list[str],
) -> dict:
    """
    Main LLM fusion call — reasons over pre-computed correlations, not raw dumps.

    v1 sent 6 raw JSON blobs → LLM treated tools independently → no correlation.
    v3 sends correlation graph + resolved contradictions + static hypotheses
    → LLM validates and enriches relationships → true fusion analysis.
    """
    context = {
        "correlation_graph": {
            "confirmed_links":    correlation_graph["links"],
            "rag_confirmations":  correlation_graph["rag_confirmations"],
            "contradictions":     correlation_graph["contradictions"],
            "total_correlations": correlation_graph["total_correlations"],
        },
        "resolution": {
            "final_classification":   resolution["final_classification"],
            "final_confidence":       resolution["final_confidence"],
            "high_confidence_actors": resolution["high_conf_actors"],
            "unconfirmed_actors":     resolution["unconfirmed_actors"],
            "resolution_notes":       resolution["resolution_notes"],
            "le_action_flags":        resolution["le_action_flags"],
        },
        "hypotheses": {
            "attack_objective":  hypotheses["h1_objective"],
            "operational_stage": hypotheses["h2_operational_stage"],
            "actor_profile":     hypotheses["h3_actor_profile"],
        },
        # RAG — highest confidence source, full content
        "rag_intelligence": [
            {
                "content":      r["content"][:400],
                "channel":      r["channel"],
                "date":         r["date"],
                "confidence":   r["confidence"],
                "actors_found": r["actors_found"],
                "cves_found":   r["cves_found"],
                "is_confirmed_event": r["is_confirmed_event"],
                "has_le_action":      r["has_le_action"],
            }
            for r in rag_normalized
        ],
        "behaviors": [
            {
                "category":   b.get("category"),
                "description": b.get("description"),
                "severity":   b.get("severity"),
                "confidence": b.get("confidence"),
                "source":     b.get("detection_source"),
                "indicators": b.get("matched_indicators", [])[:5],
            }
            for b in behaviors
        ],
        "entities": {
            "malware":       entities.get("malware_names",     []),
            "ransomware":    entities.get("ransomware_groups", []),
            "apt_groups":    entities.get("apt_groups",        []),
            "tools":         entities.get("tools_abused",      []),
            "cves":          entities.get("cves",              []),
            "threat_actors": entities.get("threat_actors",     []),
            "campaigns":     entities.get("campaign_names",    []),
            "sectors":       entities.get("targeted_sectors",  []),
            "countries":     entities.get("targeted_countries",[]),
        },
        "pattern_signals": {
            "classification":  patterns.get("threat_classification", ""),
            "semantic_summary": patterns.get("semantic_summary", ""),
            "known_cti_terms": patterns.get("known_cti_terms", []),
            "predicted_steps": patterns.get("predicted_next_steps", []),
            "emerging_terms":  patterns.get("emerging_terms", []),
        },
        "behavioral_iocs_for_mitre": behavioral_iocs,
    }

    result = llm_analyze(_FUSION_PROMPT, json.dumps(context, default=str))

    if result.get("llm_failed") or result.get("parse_error"):
        return _static_fallback(entities, behaviors, hypotheses, resolution)

    return result


def _static_fallback(
    entities:   dict,
    behaviors:  list[dict],
    hypotheses: dict,
    resolution: dict,
) -> dict:
    """Minimal static fallback when the LLM call fails."""
    h1 = hypotheses["h1_objective"]
    h2 = hypotheses["h2_operational_stage"]
    h3 = hypotheses["h3_actor_profile"]

    return {
        "attack_narrative": (
            f"[STATIC FALLBACK] Objective: {h1['assessment']} (conf={h1['confidence']}). "
            f"Stage: {h2['current_stage']} — urgency: {h2['urgency']}. "
            f"Actor: {h3['best_match']} (conf={h3['best_score']:.2f}). "
            f"Classification: {resolution['final_classification']}."
        ),
        "actor_assessment": {
            "validated_actor":      h3["best_match"],
            "confidence_adjustment": "unchanged",
            "adjustment_reason":    "LLM unavailable — hypothesis used directly",
            "validated_maturity":   h3["actor_profile"].get("actor_maturity", "unknown"),
            "rag_confirmed":        bool(resolution["high_conf_actors"]),
        },
        "environment_context": {
            "target_os":                 "unknown",
            "target_sector":             entities.get("targeted_sectors", []),
            "infrastructure_criticality": "unknown",
            "victim_profile":            "unknown",
        },
        "attack_maturity":    "unknown",
        "operational_stage":  h2["current_stage"],
        "kill_chain_narrative": f"Stages detected: {', '.join(h2['all_stages'])}",
        "intelligence_gaps":  ["LLM fusion unavailable — static fallback active"],
        "mitre_relevant_behaviors": [
            {
                "category":             b.get("category", ""),
                "description":          b.get("description", ""),
                "attack_technique_hint": "",
                "confidence":           b.get("confidence", 0.70),
                "evidence":             [b.get("detection_source", "")],
                "key_indicators":       b.get("matched_indicators", [])[:3],
            }
            for b in behaviors
        ],
        "campaign_assessment": {
            "is_known_campaign":    h3["best_score"] > 0.60,
            "campaign_name_or_type": h3["note"],
            "double_extortion":     h3["actor_profile"].get("double_extortion", False),
            "threat_still_active":  not bool(resolution["le_action_flags"]),
            "disruption_note":      None,
        },
        "noise_filtered": [],
    }


# =============================================================================
# STAGE 5 — OUTPUT ASSEMBLY
# =============================================================================

def _assemble_mitre_input(
    fusion_result:   dict,
    hypotheses:      dict,
    resolution:      dict,
    behavioral_iocs: list[str],
) -> dict:
    """Assemble enriched MITRE Agent input from fusion results."""
    actor_assess = fusion_result.get("actor_assessment", {})
    h3           = hypotheses["h3_actor_profile"]

    return {
        # Correlated behaviors with T-code hints
        "behaviors":      fusion_result.get("mitre_relevant_behaviors", []),
        # Fully resolved actor context
        "actor_context": {
            "validated_actor":    actor_assess.get("validated_actor",    h3["best_match"]),
            "confidence":         h3["actor_profile"].get("confidence",  0.0),
            "actor_type":         h3["actor_profile"].get("actor_type",  "unknown"),
            "actor_maturity":     actor_assess.get("validated_maturity", "unknown"),
            "double_extortion":   h3["actor_profile"].get("double_extortion", False),
            "typical_next_moves": h3["actor_profile"].get("typical_next_moves", []),
            "rag_confirmed":      actor_assess.get("rag_confirmed",      False),
            "disruption_risk":    h3["actor_profile"].get("disruption_risk", "unknown"),
            "campaign_assessment": fusion_result.get("campaign_assessment", {}),
        },
        "kill_chain_stages":       hypotheses["kill_chain_stages"],
        "attack_maturity":         fusion_result.get("attack_maturity",    "unknown"),
        "operational_stage":       fusion_result.get("operational_stage",  "unknown"),
        "urgency":                 hypotheses["urgency"],
        "intelligence_gaps":       fusion_result.get("intelligence_gaps",  []),
        "behavioral_iocs":         behavioral_iocs,
        "resolved_classification": resolution["final_classification"],
        "resolved_confidence":     resolution["final_confidence"],
        "attribution_gaps":        h3["actor_profile"].get("attribution_gaps", []),
    }


# =============================================================================
# PUBLIC ENTRY POINT
# =============================================================================

def build_threat_brief(
    entities:    dict,
    behaviors:   list[dict],
    patterns:    dict,
    rag_context: dict,
) -> dict:
    """
    CTI Fusion Engine — fully dynamic, 6-stage pipeline.

    No hardcoded actor lists, no keyword-based kill chain detection,
    no static TTP fingerprint databases. All intelligence is derived
    dynamically from tool outputs + LLM knowledge + RAG.

    Args:
        entities    : Output from extract_entities tool
        behaviors   : behaviors_detected list from detect_behaviors tool
        patterns    : Output from detect_patterns tool
        rag_context : Parsed output from rag_search tool
    """
    print("  [Fusion 0/5] Normalizing signals (dynamic entity matching)...")
    rag_normalized  = _normalize_rag_results(rag_context, entities)
    behavioral_iocs = _extract_behavioral_iocs(entities)

    print("  [Fusion 1/5] Building cross-signal correlation graph...")
    correlation_graph = _correlate_signals(entities, behaviors, patterns, rag_normalized)

    print("  [Fusion 2/5] Resolving contradictions (priority chain)...")
    resolution = _resolve_contradictions(
        correlation_graph, entities, patterns, rag_normalized
    )

    print("  [Fusion 3/5] Building hypotheses (kill chain + LLM actor profile)...")
    hypotheses = _build_hypotheses(
        entities, behaviors, patterns,
        rag_normalized, correlation_graph, resolution,
    )

    print("  [Fusion 4/5] Running LLM fusion analysis...")
    fusion_result = _llm_fusion_analyst(
        correlation_graph = correlation_graph,
        resolution        = resolution,
        hypotheses        = hypotheses,
        entities          = entities,
        behaviors         = behaviors,
        patterns          = patterns,
        rag_normalized    = rag_normalized,
        behavioral_iocs   = behavioral_iocs,
    )

    print("  [Fusion 5/5] Assembling output for MITRE Agent...")
    mitre_input = _assemble_mitre_input(
        fusion_result   = fusion_result,
        hypotheses      = hypotheses,
        resolution      = resolution,
        behavioral_iocs = behavioral_iocs,
    )

    return {
        # For MITRE Agent
        "mitre_input": mitre_input,

        # Full fusion intelligence
        "fusion": {
            "correlation_graph": {
                "links":             correlation_graph["links"],
                "rag_confirmations": correlation_graph["rag_confirmations"],
                "contradictions":    correlation_graph["contradictions"],
                "entity_scores":     correlation_graph["entity_scores"],
                "total_correlations": correlation_graph["total_correlations"],
            },
            "resolution":  resolution,
            "hypotheses":  hypotheses,
            "rag_results_used": len(rag_normalized),
        },

        # For final report
        "attack_intelligence": {
            "narrative":           fusion_result.get("attack_narrative", ""),
            "actor_profile":       hypotheses["h3_actor_profile"]["actor_profile"],
            "actor_assessment":    fusion_result.get("actor_assessment", {}),
            "campaign_assessment": fusion_result.get("campaign_assessment", {}),
            "environment_context": fusion_result.get("environment_context", {}),
        },

        # Legacy compatibility
        "attack_narrative":     fusion_result.get("attack_narrative", ""),
        "kill_chain_narrative": fusion_result.get("kill_chain_narrative", ""),
        "kill_chain_stages":    hypotheses["kill_chain_stages"],
        "actor_profile":        hypotheses["h3_actor_profile"]["actor_profile"],
        "environment_context":  fusion_result.get("environment_context", {}),

        # Transparency
        "behavioral_iocs_used":      behavioral_iocs,
        "noise_filtered":            fusion_result.get("noise_filtered", []),
        "ioc_count_network_raw":    entities.get("ioc_count_network", 0),   # pure network IOCs
        "ioc_count_entities_raw":   entities.get("ioc_count", 0),           # all entities
        "ioc_count_for_mitre":      len(behavioral_iocs),                   # behaviorally relevant subset
        "entity_confidence_scores":  correlation_graph["entity_scores"],
        "high_confidence_entities":  resolution["high_conf_actors"],
        "unconfirmed_entities":      resolution["unconfirmed_actors"],
        "intelligence_gaps":         fusion_result.get("intelligence_gaps", []),
        "attribution_gaps":          hypotheses["h3_actor_profile"]["actor_profile"].get(
                                         "attribution_gaps", []
                                     ),
    }

Overwriting threat_brief.py


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
%%writefile __init__.py

from extract_entities import extract_entities
from rag_search import rag_search
from detect_patterns import detect_patterns
from detect_behaviors import detect_behaviors
from threat_brief import build_threat_brief

CTI_TOOLS = [
    extract_entities,   
    rag_search,        
    detect_patterns,   
    detect_behaviors,   
]

Writing __init__.py


In [20]:
%%writefile cti_analysis_agent.py
"""
CTI ANALYSIS AGENT — Entry Point
==================================
Orchestrates the four CTI tools in sequence, then synthesizes their outputs
into a structured threat_brief that the MITRE Agent can consume directly.

Pipeline:
  raw message
    → [extract_entities]  → IOCs and named threat entities
    → [detect_behaviors]  → malicious behavior signatures
    → [detect_patterns]   → statistical + semantic classification
    → [rag_search]        → similar historical CTI messages
    → [build_threat_brief] → narrative synthesis + noise filtering
    → threat_brief         → ready for MITRE Agent

No bugs were found in the original — this version adds English comments only.
"""

import json
from langsmith import traceable

from extract_entities import extract_entities
from detect_behaviors import detect_behaviors
from detect_patterns  import detect_patterns
from rag_search       import rag_search
from threat_brief     import build_threat_brief


@traceable(name="CTI Analysis — analyze_message")
def analyze_message(message: str) -> dict:
    """
    Analyze a raw CTI message and return a structured, contextualized threat brief.

    The threat_brief["mitre_input"] is the only field the MITRE Agent needs:
      - Contextualized behaviors (not raw IOCs)
      - Target environment (OS, sector, infrastructure criticality)
      - Actor profile (APT, opportunistic, or unknown)
      - Inferred kill chain stages

    Args:
        message: Raw text from a CTI source (Telegram, web scrape, report, etc.)

    Returns:
        dict with "threat_brief" (for MITRE Agent) + raw tool outputs (for debug/audit)
    """

    # ------------------------------------------------------------------
    # STEPS 1–4: Run the four CTI tools sequentially.
    # Each tool's output can inform downstream tools — behaviors feed the
    # pattern engine, patterns feed the synthesizer, etc.
    # ------------------------------------------------------------------

    print("[Step 1/5] Extracting entities (IOCs, malware, actors)...")
    entities = extract_entities.invoke({"message": message})

    print("[Step 2/5] Detecting malicious behaviors...")
    behaviors_result = detect_behaviors.invoke({"message": message})
    behaviors        = behaviors_result.get("behaviors_detected", [])

    print("[Step 3/5] Detecting patterns and classifying threat...")
    patterns_result = detect_patterns.invoke({"messages": [message]})

    print("[Step 4/5] Searching RAG database for similar threats...")
    rag_raw    = rag_search.invoke({"query": message[:300]})  # truncate long messages for vector search
    rag_result = json.loads(rag_raw) if isinstance(rag_raw, str) else rag_raw

    # ------------------------------------------------------------------
    # STEP 5: Synthesize — build the narrative and filter noise before
    # passing anything to the MITRE Agent.
    # ------------------------------------------------------------------

    print("[Step 5/5] Building threat brief (synthesizer)...")
    brief = build_threat_brief(
        entities    = entities,
        behaviors   = behaviors,
        patterns    = patterns_result,
        rag_context = rag_result,
    )

    print(
        f"[✓] CTI Analysis complete. "
        f"Network IOCs: {brief['ioc_count_network_raw']} | "
        f"Named entities: {brief['ioc_count_entities_raw']} | "
        f"MITRE-relevant: {brief['ioc_count_for_mitre']}"
    )

    # ------------------------------------------------------------------
    # ASSEMBLE FINAL RESULT
    # ------------------------------------------------------------------
    result = {
        "status": "success",

        # ── For the MITRE Agent ─────────────────────────────────────
        # Pass threat_brief["mitre_input"] — not raw entities
        "threat_brief": brief,

        # ── Backward compatibility (other components may read these) ─
        "entities":    entities,
        "behaviors":   behaviors,
        "rag_context": {
            "num_results": len(rag_result.get("results", [])),
            "results":     rag_result.get("results", []),
        },
        "patterns": {
            "threat_classification": patterns_result.get("threat_classification", ""),
            "emerging_terms":        patterns_result.get("emerging_terms", []),
            "semantic_summary":      patterns_result.get("semantic_summary", ""),
            "predicted_next_steps":  patterns_result.get("predicted_next_steps", []),
            "is_malicious":          patterns_result.get("is_malicious", False),
        },

        # ── Full tool outputs for debugging and auditing ─────────────
        "behaviors_full": behaviors_result,
        "patterns_full":  patterns_result,
    }

    # ------------------------------------------------------------------
    # RAGAS EVALUATION (fire-and-forget — runs after result is prepared
    # so it never blocks the main pipeline even if it fails)
    # ------------------------------------------------------------------
    try:
        from ragas_worker import submit_for_evaluation
        # thread_id injecté par graph.py → seul le thread cible est évalué
        _thread_id = result.get("_thread_id", "")
        submit_for_evaluation(
            job_id     = f"cti-{abs(hash(message)) % 100000}",
            cti_result = result,
            message    = message,
            thread_id  = _thread_id,
        )
    except Exception as e:
        print(f"[RAGAS] Evaluation submission skipped: {e}")

    return result

Writing cti_analysis_agent.py


In [21]:
# # Kaggle test notebook for cti_analysis_agent.py

# from cti_analysis_agent import analyze_message
# import json

# # Step 1: Provide a sample CTI message
# sample_message = """
# Suspicious phishing email detected targeting finance department.
# Message contains a malicious attachment and attempts credential harvesting.
# Observed outbound traffic to known C2 infrastructure.
# """
# result = analyze_message(sample_message)
# print(json.dumps(result, indent=2))


In [22]:
!pip install mitreattack-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.6/566.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.8/81.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.13.2 which is incompatible.


In [23]:
import mitreattack
print(f"✅ mitreattack-python version {mitreattack.__version__} is ready!")

✅ mitreattack-python version 5.4.1 is ready!


In [24]:
import os

def find_mitre_file():
    print("🔎 Searching for enterprise-attack.json...")
    for root, dirs, files in os.walk('/kaggle/input'):
        if "enterprise-attack.json" in files:
            actual_path = os.path.join(root, "enterprise-attack.json")
            print(f"✅ FOUND IT! Use this path in your code:")
            print(f"'{actual_path}'")
            return actual_path
    print("❌ NOT FOUND. Did you click 'Add Data'?")
    return None
correct_path = find_mitre_file()

🔎 Searching for enterprise-attack.json...
✅ FOUND IT! Use this path in your code:
'/kaggle/input/datasets/chaymadallel/mitre-attack-stix/enterprise-attack.json'


In [25]:
%%writefile mitre_mapper.py
"""
TOOL 1 (MITRE Layer) — MITRE ATT&CK Mapper
============================================
Maps detected malicious behaviors to official MITRE ATT&CK techniques.

Two-phase architecture:
  Phase 1 (STIX)  : Keyword search in the official MITRE ATT&CK STIX database,
                    followed by LLM batch-validation to remove false positives.
  Phase 2 (LLM)   : Contextual mapping for behaviors not covered by STIX.

FIXES vs original:
  1. DOUBLE STIX SCAN: The `uncovered` behaviors filter re-ran _search_mitre_stix()
     for every indicator of every behavior, duplicating all work already done by
     _collect_stix_candidates(). Fixed by building a behavior→technique coverage
     index during the first pass and reusing it.

  2. MISSING source_behavior FIELD: _collect_stix_candidates() never set
     source_behavior on candidates, so _llm_batch_verify() always logged "unknown"
     for the source, making the LLM validation context useless. Fixed by recording
     the originating behavior category when each candidate is created.
"""

import os
import json
from mitreattack.stix20 import MitreAttackData
from langchain_core.tools import tool
from llm_helper import llm_analyze


# =============================================================================
# STIX DATABASE LOADER
# =============================================================================

_mitre_data: MitreAttackData | None = None


def _get_mitre_data() -> MitreAttackData:
    """
    Lazily load the MITRE ATT&CK STIX database from disk.
    Subsequent calls return the cached instance (expensive to reload).
    """
    global _mitre_data
    if _mitre_data is None:
        path = "/kaggle/input/datasets/chaymadallel/mitre-attack-stix/enterprise-attack.json"
        if os.path.exists(path):
            print(f"[MITRE] Loading STIX database from: {path}")
            _mitre_data = MitreAttackData(path)
            print("[MITRE] STIX database loaded successfully.")
        else:
            raise FileNotFoundError(
                f"Critical Error: enterprise-attack.json not found at {path}"
            )
    return _mitre_data


# =============================================================================
# PHASE 1a — STIX KEYWORD SEARCH
# =============================================================================

def _search_mitre_stix(keyword: str, limit: int = 3) -> list[dict]:
    """
    Search the STIX database for ATT&CK techniques matching a keyword.

    Scans both technique name and description fields.
    Returns up to `limit` techniques with their ATT&CK ID, name, and tactic.
    """
    mitre        = _get_mitre_data()
    results      = []
    keyword_lower = keyword.lower()

    for tech in mitre.get_techniques():
        name        = tech.get("name", "").lower()
        description = tech.get("description", "").lower()

        if keyword_lower not in name and keyword_lower not in description:
            continue

        # Extract the external ATT&CK ID 
        external_id = next(
            (
                ref.get("external_id", "")
                for ref in tech.get("external_references", [])
                if ref.get("source_name") == "mitre-attack"
            ),
            "",
        )

        # Extract the primary tactic name
        tactic = next(
            (
                phase.get("phase_name", "").replace("-", " ").title()
                for phase in tech.get("kill_chain_phases", [])
                if phase.get("kill_chain_name") == "mitre-attack"
            ),
            "",
        )

        if external_id:
            results.append({
                "id":             external_id,
                "name":           tech.get("name", ""),
                "tactic":         tactic,
                "mapping_source": "mitre_stix",
            })

        if len(results) >= limit:
            break

    return results


def _collect_stix_candidates(behaviors: list[dict]) -> tuple[list[dict], dict[str, set[str]]]:
    """
    Run STIX keyword searches for all matched indicators across all behaviors.

    FIX: Now also builds and returns a behavior_coverage index mapping each
    behavior's category to the set of ATT&CK technique IDs it produced.
    This index is reused later to identify uncovered behaviors without
    re-scanning the full STIX database a second time.

    Returns:
        candidates        : Flat list of candidate technique dicts
        behavior_coverage : {behavior_category: set(technique_ids)} index
    """
    candidates        = []
    behavior_coverage = {} 

    for behavior in behaviors:
        category       = behavior.get("category", "unknown")
        found_tech_ids = set()

        for indicator in behavior.get("matched_indicators", [])[:2]:
            stix_results = _search_mitre_stix(indicator, limit=2)
            for result in stix_results:
                # FIX: record which behavior generated this candidate
                result["source_behavior"] = category
                candidates.append(result)
                found_tech_ids.add(result["id"])

        behavior_coverage[category] = found_tech_ids

    return candidates, behavior_coverage


# =============================================================================
# PHASE 1b — LLM BATCH VALIDATION
# =============================================================================

_VALIDATION_PROMPT = """You are a MITRE ATT&CK Validator.

Review the observed behaviors and the candidate technique mappings.

REJECTION RULES:
- Reject mappings for administrative or benign activity (routine login, backup, system check).
- Reject mappings where the technique does not match the described behavior.

Return ONLY valid JSON — no markdown, no explanation:
{"validated_ids": ["T1234", "T1059.001"]}"""


def _llm_batch_verify(candidates: list[dict], behaviors: list[dict]) -> list[dict]:
    """
    Ask the LLM to validate STIX candidates and reject false positives.

    Provides the LLM with both the observed behaviors AND the candidate mappings
    so it can make an informed accept/reject decision for each technique.

    Falls back to the first 3 candidates if the LLM call fails or returns nothing,
    rather than returning empty results and losing all coverage.
    """
    if not candidates:
        return []

    context = {
        "observed_behaviors": [
            {"desc": b.get("description"), "cat": b.get("category")}
            for b in behaviors
        ],
        "potential_mappings": [
            {
                "id":              c["id"],
                "name":            c["name"],
                "source_behavior": c.get("source_behavior", "unknown"), 
            }
            for c in candidates
        ],
    }

    result        = llm_analyze(_VALIDATION_PROMPT, json.dumps(context))
    validated_ids = result.get("validated_ids", [])

    if not validated_ids:
        return candidates[:3]

    return [c for c in candidates if c["id"] in validated_ids]


# =============================================================================
# PHASE 2 — LLM CONTEXTUAL MAPPING (for uncovered behaviors)
# =============================================================================

_MAPPING_PROMPT = """You are a MITRE ATT&CK mapping expert.

Given a list of malicious behaviors that were NOT matched by the STIX database,
map each to the most relevant MITRE ATT&CK technique(s).

STRICT RULES:
- Only use REAL technique IDs from the MITRE ATT&CK framework.
- "Dumping memory"  → T1003 (OS Credential Dumping), NOT T1059 (PowerShell)
- "Keylogger"       → T1056.001 (Keylogging),         NOT T1059
- If you are less than 85% confident, omit the mapping entirely.
- Never guess.

Return ONLY valid JSON — no markdown, no explanation:
{
    "mappings": [
        {
            "behavior": "original behavior description",
            "technique_id": "T1003.001",
            "technique_name": "LSASS Memory",
            "tactic": "Credential Access",
            "confidence": 0.9
        }
    ]
}"""


def _llm_mapping(behaviors: list[dict]) -> list[dict]:
    """
    Ask the LLM to map behaviors that produced no STIX hits.

    Only called for behaviors where _collect_stix_candidates() found no
    technique candidates (or none survived LLM validation).

    Returns a list of technique dicts tagged with mapping_source="llm".
    """
    if not behaviors:
        return []

    descriptions = [
        f"- {b.get('category', 'unknown')}: {b.get('description', '')}"
        for b in behaviors
    ]
    content = "Behaviors to map:\n" + "\n".join(descriptions)
    result  = llm_analyze(_MAPPING_PROMPT, content)

    if result.get("llm_failed") or result.get("parse_error"):
        return []

    return [
        {
            "id":              m.get("technique_id", ""),
            "name":            m.get("technique_name", ""),
            "tactic":          m.get("tactic", ""),
            "source_behavior": m.get("behavior", ""),
            "mapping_source":  "llm",
            "confidence":      m.get("confidence", 0.7),
        }
        for m in result.get("mappings", [])
        if m.get("technique_id")
    ]


# =============================================================================
# MERGE & DEDUPLICATION
# =============================================================================

def _merge_mappings(stix: list[dict], llm: list[dict]) -> list[dict]:
    """
    Merge STIX-validated and LLM-derived technique mappings.

    Deduplicates on technique ID. STIX results take priority — if both
    sources map to the same technique, the STIX entry is kept and its
    mapping_source field is updated to reflect both sources.
    """
    seen: dict[str, dict] = {}

    for mapping in stix + llm:
        tech_id = mapping.get("id", "")
        if not tech_id:
            continue

        if tech_id not in seen:
            seen[tech_id] = mapping
        else:
            # Both sources agree — record that for confidence downstream
            seen[tech_id]["mapping_source"] += f"+{mapping['mapping_source']}"

    return list(seen.values())


# =============================================================================
# LANGCHAIN TOOL ENTRY POINT
# =============================================================================

@tool
def map_to_mitre(behaviors: list[dict]) -> dict:
    """
    Map detected malicious behaviors to MITRE ATT&CK techniques.

    Phase 1: STIX keyword search → LLM validation (removes false positives).
    Phase 2: LLM contextual mapping for behaviors with no STIX coverage.
    Phase 3: Merge + deduplicate results from both phases.

    FIX: The uncovered-behavior detection now reuses the coverage index built
    during Phase 1 instead of re-running the full STIX scan a second time.

    Args:
        behaviors: List of behavior dicts from detect_behaviors tool

    Returns:
        {techniques, total_techniques, tactics_covered, coverage_summary}
    """
    if not behaviors:
        return {
            "techniques":       [],
            "total_techniques": 0,
            "error":            "No behaviors provided",
        }

    # --- PHASE 1: STIX CANDIDATE COLLECTION ---
    # FIX: _collect_stix_candidates now returns both the candidates AND the
    # behavior_coverage index, eliminating the need for a second STIX scan.
    stix_candidates, behavior_coverage = _collect_stix_candidates(behaviors)

    # --- PHASE 1b: LLM VALIDATION ---
    validated_stix  = _llm_batch_verify(stix_candidates, behaviors)
    covered_ids     = {c["id"] for c in validated_stix}

    # --- PHASE 2: LLM MAPPING FOR UNCOVERED BEHAVIORS ---
    # A behavior is "uncovered" if none of its STIX candidates survived validation.
    uncovered = [
        b for b in behaviors
        if not behavior_coverage.get(b.get("category", ""), set()).intersection(covered_ids)
    ]
    llm_results = _llm_mapping(uncovered)

    # --- PHASE 3: MERGE & REPORT ---
    techniques = _merge_mappings(validated_stix, llm_results)
    tactics    = sorted({t.get("tactic", "") for t in techniques if t.get("tactic")})

    return {
        "techniques":       techniques,
        "total_techniques": len(techniques),
        "tactics_covered":  tactics,
        "coverage_summary": (
            f"{len(techniques)} validated technique(s) across {len(tactics)} tactic(s)."
        ),
    }

Writing mitre_mapper.py


In [26]:
# from mitre_mapper import map_to_mitre as map_to_mitre_tool
# import json
# mock_behaviors = [
#     {"behavior": "Suspicious PowerShell execution", "category": "Execution"},
#     {"behavior": "Multiple failed logins", "category": "Credential Access"},
#     {"behavior": "Unusual outbound traffic to unknown IP", "category": "Command and Control"},
# ]
# results = map_to_mitre_tool.func(mock_behaviors)
# print(json.dumps(results, indent=2))


In [27]:
%%writefile severity_engine.py
"""
TOOL 2 (MITRE Layer) — Severity Engine
========================================
Calculates risk severity scores for MITRE ATT&CK techniques.

Two-phase architecture:
  Phase 1 (LLM): Score each technique individually on four axes.
  Phase 2 (LLM): Evaluate the full set for attack-chain coherence and award
                 a bonus (0.0–0.5) when a multi-stage kill-chain is confirmed.

Scoring axes (each 1.0–5.0):
  Impact         (35%) : Damage potential if the technique succeeds
  Exploitability (25%) : Ease of use in the wild (tooling, skill required)
  Prevalence     (20%) : How frequently observed in real threat actor activity
  Stealth        (20%) : Difficulty for defenders to detect
"""

from langchain_core.tools import tool
from llm_helper import llm_analyze


# =============================================================================
# SCORING WEIGHTS — must sum to 1.0
# =============================================================================

W_IMPACT     = 0.35
W_EXPLOIT    = 0.25
W_PREVALENCE = 0.20
W_STEALTH    = 0.20


def _compute_score(impact: float, exploit: float, prevalence: float, stealth: float) -> float:
    """
    Weighted severity score clamped to [0.0, 5.0].

    With valid 1.0–5.0 inputs and weights summing to 1.0, the natural output
    is [1.0, 5.0]. The clamp guards against malformed LLM responses (e.g. 0 or 6).
    """
    raw = W_IMPACT * impact + W_EXPLOIT * exploit + W_PREVALENCE * prevalence + W_STEALTH * stealth
    return round(min(5.0, max(0.0, raw)), 2)


def _classify(score: float) -> str:
    """Map a numeric severity score to a human-readable severity level."""
    if score >= 4.0:
        return "Critical"
    if score >= 3.0:
        return "High"
    if score >= 2.0:
        return "Medium"
    return "Low"


# =============================================================================
# PHASE 1 — LLM PER-TECHNIQUE SCORING
# =============================================================================

_SCORING_PROMPT = """You are an autonomous Cyber Risk Scoring Engine.

For each MITRE ATT&CK technique provided, assign scores on four axes
based on real-world threat intelligence — NOT generic defaults.

You will also receive environment context (target OS, sector, infrastructure
criticality, actor maturity). Use it to calibrate scores:
- Same technique scores HIGHER on critical infrastructure than on an isolated workstation.
- APT-level actor → boost stealth and exploitability.
- Opportunistic actor → reduce stealth, reduce prevalence for advanced techniques.

Axes (all 1.0 – 5.0, one decimal):
  imp : Impact         — damage potential if the technique succeeds
  exp : Exploitability — ease of execution / tooling availability in the wild
  pre : Prevalence     — frequency of observed use by threat actors
  ste : Stealth        — difficulty of detection by defenders

RULES:
- Score every technique individually. Do NOT group by tactic.
- Base scores on the specific sub-technique, not the parent tactic.
- If you are unsure, use 2.5 as a neutral mid-point rather than guessing high.
- Return ONLY valid JSON — no markdown, no explanation.

{
  "scores": [
    {
      "id":    "T1003.001",
      "imp":   4.5,
      "exp":   3.5,
      "pre":   4.0,
      "ste":   3.0,
      "reason": "LSASS dumping is trivial with Mimikatz and yields high-value credentials"
    }
  ]
}"""


def _llm_score_techniques(techniques: list[dict], env_context: dict) -> dict[str, dict]:
    """
    Ask the LLM to score every technique individually, calibrated to env_context.

    env_context keys used: target_os, target_sector, infrastructure_criticality,
    actor_maturity (injected by mitre_agent before calling calculate_severity).

    Returns a dict keyed by technique ID for O(1) lookup.
    Returns an empty dict if the LLM call fails.
    """
    env_lines = [
        f"  target_os:                 {env_context.get('target_os', 'unknown')}",
        f"  target_sector:             {', '.join(env_context.get('target_sector', [])) or 'unknown'}",
        f"  infrastructure_criticality:{env_context.get('infrastructure_criticality', 'unknown')}",
        f"  actor_maturity:            {env_context.get('actor_maturity', 'unknown')}",
    ]

    tech_lines = "\n".join(
        f"- {t.get('id')} | {t.get('name')} | tactic: {t.get('tactic', 'unknown')}"
        for t in techniques
    )

    content = (
        "Environment context:\n" + "\n".join(env_lines)
        + "\n\nTechniques to score:\n" + tech_lines
    )

    result = llm_analyze(_SCORING_PROMPT, content)

    if result.get("llm_failed") or result.get("parse_error"):
        return {}

    return {
        entry["id"]: entry
        for entry in result.get("scores", [])
        if entry.get("id")
    }


# =============================================================================
# PHASE 2 — LLM ATTACK CHAIN BONUS
# =============================================================================

_CHAIN_PROMPT = """You are a MITRE ATT&CK attack-chain analyst.

Given a set of techniques observed together, assess whether they form
a coherent, multi-stage attack chain that amplifies overall risk.

RULES:
- Only award a bonus when the techniques form a logical, multi-stage sequence.
- bonus = 0.0 for isolated or unrelated techniques.
- bonus range: 0.0 (none) – 0.5 (full kill-chain confirmed).
- Return ONLY valid JSON — no markdown, no explanation.

{"bonus": 0.3, "reason": "Credential dumping followed by lateral movement and ransomware deployment."}"""


def _llm_chain_bonus(techniques: list[dict]) -> tuple[float, str]:
    """
    Evaluate whether the observed techniques form a cohesive attack chain.

    A chain bonus (0.0–0.5) is added to each LLM-scored technique's final score
    when the techniques collectively demonstrate multi-stage attacker behavior.

    Returns:
        (bonus, reasoning) — bonus is 0.0 if chain analysis fails
    """
    content = "Observed techniques:\n" + "\n".join(
        f"- {t.get('id')} {t.get('name')} ({t.get('tactic', '')})"
        for t in techniques
    )

    result = llm_analyze(_CHAIN_PROMPT, content)

    if result.get("llm_failed") or result.get("parse_error"):
        return 0.0, "Chain analysis unavailable."

    bonus  = min(0.5, float(result.get("bonus", 0.0)))
    reason = result.get("reason", "")
    return bonus, reason


# =============================================================================
# LANGCHAIN TOOL ENTRY POINT
# =============================================================================

@tool
def calculate_severity(techniques: list[dict]) -> dict:
    """
    Calculate risk severity for a list of MITRE ATT&CK techniques.

    Phase 1: LLM scores each technique on four axes (impact, exploitability,
             prevalence, stealth), calibrated to the environment context injected
             by mitre_agent (target_os, sector, infrastructure_criticality,
             actor_maturity). Produces a weighted score in [0.0, 5.0].
    Phase 2: LLM evaluates the full set for attack-chain coherence and applies
             a bonus (0.0–0.5) when a multi-stage kill-chain is confirmed.

    Args:
        techniques: List of technique dicts (id, name, tactic) — each may carry
                    an `environment_context` dict injected by mitre_agent.py.

    Returns:
        {scored_techniques, global_severity_score, global_severity_level,
         chain_bonus_applied, overall_reasoning, unscored_techniques}
    """
    if not techniques:
        return {"error": "No techniques provided"}

    # Pull shared env_context from the first technique (all share the same context)
    env_context = techniques[0].get("environment_context", {}) if techniques else {}

    # --- PHASE 1: PER-TECHNIQUE SCORING (env-calibrated) ---
    score_map    = _llm_score_techniques(techniques, env_context)
    scored       = []
    unscored_ids = []

    for tech in techniques:
        tech_id = tech.get("id", "")
        entry   = score_map.get(tech_id)

        if entry:
            imp   = entry.get("imp", 2.5)
            exp   = entry.get("exp", 2.5)
            pre   = entry.get("pre", 2.5)
            ste   = entry.get("ste", 2.5)
            score = _compute_score(imp, exp, pre, ste)

            scored.append({
                **tech,
                "severity_details": {
                    "impact":         imp,
                    "exploitability": exp,
                    "prevalence":     pre,
                    "stealth":        ste,
                },
                "severity_score": score,
                "severity_level": _classify(score),
                "llm_reasoning":  entry.get("reason", ""),
                "scoring_source": "llm",
            })
        else:
            # LLM did not return a score for this technique — flag explicitly
            unscored_ids.append(tech_id)
            scored.append({
                **tech,
                "severity_score": None,
                "severity_level": "Unknown",
                "llm_reasoning":  "No LLM score returned for this technique.",
                "scoring_source": "none",
            })

    # --- PHASE 2: ATTACK CHAIN BONUS ---
    # Apply bonus only to techniques that were actually LLM-scored.
    chain_bonus, chain_reason = _llm_chain_bonus(techniques)

    if chain_bonus > 0:
        for item in scored:
            if item["scoring_source"] == "llm":
                item["severity_score"] = round(min(5.0, item["severity_score"] + chain_bonus), 2)
                item["severity_level"] = _classify(item["severity_score"])

    # --- GLOBAL SCORE: maximum individual score ---
    valid_scores = [s["severity_score"] for s in scored if s["severity_score"] is not None]
    global_score = round(max(valid_scores), 2) if valid_scores else 0.0

    return {
        "scored_techniques":     scored,
        "global_severity_score": global_score,
        "global_severity_level": _classify(global_score),
        "chain_bonus_applied":   chain_bonus,
        "overall_reasoning":     chain_reason,
        "unscored_techniques":   unscored_ids,
    }

Writing severity_engine.py


In [28]:
%%writefile mitigation_engine.py
"""
TOOL 3 (MITRE Layer) — Mitigation Engine
==========================================
Generates defense recommendations for each ATT&CK technique.

Two sources are combined per technique:
  Source 1 (STIX) : Official MITRE CourseOfAction objects from the STIX database
  Source 2 (LLM)  : Contextual CIS Controls + NIST 800-53 recommendations

FIX vs original:
  - In _llm_mitigations(), the f-string used t.get('severity_score', '?').
    When severity_score is None (unscored techniques), dict.get() returns None
    (the actually stored value), NOT the '?' fallback — the fallback only triggers
    when the KEY is absent, not when its value is None.
    Fixed using `t.get('severity_score') or '?'` which correctly falls back to
    '?' when the value is None (falsy) as well as when the key is absent.
"""

from langchain_core.tools import tool
from llm_helper import llm_analyze
from mitre_mapper import _get_mitre_data


# =============================================================================
# SOURCE 1 — OFFICIAL STIX MITIGATIONS
# =============================================================================

def _get_stix_mitigations(technique_id: str) -> list[str]:
    """
    Retrieve official MITRE ATT&CK mitigations for a technique from the STIX database.

    Steps:
    1. Resolve the ATT&CK external ID (e.g. "T1003") to a STIX object ID.
    2. Fetch all CourseOfAction objects related to that technique.
    3. Return formatted strings like "M1042: Disable or Remove Feature or Capability".

    Returns an empty list if the technique is not found or an error occurs.
    """
    mitre      = _get_mitre_data()
    mitigations = []

    try:
        # Step 1: Find the STIX internal ID for this ATT&CK technique
        tech_stix_id = None
        for t in mitre.get_techniques():
            for ref in t.get("external_references", []):
                if ref.get("external_id") == technique_id:
                    tech_stix_id = t.get("id")
                    break
            if tech_stix_id:
                break

        if not tech_stix_id:
            print(f"[STIX] No STIX ID found for {technique_id}")
            return []

        # Step 2: Fetch all CourseOfAction objects linked to this technique
        for item in mitre.get_mitigations_mitigating_technique(tech_stix_id):
            coa    = item.get("object")
            name   = coa.get("name", "")
            ext_id = next(
                (
                    ref.get("external_id", "")
                    for ref in coa.get("external_references", [])
                    if ref.get("source_name") == "mitre-attack"
                ),
                "",
            )
            if name:
                # Format: "M1042: Disable or Remove Feature" or just the name
                label = f"{ext_id}: {name}" if ext_id else name
                mitigations.append(label)

        print(f"[STIX] {technique_id} → {len(mitigations)} mitigation(s): {mitigations}")

    except Exception as e:
        print(f"[STIX ERROR] {technique_id} → {e}")

    return mitigations


# =============================================================================
# SOURCE 2 — LLM ENRICHMENT (CIS Controls + NIST 800-53)
# =============================================================================

_LLM_MITIGATION_PROMPT = """You are a Defense Expert specialized in MITRE ATT&CK mitigations.

For each technique provided, produce 2–3 actionable mitigation steps.
Each step MUST reference at least one of: MITRE M-code, CIS Control, or NIST 800-53 control.
Tailor recommendations to the specific technique — do NOT give generic tactic-level advice.

Priority values: "immediate" | "short_term" | "long_term"

Return ONLY valid JSON — no markdown, no explanation:
{
  "mitigations": [
    {
      "technique_id": "T1003.001",
      "recommendations": [
        "M1043: Enable Credential Guard to protect LSASS memory (Hardening)",
        "CIS 6.3: Restrict admin tool access via Just-In-Time PAM (Preventative)",
        "NIST AC-6: Apply least-privilege to accounts with SeDebugPrivilege (Policy)"
      ],
      "priority": "immediate"
    }
  ],
  "global_recommendations": [
    "Deploy EDR with memory protection on all endpoints (CIS 10.1)",
    "Enforce MFA across all privileged accounts (NIST IA-5)"
  ]
}"""


def _llm_mitigations(techniques: list[dict]) -> dict:
    """
    Generate CIS/NIST-referenced mitigation recommendations via LLM.

    A single batched call covers all techniques to minimize LLM round-trips.

    FIX: severity_score can be None for unscored techniques.
    Using `t.get('severity_score') or '?'` correctly handles:
      - Key absent      → returns None → 'or '?'' activates → shows '?'
      - Key present, value is None → 'or '?'' activates    → shows '?'
      - Key present, value is 0.0  → 'or '?'' activates    → shows '?' (0 is falsy)
    Note: a score of 0.0 is theoretically impossible given our 1.0–5.0 range,
    so treating it as missing is safe here.

    Returns the parsed LLM result, or an empty fallback on failure.
    """
    content = "\n".join(
        # FIX: use `or '?'` instead of get(..., '?') to handle None values
        f"- {t['id']} | {t.get('name', '')} | tactic: {t.get('tactic', 'unknown')}"
        f" | severity: {t.get('severity_score') or '?'} ({t.get('severity_level', '?')})"
        for t in techniques
    )

    result = llm_analyze(_LLM_MITIGATION_PROMPT, content)

    if result.get("llm_failed") or result.get("parse_error"):
        return {"mitigations": [], "global_recommendations": []}

    return result


# =============================================================================
# LANGCHAIN TOOL ENTRY POINT
# =============================================================================

@tool
def recommend_mitigations(scored_techniques: list[dict]) -> dict:
    """
    Generate mitigations for each scored ATT&CK technique.

    Combines:
    - Official MITRE STIX mitigations (CourseOfAction objects, per technique)
    - LLM contextual recommendations with CIS Controls + NIST 800-53 references

    Results are sorted by descending severity. Techniques marked Critical or High
    have their top-2 mitigations promoted to a priority_actions list for executive reports.

    Args:
        scored_techniques: Output from calculate_severity (must include
                           id, name, tactic, severity_score, severity_level)

    Returns:
        {technique_mitigations, global_recommendations, priority_actions}
    """
    if not scored_techniques:
        return {"error": "No techniques provided"}

    # --- SOURCE 2: LLM (single batched call for all techniques) ---
    llm_result = _llm_mitigations(scored_techniques)
    llm_map    = {
        m["technique_id"]: m
        for m in llm_result.get("mitigations", [])
        if m.get("technique_id")
    }

    # --- SOURCE 1 + 2: MERGE PER TECHNIQUE ---
    technique_mitigations = []

    for tech in scored_techniques:
        tech_id  = tech.get("id", "")

        # Fetch official STIX mitigations for this technique
        stix_mits = _get_stix_mitigations(tech_id)

        # Fetch LLM recommendations for this technique
        llm_entry = llm_map.get(tech_id, {})
        llm_mits  = llm_entry.get("recommendations", [])
        priority  = llm_entry.get("priority", "short_term")

        # Merge and deduplicate across both sources (case-insensitive)
        seen     = set()
        all_mits = []
        for mit in stix_mits + llm_mits:
            key = mit.lower()
            if key not in seen:
                seen.add(key)
                all_mits.append(mit)

        technique_mitigations.append({
            "id":             tech_id,
            "name":           tech.get("name", ""),
            "tactic":         tech.get("tactic", ""),
            "severity_score": tech.get("severity_score"),
            "severity_level": tech.get("severity_level", ""),
            "mitigations":    all_mits,
            "priority":       priority,
            "sources": {
                "stix_count": len(stix_mits),
                "llm_count":  len(llm_mits),
            },
        })

    # Sort by descending severity (techniques with None scores go last)
    technique_mitigations.sort(
        key=lambda t: t["severity_score"] if t["severity_score"] is not None else -1,
        reverse=True,
    )

    # Build priority actions: top-2 mitigations from Critical/High techniques only
    priority_actions = [
        f"[{tm['id']}] {mit}"
        for tm in technique_mitigations
        if tm["severity_level"] in ("Critical", "High")
        for mit in tm["mitigations"][:2]
    ]

    return {
        "technique_mitigations":  technique_mitigations,
        "global_recommendations": llm_result.get("global_recommendations", []),
        "priority_actions":       priority_actions[:10],  # cap at 10 for executive readability
    }

Writing mitigation_engine.py


In [29]:
%%writefile mitre_agent.py
"""
MITRE ATT&CK AGENT
===================
Receives the threat_brief from the CTI Agent and orchestrates:
  1. map_to_mitre          → MITRE ATT&CK technique mapping
  2. calculate_severity    → context-calibrated risk scores
  3. recommend_mitigations → CIS/NIST/STIX defense recommendations

Input contract (from build_threat_brief → mitre_input):
  behaviors           : Fusion-enriched behaviors with attack_technique_hint T-codes
  actor_context       : Validated actor, maturity, double_extortion, disruption_risk
  kill_chain_stages   : Ordered list of observed ATT&CK stages
  urgency             : CRITICAL | HIGH | MEDIUM | LOW
  intelligence_gaps   : What NOT to hallucinate
  behavioral_iocs     : Malware/tool/CVE names for STIX keyword search
  resolved_confidence : Overall analysis confidence score
"""

from langsmith import traceable

from mitre_mapper import map_to_mitre
from severity_engine import calculate_severity
from mitigation_engine import recommend_mitigations


def _build_env_context(threat_brief: dict, mitre_input: dict) -> dict:
    """
    Build the enriched environment context injected into severity scoring.

    Merges the fusion LLM's environment_context with actor intelligence from
    mitre_input so the severity engine can calibrate scores on two axes:
      - Infrastructure: target_os, sector, criticality
      - Actor:          maturity, disruption_risk

    Priority: mitre_input actor fields > threat_brief environment fields.
    """
    base = dict(threat_brief.get("environment_context", {}))

    actor_ctx = mitre_input.get("actor_context", {})
    base["actor_maturity"]    = actor_ctx.get("actor_maturity",  base.get("actor_maturity",  "unknown"))
    base["disruption_risk"]   = actor_ctx.get("disruption_risk", "unknown")
    base["urgency"]           = mitre_input.get("urgency",        "unknown")
    base["resolved_confidence"] = mitre_input.get("resolved_confidence", 0.5)

    return base


def _enrich_behaviors_with_hints(behaviors: list[dict]) -> list[dict]:
    """
    Forward attack_technique_hint T-codes from fusion output to map_to_mitre.

    The fusion LLM already assigned T-code hints (e.g. "T1003.001 — LSASS Memory")
    to each behavior. Forwarding these as additional matched_indicators gives the
    STIX keyword scanner strong seeds, dramatically reducing missed mappings.

    The original matched_indicators are preserved — hints are appended only.
    """
    enriched = []
    for b in behaviors:
        hint = b.get("attack_technique_hint", "")
        b_copy = dict(b)

        if hint:
            # Append the T-code + name as an extra indicator for STIX scanning
            existing = list(b_copy.get("matched_indicators", []))
            if hint not in existing:
                existing.append(hint)
            b_copy["matched_indicators"] = existing

        enriched.append(b_copy)
    return enriched

def extract_behaviors_from_cti(cti_result: dict) -> list:
    """
    Robustly extract behavior objects from different possible CTI result schemas.
    """

    if not isinstance(cti_result, dict):
        return []

    candidates = []

    # Common locations
    candidates.append(cti_result.get("behaviors"))

    candidates.append(
        cti_result.get("malicious_behaviors", {}).get("behaviors")
        if isinstance(cti_result.get("malicious_behaviors"), dict)
        else None
    )

    candidates.append(
        cti_result.get("behavior_analysis", {}).get("behaviors")
        if isinstance(cti_result.get("behavior_analysis"), dict)
        else None
    )

    candidates.append(
        cti_result.get("behavior_detection", {}).get("behaviors")
        if isinstance(cti_result.get("behavior_detection"), dict)
        else None
    )

    candidates.append(
        cti_result.get("mitre_input", {}).get("behaviors")
        if isinstance(cti_result.get("mitre_input"), dict)
        else None
    )

    candidates.append(
        cti_result.get("fusion_analysis", {}).get("mitre_relevant_behaviors")
        if isinstance(cti_result.get("fusion_analysis"), dict)
        else None
    )

    candidates.append(
        cti_result.get("llm_fusion_analysis", {}).get("mitre_relevant_behaviors")
        if isinstance(cti_result.get("llm_fusion_analysis"), dict)
        else None
    )

    # Pick first non-empty list
    for candidate in candidates:
        if isinstance(candidate, list) and len(candidate) > 0:
            return candidate

    # Deep fallback search
    def recursive_find_behaviors(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key in {"behaviors", "mitre_relevant_behaviors"} and isinstance(value, list) and value:
                    return value
                found = recursive_find_behaviors(value)
                if found:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = recursive_find_behaviors(item)
                if found:
                    return found
        return []

    return recursive_find_behaviors(cti_result)
    
@traceable(name="MITRE Agent — run_mitre_analysis")
def run_mitre_analysis(cti_result: dict) -> dict:
    """
    Orchestrate the full MITRE ATT&CK pipeline from the CTI analysis result.

    Args:
        cti_result: Output of analyze_message() — must contain "threat_brief"
                    with a populated "mitre_input" sub-dict.

    Returns:
        Full MITRE analysis: techniques, severity, mitigations, executive summary.
    """
    threat_brief = cti_result.get("threat_brief", {})
    mitre_input  = threat_brief.get("mitre_input", {})

    # ------------------------------------------------------------------
    # INPUT VALIDATION & FALLBACK
    # ------------------------------------------------------------------
    behaviors = mitre_input.get("behaviors", [])
    
    # Check if behaviors have technique hints (proper fusion output)
    has_hints = any(b.get("attack_technique_hint") for b in behaviors)
    
    if not behaviors or not has_hints:
        raw_behaviors = cti_result.get("behaviors", [])
        if raw_behaviors:
            # Convert raw behaviors to mitre_input format
            behaviors = [
                {
                    "category": b.get("category", ""),
                    "description": b.get("description", ""),
                    "attack_technique_hint": "",
                    "confidence": b.get("confidence", 0.7),
                    "evidence": [b.get("detection_source", "")],
                    "key_indicators": b.get("indicators", b.get("matched_indicators", []))[:3],
                    "matched_indicators": b.get("indicators", b.get("matched_indicators", [])),
                    "detection_source": b.get("detection_source", "llm_only"),
                    "severity": b.get("severity", "medium"),
                }
                for b in raw_behaviors
            ]
            print(f"[MITRE Agent] Reconstructed {len(behaviors)} behaviors from raw output")
    if not behaviors:
        print("[MITRE Agent] ❌ Aucun behavior trouvé nulle part — pipeline CTI incomplet")
        return {
            "status": "no_behaviors",
            "message": "Le CTI agent n'a produit aucun behavior détectable.",
            "techniques_mapped": {"techniques": [], "total_techniques": 0},
            "severity_analysis": {"global_severity_level": "Unknown", "global_severity_score": 0.0},
            "mitigations": {},
            "executive_summary": {}
        }
    # Build enriched environment context (infra + actor intelligence)
    env_context = _build_env_context(threat_brief, mitre_input)

    # ------------------------------------------------------------------
    # STEP 1: ATT&CK MAPPING
    # Forward fusion T-code hints as STIX keyword seeds.
    # ------------------------------------------------------------------
    enriched_behaviors = _enrich_behaviors_with_hints(behaviors)

    print(f"[MITRE 1/3] Mapping {len(enriched_behaviors)} behavior(s) to ATT&CK techniques...")
    mapping_result = map_to_mitre.invoke({"behaviors": enriched_behaviors})
    techniques     = mapping_result.get("techniques", [])

    if not techniques:
        print("[MITRE 1/3] ⚠️  No techniques mapped.")
        return {
            "status":         "no_techniques",
            "message":        "No ATT&CK techniques identified.",
            "mapping_result": mapping_result,
            "severity":       {},
            "mitigations":    {},
        }

    print(
        f"[MITRE 1/3] ✅ {mapping_result.get('total_techniques', 0)} technique(s) mapped "
        f"across {len(mapping_result.get('tactics_covered', []))} tactic(s)."
    )

    # ------------------------------------------------------------------
    # STEP 2: SEVERITY SCORING
    # Inject enriched env_context (OS + sector + criticality + actor maturity
    # + urgency) so the LLM calibrates scores to the specific threat context.
    # ------------------------------------------------------------------
    print(f"[MITRE 2/3] Calculating severity for {len(techniques)} technique(s)...")

    techniques_with_context = [{**t, "environment_context": env_context} for t in techniques]
    severity_result         = calculate_severity.invoke({"techniques": techniques_with_context})
    scored_techniques       = severity_result.get("scored_techniques", [])

    print(
        f"[MITRE 2/3] ✅ Global severity: "
        f"{severity_result.get('global_severity_level', '?')} "
        f"({severity_result.get('global_severity_score', 0)}/5.0) "
        f"| Chain bonus: {severity_result.get('chain_bonus_applied', 0.0)}"
    )

    # ------------------------------------------------------------------
    # STEP 3: MITIGATIONS
    # env_context injected so recommendations are OS/sector-specific.
    # ------------------------------------------------------------------
    print("[MITRE 3/3] Generating mitigations...")

    scored_with_context = [{**t, "environment_context": env_context} for t in scored_techniques]
    mitigation_result   = recommend_mitigations.invoke({"scored_techniques": scored_with_context})

    print(
        f"[MITRE 3/3] ✅ {len(mitigation_result.get('priority_actions', []))} "
        f"priority action(s) generated."
    )

    # ------------------------------------------------------------------
    # FINAL RESULT
    # ------------------------------------------------------------------
    actor_ctx = mitre_input.get("actor_context", {})

    return {
        "status": "success",

        # ── Intelligence context ────────────────────────────────────────
        "attack_narrative":   threat_brief.get("attack_narrative", ""),
        "kill_chain_stages":  mitre_input.get("kill_chain_stages",  threat_brief.get("kill_chain_stages", [])),
        "actor_profile":      threat_brief.get("actor_profile",     {}),
        "actor_context":      actor_ctx,
        "environment_context": env_context,
        "attack_maturity":    mitre_input.get("attack_maturity",    "unknown"),
        "urgency":            mitre_input.get("urgency",            "unknown"),
        # Surface intelligence gaps so consumers know what is uncertain
        "intelligence_gaps":  mitre_input.get("intelligence_gaps",  []),
        "attribution_gaps":   mitre_input.get("attribution_gaps",   []),

        # ── ATT&CK results ─────────────────────────────────────────────
        "techniques_mapped":  mapping_result,
        "severity_analysis":  severity_result,
        "mitigations":        mitigation_result,

        # ── Executive summary ───────────────────────────────────────────
        "executive_summary": {
            "total_techniques":    mapping_result.get("total_techniques",     0),
            "tactics_covered":     mapping_result.get("tactics_covered",      []),
            "global_severity":     severity_result.get("global_severity_level", "Unknown"),
            "global_score":        severity_result.get("global_severity_score",  0.0),
            "chain_bonus":         severity_result.get("chain_bonus_applied",    0.0),
            "priority_actions":    mitigation_result.get("priority_actions",     []),
            "unscored_techniques": severity_result.get("unscored_techniques",    []),
            # Actor summary for executive reports
            "actor":               actor_ctx.get("validated_actor", threat_brief.get("actor_profile", {}).get("likely_actor", "unknown")),
            "double_extortion":    actor_ctx.get("double_extortion", False),
            "disruption_risk":     actor_ctx.get("disruption_risk",  "unknown"),
            "rag_confirmed":       actor_ctx.get("rag_confirmed",     False),
        },
    }

Writing mitre_agent.py


In [30]:
%%writefile aggregator.py
"""
==========================================
 AGGREGATOR — Final Report Orchestrator
==========================================
Report sections:
  1. Summary & Context        (narrative + classification)
  2. Key Takeaways            (actor, vector, kill chain, environment)
  3. Severity                 (score, level, chain bonus)
  4. TTPs                     (techniques linked to behaviors)
  5. IOCs & Artifacts         (clean, filtered)
  6. Mitigation & Remediation (priority actions + global recs)
"""

from datetime import datetime, timezone
from collections import Counter
from langsmith import traceable
import re


# ---------------------------------------------------------------------------
#  CONSTANTS
# ---------------------------------------------------------------------------

_LLM_CATEGORY_MAP = {
    "process manipulation":   "process_creation",
    "persistence mechanisms": "registry_modification",
    "network activity":       "network_communication",
    "file system operations": "file_system_activity",
    "defense evasion":        "defense_evasion",
    "discovery":              "discovery",
    "lateral movement":       "lateral_movement",
    "impact":                 "file_system_activity",
    "credential theft":       "credential_access",
    "credential access":      "credential_access",
}

LLM_CATEGORY_MAP = _LLM_CATEGORY_MAP  # kept for external imports

_TERMINAL_STAGES = {"impact", "exfiltration", "command_and_control"}
_SEV_ORDER = {"Critical": 4, "High": 3, "Medium": 2, "Low": 1, "Unknown": 0}

_OBJECTIVE_TO_VECTOR = {
    "ransomware_extortion":        "Ransomware delivery (see TTPs for entry vector)",
    "data_exfiltration_espionage": "Targeted intrusion / spear-phishing (see TTPs)",
    "financial_theft":             "Financial fraud / credential theft (see TTPs)",
}
_OBJECTIVE_TO_IMPACT = {
    "ransomware_extortion":        "Ransomware / Data Encryption",
    "data_exfiltration_espionage": "Data Exfiltration / Espionage",
    "financial_theft":             "Financial Theft / Fraud",
}

_KILL_CHAIN_ORDER = [
    "initial_access", "execution", "persistence", "privilege_escalation",
    "defense_evasion", "credential_access", "discovery", "lateral_movement",
    "collection", "exfiltration", "command_and_control", "impact",
]

def _kc_sort_key(tactic: str) -> int:
    norm = tactic.lower().replace(" ", "_").replace("-", "_").replace("&", "and")
    return _KILL_CHAIN_ORDER.index(norm) if norm in _KILL_CHAIN_ORDER else 99


def _format_priority_actions(raw_actions: list) -> list:
    """
    Convert flat priority action strings into {text, refs} dicts.
    Extracts standard control references so PDF is pure display.
    """
    result = []
    for action in raw_actions:
        refs = re.findall(r"(?:M\d{4}|CIS\s[\d.]+|NIST\s[\w-]+)", str(action))
        result.append({
            "text": action,
            "refs": ", ".join(refs) if refs else "Best Practice",
        })
    return result


def _format_ioc_tables(iocs: dict) -> list:
    """
    Convert the nested IOC dict into a flat list of {title, rows} dicts
    ready for direct table rendering — no logic needed in pdf_report.
    """
    tables = []

    network_rows = [
        (k.upper(), v)
        for k, vals in iocs.get("network", {}).items()
        for v in vals
        if isinstance(v, str) and v.strip()
    ]
    if network_rows:
        tables.append({"title": "Network Indicators", "rows": network_rows})

    file_rows = [
        (k.upper(), v)
        for k, vals in iocs.get("files", {}).items()
        for v in vals
        if isinstance(v, str) and v.strip()
    ]
    if file_rows:
        tables.append({"title": "Host Indicators (Hashes)", "rows": file_rows})

    crypto_rows = [
        (k.upper(), v)
        for k, vals in iocs.get("crypto", {}).items()
        for v in vals
        if isinstance(v, str) and v.strip()
    ]
    if crypto_rows:
        tables.append({"title": "Cryptocurrency Wallets", "rows": crypto_rows})

    return tables
# ---------------------------------------------------------------------------
#  UTILITIES
# ---------------------------------------------------------------------------

def _normalize_category(category: str) -> str:
    lower = category.lower().strip()
    return _LLM_CATEGORY_MAP.get(lower, lower.replace(" ", "_"))


def _deduplicate_ordered(items: list) -> list:
    seen, out = set(), []
    for item in items:
        key = str(item).lower().strip()
        if key and key not in seen:
            seen.add(key)
            out.append(item)
    return out


def _classify_score(score: float) -> str:
    if score >= 4.0: return "Critical"
    if score >= 3.0: return "High"
    if score >= 2.0: return "Medium"
    return "Low"


# ---------------------------------------------------------------------------
#  IOC CLEANING
# ---------------------------------------------------------------------------

def _clean_iocs(entities: dict) -> dict:
    def _clean(lst):
        return _deduplicate_ordered([x for x in lst if isinstance(x, str) and x.strip()])

    wallets = entities.get("crypto_wallets", {})
    hashes  = entities.get("hashes", {})
    return {
        "network": {k: _clean(entities.get(k, [])) for k in ("ips", "urls", "domains")},
        "files":   {k: _clean(hashes.get(k, [])) for k in ("md5", "sha1", "sha256")},
        "crypto":  {k: _clean(wallets.get(k, [])) for k in ("btc", "eth", "xmr")},
        "threat_intelligence": {
            k: _clean(entities.get(k, []))
            for k in ("malware_names", "ransomware_groups", "apt_groups",
                      "tools_abused", "cves", "campaign_names")
        },
    }


# ---------------------------------------------------------------------------
#  SECTION 2 — KEY TAKEAWAYS
# ---------------------------------------------------------------------------

def _build_key_takeaways(cti_result: dict, global_severity: str) -> dict:
    threat_brief  = cti_result.get("threat_brief", {})
    patterns      = cti_result.get("patterns", {})
    actor_profile = threat_brief.get("actor_profile", {})
    env_context   = threat_brief.get("environment_context", {})
    kill_chain    = threat_brief.get("kill_chain_stages", [])
    objective     = (threat_brief.get("fusion", {})
                                 .get("hypotheses", {})
                                 .get("h1_objective", {})
                                 .get("assessment", "unknown"))

    if "initial_access" in kill_chain:
        vector = "Initial Access detected (see TTPs)"
    elif patterns.get("threat_classification", "").upper() in ("PHISHING", "SPEAR_PHISHING"):
        vector = "Phishing / Social Engineering"
    else:
        vector = _OBJECTIVE_TO_VECTOR.get(objective, "Unknown (see TTPs)")

    impact = next(
        (s.replace("_", " ").title() for s in reversed(kill_chain) if s in _TERMINAL_STAGES),
        _OBJECTIVE_TO_IMPACT.get(objective)
        or ("Critical impact — see severity" if global_severity == "Critical" else "Unknown"),
    )

    return {
        "threat_actor": {
            "name":       actor_profile.get("likely_actor", "Unknown"),
            "confidence": actor_profile.get("confidence", 0.0),
        },
        "initial_access_vector": vector,
        "impact":                impact,
        "target_environment": {
            "os":          env_context.get("target_os", "unknown"),
            "sector":      env_context.get("target_sector", []),
            "criticality": env_context.get("infrastructure_criticality", "unknown"),
        },
        "kill_chain_stages":    kill_chain,
        "attack_maturity":      threat_brief.get("mitre_input", {}).get("attack_maturity", "unknown"),
        "predicted_next_steps": patterns.get("predicted_next_steps", []),
    }


# ---------------------------------------------------------------------------
#  SECTION 4 — TTP LINKER
# ---------------------------------------------------------------------------

def _build_behavior_index(behaviors: list[dict]) -> dict[str, dict]:
    return {_normalize_category(b.get("category", "")): b for b in behaviors}


def _find_linked_behavior(tech, behavior_by_cat, brief_by_cat, behaviors):
    src = _normalize_category(tech.get("source_behavior", ""))
    if src and src in behavior_by_cat:
        return behavior_by_cat[src], "source_behavior"

    tactic_norm = _normalize_category(tech.get("tactic", ""))
    if tactic_norm in behavior_by_cat:
        return behavior_by_cat[tactic_norm], "tactic_category"

    tech_text = (tech.get("name", "") + " " + tech.get("tactic", "")).lower()
    for cat_key, brief_b in brief_by_cat.items():
        if cat_key in tech_text:
            return brief_b, "threat_brief_keyword"

    if behaviors:
        sev = {"critical": 4, "high": 3, "medium": 2, "low": 1}
        pool = [b for b in behaviors if b.get("detection_source") in ("static_regex", "static+llm")] or behaviors
        return max(pool, key=lambda b: sev.get(b.get("severity", "low"), 0)), "fallback"

    return None, "none"


def _link_techniques_to_behaviors(scored_techniques, behaviors, mitre_relevant_behaviors):
    behavior_by_cat = _build_behavior_index(behaviors)
    brief_by_cat    = _build_behavior_index(mitre_relevant_behaviors)

    ttps = []
    for tech in scored_techniques:
        linked, link_method = _find_linked_behavior(tech, behavior_by_cat, brief_by_cat, behaviors)
        ttps.append({
            "technique_id":   tech.get("id", ""),
            "technique_name": tech.get("name", ""),
            "tactic":         tech.get("tactic", ""),
            "mapping_source": tech.get("mapping_source", ""),
            "severity_score": tech.get("severity_score"),
            "severity_level": tech.get("severity_level", "Unknown"),
            "llm_reasoning":  tech.get("llm_reasoning", ""),
            "observed_behavior": {
                "category":    linked.get("category", "")         if linked else "",
                "description": linked.get("description", "")      if linked else "",
                "source":      linked.get("detection_source", "") if linked else "",
                "confidence":  linked.get("confidence", 0)        if linked else 0,
                "indicators":  linked.get("matched_indicators", [])[:3] if linked else [],
            },
            "link_method": link_method,
        })

    ttps.sort(
        key=lambda t: (_SEV_ORDER.get(t["severity_level"], 0), t["severity_score"] or 0),
        reverse=True,
    )
    return ttps


# ---------------------------------------------------------------------------
#  MAIN ORCHESTRATOR
# ---------------------------------------------------------------------------

@traceable(name="Aggregator — build_report_metadata")
def build_report_metadata(
    original_message: str,
    cti_result: dict,
    mitre_result: dict,
) -> dict:
    """
    3 params instead of 7. All sub-fields extracted internally.
    """
    if not cti_result or not mitre_result:
        return {"error": "CTI result or MITRE result missing"}

    # ── Single unpack point ──────────────────────────────────────────────────
    threat_brief      = cti_result.get("threat_brief", {})
    entities          = cti_result.get("entities", {})
    behaviors         = cti_result.get("behaviors", [])
    patterns          = cti_result.get("patterns", {})
    severity_result   = mitre_result.get("severity_analysis", {})
    mapping_result    = mitre_result.get("techniques_mapped", {})
    mitigation_result = mitre_result.get("mitigations", {})

    scored_techniques        = severity_result.get("scored_techniques", [])
    mitre_relevant_behaviors = threat_brief.get("mitre_input", {}).get("behaviors", [])
    env_context              = threat_brief.get("environment_context", {})
    global_severity          = severity_result.get("global_severity_level", "Unknown")

    # ── Sections ─────────────────────────────────────────────────────────────
    s1 = {
        "raw_message":           original_message,
        "attack_narrative":      threat_brief.get("attack_narrative", ""),
        "kill_chain_narrative":  threat_brief.get("kill_chain_narrative", ""),
        "threat_classification": patterns.get("threat_classification", ""),
        "is_malicious":          patterns.get("is_malicious", False),
        "semantic_summary":      patterns.get("semantic_summary", ""),
        "rag_context_used":      cti_result.get("rag_context", {}).get("num_results", 0),
    }

    s2 = _build_key_takeaways(cti_result, global_severity)

    s3 = {
        "global_score":        severity_result.get("global_severity_score", 0.0),
        "global_level":        global_severity,
        "chain_bonus_applied": severity_result.get("chain_bonus_applied", 0.0),
        "overall_reasoning":   severity_result.get("overall_reasoning", ""),
        "distribution":        dict(Counter(
            t.get("severity_level", "Unknown") for t in scored_techniques
        )),
        "environment_context": env_context,
    }

    s4 = _link_techniques_to_behaviors(scored_techniques, behaviors, mitre_relevant_behaviors)
    s4.sort(key=lambda t: _kc_sort_key(t.get("tactic", "")))
    # Add procedure_text to each TTP so pdf_report doesn't assemble strings
    for ttp in s4:
        obs = ttp.get("observed_behavior", {})
        ttp["procedure_text"] = (
            f"{obs.get('description', '')} {ttp.get('llm_reasoning', '')}".strip()
            or "Behavior observed."
        )
    s5 = _clean_iocs(entities)
    s5["_stats"] = {
        "total_network_iocs": sum(len(s5["network"][k]) for k in ("ips", "urls", "domains")),
        "total_file_iocs":    sum(len(s5["files"][k])   for k in ("md5", "sha1", "sha256")),
        "total_cves":         len(s5["threat_intelligence"]["cves"]),
        "noise_filtered_for_mitre": threat_brief.get("noise_filtered", []),
    }
    s5["ioc_tables"] = _format_ioc_tables(s5)
    s6 = {
        "environment_grounding": {
            "target_os":     env_context.get("target_os", "unknown"),
            "target_sector": env_context.get("target_sector", []),
            "criticality":   env_context.get("infrastructure_criticality", "unknown"),
        },
        "priority_actions": _deduplicate_ordered(
            mitigation_result.get("priority_actions", [])
        )[:10],
        "global_recommendations": _deduplicate_ordered(
            mitigation_result.get("global_recommendations", [])
        ),
        "by_technique": [
            {
                "technique_id":   tm.get("id", ""),
                "technique_name": tm.get("name", ""),
                "severity_level": tm.get("severity_level", ""),
                "priority":       tm.get("priority", "short_term"),
                "mitigations":    tm.get("mitigations", []),
            }
            for tm in mitigation_result.get("technique_mitigations", [])
        ],
    }
    # Pre-format priority actions with extracted control refs
    s6["priority_actions"] = _format_priority_actions(s6.get("priority_actions", []))

    return {
        "report_metadata": {
            "type":            "threat_intelligence_report",
            "generated_at":    datetime.now(timezone.utc).isoformat(),
            "pipeline_status": mitre_result.get("status", "unknown"),
        },
        "raw_input":                 original_message,        # pdf_report.py compat
        "section_1_summary_context": s1,
        "section_2_key_takeaways":   s2,
        "section_3_severity":        s3,
        "section_4_ttps":            s4,
        "section_5_iocs_artifacts":  s5,
        "section_6_mitigations":     s6,
        "executive_summary": {
            "threat_classification":  patterns.get("threat_classification", ""),
            "global_severity":        global_severity,
            "global_score":           severity_result.get("global_severity_score", 0.0),
            "total_techniques":       mapping_result.get("total_techniques", 0),
            "tactics_covered":        mapping_result.get("tactics_covered", []),
            "total_iocs":             entities.get("ioc_count", 0),
            "threat_actor":           threat_brief.get("actor_profile", {}).get("likely_actor", "Unknown"),
            "priority_actions_count": len(mitigation_result.get("priority_actions", [])),
            "kill_chain_stages":      threat_brief.get("kill_chain_stages", []),
        },
    }


# ---------------------------------------------------------------------------
#  MULTI-REPORT AGGREGATION
# ---------------------------------------------------------------------------

@traceable(name="Aggregator — aggregate_reports")
def aggregate_reports(report_metadatas: list[dict]) -> dict:
    """Merges N reports into one consolidated campaign report."""
    if not report_metadatas:
        return {"error": "No reports to aggregate"}

    all_techniques, all_scores, all_actors, all_actions, all_malware = [], [], [], [], []
    all_iocs: dict[str, list] = {"ips": [], "urls": [], "domains": [], "cves": []}

    for meta in report_metadatas:
        all_techniques.extend(meta.get("section_4_ttps", []))

        if score := meta.get("section_3_severity", {}).get("global_score", 0):
            all_scores.append(score)

        actor = meta.get("section_2_key_takeaways", {}).get("threat_actor", {}).get("name", "")
        if actor and actor.lower() not in ("unknown", ""):
            all_actors.append(actor)

        network = meta.get("section_5_iocs_artifacts", {}).get("network", {})
        for key in ("ips", "urls", "domains"):
            all_iocs[key].extend(network.get(key, []))
        all_iocs["cves"].extend(
            meta.get("section_5_iocs_artifacts", {})
                .get("threat_intelligence", {})
                .get("cves", [])
        )

        all_actions.extend(meta.get("section_6_mitigations", {}).get("priority_actions", []))

        ti = meta.get("section_5_iocs_artifacts", {}).get("threat_intelligence", {})
        all_malware.extend(ti.get("malware_names", []) + ti.get("ransomware_groups", []))

    for key in all_iocs:
        all_iocs[key] = _deduplicate_ordered(all_iocs[key])

    global_score = max(all_scores) if all_scores else 0.0
    return {
        "report_metadata": {
            "type":               "consolidated_threat_report",
            "generated_at":       datetime.now(timezone.utc).isoformat(),
            "reports_aggregated": len(report_metadatas),
        },
        "global_severity":       _classify_score(global_score),
        "global_severity_score": global_score,
        "threats_identified":    _deduplicate_ordered(all_malware),
        "threat_actors":         _deduplicate_ordered(all_actors),
        "statistics": {
            "total_techniques":      len(all_techniques),
            "unique_techniques":     len({t.get("technique_id", "") for t in all_techniques}),
            "tactics_distribution":  dict(Counter(t.get("tactic", "")         for t in all_techniques)),
            "severity_distribution": dict(Counter(t.get("severity_level", "") for t in all_techniques)),
            "total_iocs":            sum(len(v) for v in all_iocs.values()),
        },
        "all_iocs":        all_iocs,
        "priority_actions": _deduplicate_ordered(all_actions)[:15],
    }

Writing aggregator.py


In [31]:
!pip install fpdf2 matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 19.2 MB/s eta 0:00:00


In [32]:
%%writefile dashboard.py
"""
CTI DASHBOARD — Reads from FULL pipeline data, not just mapped TTPs.

Key fixes vs prior version:
  - Tactics/Kill Chain pulled from BEHAVIORS (real attack flow), not just
    mapped MITRE techniques (which may be incomplete).
  - IOCs categorized by type with stacked bar chart (IPs, URLs, Domains,
    Hashes, CVEs, Malware, Tools).
  - Severity uses behavior severities, falling back to TTP severities.
"""

import io
import os
import base64
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from langsmith import traceable


PALETTE = {
    "bg": "#ffffff", "panel": "#f5f7fb", "border": "#d0d7de",
    "text": "#1f2328", "subtext": "#57606a",
    "Critical": "#d1242f", "High": "#d97706", "Medium": "#f59e0b",
    "Low": "#2da44e", "Unknown": "#6e7781",
    "accent": "#0969da", "accent2": "#1f6feb", "accent3": "#54aeff",
}


KILL_CHAIN_ORDER = [
    "initial_access", "execution", "persistence", "privilege_escalation",
    "defense_evasion", "credential_access", "discovery", "lateral_movement",
    "collection", "exfiltration", "command_and_control", "impact",
]
KILL_CHAIN_LABELS = [
    "Init.Access", "Execution", "Persistence", "Priv.Esc.", "Def.Evasion",
    "Cred.Access", "Discovery", "Lat.Movement", "Collection",
    "Exfiltration", "C2", "Impact",
]


# Mirror of aggregator/threat_brief category aliases
_CAT_TO_TACTIC = {
    "process_creation":       "execution",
    "process_manipulation":   "execution",
    "registry_modification":  "persistence",
    "persistence_mechanisms": "persistence",
    "network_communication":  "command_and_control",
    "network_activity":       "command_and_control",
    "file_system_activity":   "impact",
    "file_system_operations": "impact",
    "credential_access":      "credential_access",
    "credential_theft":       "credential_access",
    "defense_evasion":        "defense_evasion",
    "discovery":              "discovery",
    "lateral_movement":       "lateral_movement",
    "collection":             "collection",
    "exfiltration":           "exfiltration",
    "initial_access":         "initial_access",
    "privilege_escalation":   "privilege_escalation",
    "impact":                 "impact",
    # Phishing/macro hints from descriptions
    "phishing":               "initial_access",
    "macro":                  "execution",
}


def _norm(x):
    s = str(x or "").strip()
    if not s:
        return "Unknown"
    s = s.replace("_", " ").replace("-", " ")
    return " ".join(w[:1].upper() + w[1:].lower() for w in s.split())


def _sev_name(x):
    n = _norm(x)
    for valid in ("Critical", "High", "Medium", "Low", "Unknown"):
        if n.lower() == valid.lower():
            return valid
    return "Unknown"


def _as_int(v):
    try:
        return int(v)
    except Exception:
        return 0


def _category_to_tactic(category: str, description: str = "") -> str:
    """Map behavior category to ATT&CK tactic (kill chain stage)."""
    cat = (category or "").lower().strip().replace(" ", "_")
    if cat in _CAT_TO_TACTIC:
        return _CAT_TO_TACTIC[cat]

    # Description-based fallback
    desc = (description or "").lower()
    if any(k in desc for k in ("phishing", "spear", "lure", "email attachment")):
        return "initial_access"
    if any(k in desc for k in ("macro", "execute", "powershell", "script")):
        return "execution"
    if any(k in desc for k in ("registry", "scheduled task", "autorun", "persistence")):
        return "persistence"
    if any(k in desc for k in ("c2", "command and control", "outbound", "beacon")):
        return "command_and_control"
    if any(k in desc for k in ("encrypt", "ransom", "destroy", "wipe")):
        return "impact"
    return "unknown"


# ---------------------------------------------------------------------------
#  DATA EXTRACTION — uses behaviors AND mapped TTPs together
# ---------------------------------------------------------------------------

def _collect_behaviors(report):
    """
    Pull behaviors from all available sources in the report.
    Aggregator sometimes embeds these in different paths.
    """
    behaviors = []

    # Try threat_brief.mitre_input.behaviors
    tb = report.get("threat_brief", {})
    if isinstance(tb, dict):
        behaviors.extend(tb.get("mitre_input", {}).get("behaviors", []))

    # Try top-level behaviors (passed by orchestrator)
    behaviors.extend(report.get("behaviors", []) or [])

    # If aggregator stored them in section_4 procedures, extract
    for ttp in report.get("section_4_ttps", []) or []:
        ob = ttp.get("observed_behavior", {})
        if ob and ob.get("category"):
            behaviors.append({
                "category":    ob.get("category", ""),
                "description": ob.get("description", ""),
                "severity":    ttp.get("severity_level", "Unknown"),
            })

    return behaviors


def _tactics_from_all(report):
    """Tactics counter built from BOTH behaviors AND mapped techniques."""
    counter = Counter()

    # 1) From mapped techniques
    for ttp in report.get("section_4_ttps", []) or []:
        if t := ttp.get("tactic"):
            counter[_norm(t)] += 1

    # 2) From behaviors (catches stages that didn't map to ATT&CK)
    for b in _collect_behaviors(report):
        tactic = _category_to_tactic(b.get("category", ""), b.get("description", ""))
        if tactic and tactic != "unknown":
            counter[_norm(tactic)] += 1

    # 3) From kill_chain_stages list (last-resort backstop)
    if not counter:
        for s in (
            report.get("executive_summary", {}).get("kill_chain_stages")
            or report.get("section_2_key_takeaways", {}).get("kill_chain_stages")
            or []
        ):
            counter[_norm(s)] += 1

    return {k: v for k, v in counter.items() if v > 0}


def _killchain_from_all(report):
    """Kill chain coverage from BOTH behaviors AND mapped techniques."""
    active = set()

    for ttp in report.get("section_4_ttps", []) or []:
        if t := ttp.get("tactic"):
            active.add(_norm(t).lower().replace(" ", "_"))

    for b in _collect_behaviors(report):
        tactic = _category_to_tactic(b.get("category", ""), b.get("description", ""))
        if tactic and tactic != "unknown":
            active.add(tactic)

    for s in (
        report.get("executive_summary", {}).get("kill_chain_stages")
        or report.get("section_2_key_takeaways", {}).get("kill_chain_stages")
        or []
    ):
        active.add(str(s).lower().replace(" ", "_").replace("-", "_"))

    return active


def _techniques(report):
    counter = Counter()
    for ttp in report.get("section_4_ttps", []) or []:
        tid = ttp.get("technique_id", "")
        name = ttp.get("technique_name", "")
        label = f"{tid} {name}".strip() or "Unknown"
        counter[label] += 1
    return {k: v for k, v in counter.items() if v > 0}


def _severity(report):
    """Severity from BOTH mapped TTPs AND behaviors."""
    counter = Counter()

    # Mapped TTPs first
    dist = report.get("section_3_severity", {}).get("distribution", {})
    if dist:
        for k, v in dist.items():
            counter[_sev_name(k)] += _as_int(v)

    # Add behavior severities (so phishing/macro show up too)
    for b in _collect_behaviors(report):
        sev = b.get("severity", "Unknown")
        # Behaviors may use lowercase
        counter[_sev_name(sev)] += 1

    return {k: v for k, v in counter.items() if v > 0}


def _iocs_categorized(report):
    """Return ordered dict of IOC type -> count, no inflation."""
    s5 = report.get("section_5_iocs_artifacts", {})
    network = s5.get("network", {})
    files = s5.get("files", {})
    crypto = s5.get("crypto", {})
    ti = s5.get("threat_intelligence", {})

    counts = {
        "IPs":        len(network.get("ips", [])),
        "URLs":       len(network.get("urls", [])),
        "Domains":    len(network.get("domains", [])),
        "Hashes":     (len(files.get("md5", [])) +
                       len(files.get("sha1", [])) +
                       len(files.get("sha256", []))),
        "CVEs":       len(ti.get("cves", [])),
        "Malware":    len(ti.get("malware_names", [])) +
                      len(ti.get("ransomware_groups", [])),
        "Tools":      len(ti.get("tools_abused", [])),
        "APT Groups": len(ti.get("apt_groups", [])),
        "Crypto":     (len(crypto.get("btc", [])) +
                       len(crypto.get("eth", [])) +
                       len(crypto.get("xmr", []))),
    }

    # Drop zero-count categories
    return {k: v for k, v in counts.items() if v > 0}


def _summary(report):
    e = report.get("executive_summary", {})
    s2 = report.get("section_2_key_takeaways", {})

    actor = (
        e.get("threat_actor")
        or s2.get("threat_actor", {}).get("name")
        or "Unknown"
    )

    techniques_count = e.get("total_techniques") or len(report.get("section_4_ttps", []))
    behaviors_count = len(_collect_behaviors(report))
    iocs = _iocs_categorized(report)
    total_iocs = sum(iocs.values())

    return {
        "techniques": techniques_count,
        "behaviors":  behaviors_count,
        "iocs":       total_iocs,
        "actor":      actor,
        "class":      e.get("threat_classification", "N/A"),
    }


# ---------------------------------------------------------------------------
#  RENDERERS
# ---------------------------------------------------------------------------

def _no_data(ax, message, p):
    ax.set_facecolor(p["panel"])
    ax.text(0.5, 0.5, message, ha="center", va="center",
            color=p["subtext"], fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


def render_tactics(ax, data, p):
    ax.set_facecolor(p["panel"])
    if not data:
        _no_data(ax, "No tactic data", p)
        return

    items = sorted(data.items(), key=lambda x: x[1])
    labels = [i[0] for i in items]
    values = [i[1] for i in items]

    bars = ax.barh(labels, values, color=p["accent"], edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(v), va="center", fontsize=9, color=p["text"])

    ax.set_title("ATT&CK Tactics Observed", color=p["text"], fontweight="bold")
    ax.tick_params(colors=p["text"], labelsize=9)
    ax.grid(axis="x", alpha=0.2)
    ax.set_xlim(0, max(values) * 1.15)


def render_severity(ax, data, p):
    ax.set_facecolor(p["panel"])
    order = ["Critical", "High", "Medium", "Low", "Unknown"]
    labels = [k for k in order if data.get(k, 0) > 0]
    sizes = [data[k] for k in labels]

    if not sizes or sum(sizes) <= 0:
        _no_data(ax, "No severity data", p)
        return

    colors = [p.get(k, p["Unknown"]) for k in labels]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors,
        autopct=lambda pct: f"{pct:.0f}%\n({int(pct*sum(sizes)/100)})",
        wedgeprops={"edgecolor": "#ffffff", "linewidth": 2},
        textprops={"color": p["text"], "fontsize": 9},
        startangle=90,
    )
    for at in autotexts:
        at.set_color("white")
        at.set_fontweight("bold")
        at.set_fontsize(8)

    ax.set_title("Severity Distribution", color=p["text"], fontweight="bold")


def render_killchain(ax, active_stages, p):
    ax.set_facecolor(p["panel"])

    vals = [1 if stage in active_stages else 0 for stage in KILL_CHAIN_ORDER]
    colors = [p["accent"] if v > 0 else p["border"] for v in vals]

    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor="white", linewidth=1)

    # Annotate active stages
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, 1.05,
                    "✓", ha="center", va="bottom",
                    color=p["accent"], fontsize=12, fontweight="bold")

    ax.set_ylim(0, 1.3)
    ax.set_xticks(range(len(KILL_CHAIN_ORDER)))
    ax.set_xticklabels(KILL_CHAIN_LABELS, rotation=45, ha="right", fontsize=7.5)
    ax.set_yticks([])

    n_active = sum(vals)
    ax.set_title(f"Kill Chain Coverage ({n_active}/{len(KILL_CHAIN_ORDER)} stages)",
                 color=p["text"], fontweight="bold")
    ax.tick_params(colors=p["text"])
    ax.grid(axis="y", alpha=0.0)

    # Legend
    ax.text(0.0, -0.15, "■ Detected", color=p["accent"],
        fontsize=8, fontweight="bold", transform=ax.transAxes)
    ax.text(0.25, -0.15, "■ Not observed", color=p["border"],
            fontsize=8, transform=ax.transAxes)


def render_iocs(ax, ioc_data, p):
    """Categorized IOC bar chart — replaces vague 'top techniques'."""
    ax.set_facecolor(p["panel"])
    if not ioc_data:
        _no_data(ax, "No IOCs extracted", p)
        return

    items = sorted(ioc_data.items(), key=lambda x: x[1], reverse=True)
    labels = [i[0] for i in items]
    values = [i[1] for i in items]

    # Color-code by type category
    color_map = {
        "IPs": p["accent"], "URLs": p["accent2"], "Domains": p["accent3"],
        "Hashes": p["High"], "CVEs": p["Critical"], "Malware": p["Critical"],
        "Tools": p["Medium"], "APT Groups": p["Critical"], "Crypto": p["Medium"],
    }
    colors = [color_map.get(l, p["accent"]) for l in labels]

    bars = ax.barh(labels, values, color=colors, edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(v), va="center", fontsize=9, color=p["text"], fontweight="bold")

    ax.invert_yaxis()
    total = sum(values)
    ax.set_title(f"IOCs by Category (total: {total})",
                 color=p["text"], fontweight="bold")
    ax.tick_params(colors=p["text"], labelsize=9)
    ax.grid(axis="x", alpha=0.2)
    ax.set_xlim(0, max(values) * 1.2)


# ---------------------------------------------------------------------------
#  MAIN ENTRY
# ---------------------------------------------------------------------------

@traceable(name="CTI Dashboard")
def generate_dashboard(report, output="dashboard.png"):
    p = PALETTE

    tactics = _tactics_from_all(report)
    severity = _severity(report)
    killchain_active = _killchain_from_all(report)
    iocs = _iocs_categorized(report)
    summary = _summary(report)

    os.makedirs(os.path.dirname(os.path.abspath(output)) or ".", exist_ok=True)

    fig = plt.figure(figsize=(14, 8), facecolor=p["bg"])
    gs = gridspec.GridSpec(2, 2, figure=fig)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    render_tactics(ax1, tactics, p)
    render_severity(ax2, severity, p)
    render_killchain(ax3, killchain_active, p)
    render_iocs(ax4, iocs, p)

    title = (
        f"CTI DASHBOARD  |  Actor: {summary['actor']}  |  "
        f"{summary['behaviors']} behaviors  |  "
        f"{summary['techniques']} techniques  |  "
        f"{summary['iocs']} IOCs"
    )
    fig.suptitle(title, color=p["text"], fontsize=14, fontweight="bold", y=0.98)

    fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.12, hspace=0.45, wspace=0.30)

    plt.savefig(output, dpi=100,facecolor=p["bg"])

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100,facecolor=p["bg"])
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)

    return {
        "path": os.path.abspath(output),
        "base64": b64,
        "debug_counts": {
            "tactics": len(tactics),
            "severity": len(severity),
            "killchain_active": len(killchain_active),
            "iocs_categories": len(iocs),
            "behaviors": summary["behaviors"],
            "techniques": summary["techniques"],
        },
    }

Writing dashboard.py


In [33]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.0 MB/s eta 0:00:00:00:01


In [34]:
%%writefile pdf_report.py

import os

from reportlab.platypus import Image, PageBreak
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, KeepTogether, Flowable,
)

# ---------------------------------------------------------------------------
#  THEME
# ---------------------------------------------------------------------------

PRIMARY_DARK  = colors.HexColor("#0F172A")
PRIMARY_LIGHT = colors.HexColor("#F8FAFC")
ACCENT        = colors.HexColor("#0EA5E9")
TEXT_MAIN     = colors.HexColor("#334155")
TEXT_MUTED    = colors.HexColor("#64748B")
BORDER        = colors.HexColor("#E2E8F0")
WHITE         = colors.white

_SEV_COLORS = {
    "Critical": colors.HexColor("#DC2626"),
    "High":     colors.HexColor("#EA580C"),
    "Medium":   colors.HexColor("#EAB308"),
    "Low":      colors.HexColor("#16A34A"),
}

_KILL_CHAIN_ORDER = [
    "initial_access", "execution", "persistence", "privilege_escalation",
    "defense_evasion", "credential_access", "discovery", "lateral_movement",
    "collection", "exfiltration", "command_and_control", "impact",
]
_KILL_CHAIN_LABELS = [
    "Init.", "Exec.", "Persist.", "Priv.Esc.", "Def.Ev.",
    "Cred.", "Discov.", "Lat.Mv.", "Collect.", "Exfil.", "C2", "Impact",
]


def _sev_color(level: str):
    return _SEV_COLORS.get(level, TEXT_MUTED)


def _safe(val, maxlen: int = 0) -> str:
    """ReportLab ASCII encoding only. Aggregator is responsible for truncation."""
    subs = {
        "\u2014": "-", "\u2013": "-", "\u2018": "'", "\u2019": "'",
        "\u201c": '"', "\u201d": '"', "\u2026": "...", "\u00b7": "*",
    }
    text = str(val) if val is not None else ""
    result = "".join(subs.get(ch, ch) if ord(ch) < 256 else "?" for ch in text)
    return result[:maxlen - 1] + "..." if maxlen and len(result) > maxlen else result


# ---------------------------------------------------------------------------
#  STYLES
# ---------------------------------------------------------------------------

def _build_styles() -> dict:
    S = ParagraphStyle
    return {
        "subsection":   S("sub",    fontSize=10, textColor=PRIMARY_DARK, fontName="Helvetica-Bold",    leading=14, spaceBefore=8,  spaceAfter=4),
        "body":         S("body",   fontSize=9,  textColor=TEXT_MAIN,    fontName="Helvetica",         leading=14, spaceAfter=4,   alignment=TA_JUSTIFY),
        "kv_key":       S("kv_k",   fontSize=9,  textColor=TEXT_MUTED,   fontName="Helvetica-Bold",    leading=12),
        "kv_val":       S("kv_v",   fontSize=9,  textColor=TEXT_MAIN,    fontName="Helvetica",         leading=12),
        "mono":         S("mono",   fontSize=7.5,textColor=TEXT_MAIN,    fontName="Courier",           leading=10, spaceAfter=2),
        "bullet":       S("blt",    fontSize=9,  textColor=TEXT_MAIN,    fontName="Helvetica",         leading=13, leftIndent=12,  bulletIndent=4, spaceAfter=3),
        "table_hdr":    S("thdr",   fontSize=8,  textColor=WHITE,        fontName="Helvetica-Bold",    leading=10, alignment=TA_CENTER),
        "table_cell":   S("tcell",  fontSize=8,  textColor=TEXT_MAIN,    fontName="Helvetica",         leading=11, alignment=TA_LEFT),
        "table_cell_c": S("tcellc", fontSize=8,  textColor=TEXT_MAIN,    fontName="Helvetica",         leading=11, alignment=TA_CENTER),
        "procedure":    S("proc",   fontSize=8,  textColor=TEXT_MAIN,    fontName="Helvetica-Oblique", leading=11, leftIndent=4),
        "caption":      S("cap",    fontSize=7,  textColor=TEXT_MUTED,   fontName="Helvetica",         leading=9,  alignment=TA_CENTER, spaceAfter=4),
        "raw":          S("raw",    fontSize=7.5,textColor=TEXT_MUTED,   fontName="Courier",           leading=10, backColor=PRIMARY_LIGHT, leftIndent=6, rightIndent=6),
    }


def _table_style() -> TableStyle:
    return TableStyle([
        ("BACKGROUND",    (0, 0), (-1,  0), PRIMARY_DARK),
        ("TEXTCOLOR",     (0, 0), (-1,  0), WHITE),
        ("ROWBACKGROUNDS",(0, 1), (-1, -1), [WHITE, PRIMARY_LIGHT]),
        ("LINEABOVE",     (0, 0), (-1,  0), 2, ACCENT),
        ("LINEBELOW",     (0, 0), (-1, -1), 0.5, BORDER),
        ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING",   (0, 0), (-1, -1), 6),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 6),
        ("TOPPADDING",    (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ])


def _render_dashboard(report_data, st):
    dashboard_path = (
        report_data.get("dashboard_path")
        or report_data.get("dashboard", {}).get("path")
    )

    if not dashboard_path or not os.path.exists(dashboard_path):
        return []

    img = Image(dashboard_path)

    iw = img.imageWidth[0] if isinstance(img.imageWidth, list) else img.imageWidth
    ih = img.imageHeight[0] if isinstance(img.imageHeight, list) else img.imageHeight
    
    max_width  = 170 * mm
    max_height = 110 * mm
    aspect = ih / iw
    
    if aspect * max_width <= max_height:
        img.drawWidth  = max_width
        img.drawHeight = max_width * aspect
    else:
        img.drawHeight = max_height
        img.drawWidth  = max_height / aspect
    
    img.hAlign = "CENTER"

    return [
        PageBreak(),
        SectionHeader(6, "CTI Visual Dashboard"),
        Spacer(1, 5 * mm),
        img,
        Spacer(1, 3 * mm),
        Paragraph("Generated from CTI dashboard module.", st["caption"]),
    ]


# ---------------------------------------------------------------------------
#  CUSTOM FLOWABLES
# ---------------------------------------------------------------------------

class SectionHeader(Flowable):
    def __init__(self, number: int, title: str, width: float = 170 * mm):
        super().__init__()
        self.number, self.title = number, title
        self.width, self.height = width, 12 * mm

    def draw(self):
        self.canv.setFillColor(PRIMARY_DARK)
        self.canv.rect(0, 0, self.width, self.height, fill=1, stroke=0)
        self.canv.setFillColor(ACCENT)
        self.canv.rect(0, 0, 4 * mm, self.height, fill=1, stroke=0)
        self.canv.setFillColor(WHITE)
        self.canv.setFont("Helvetica-Bold", 11)
        self.canv.drawString(8 * mm, 4 * mm, f"{self.number}. {_safe(self.title).upper()}")


class SeverityBadge(Flowable):
    def __init__(self, level: str, score: float, width: float = 170 * mm):
        super().__init__()
        self.level, self.score = level, score
        self.width, self.height = width, 14 * mm

    def draw(self):
        c = _sev_color(self.level)
        self.canv.setFillColor(c)
        self.canv.roundRect(0, 0, self.width, self.height, 2, fill=1, stroke=0)
        self.canv.setFillColor(WHITE)
        self.canv.roundRect(self.width - 35 * mm, 2 * mm, 33 * mm, 10 * mm, 2, fill=1, stroke=0)
        self.canv.setFillColor(WHITE)
        self.canv.setFont("Helvetica-Bold", 12)
        self.canv.drawString(6 * mm, 4.5 * mm, f"GLOBAL SEVERITY: {self.level.upper()}")
        self.canv.setFillColor(c)
        self.canv.drawCentredString(self.width - 18.5 * mm, 4.5 * mm, f"{self.score:.1f} / 5.0")


class KillChainBar(Flowable):
    """Renders aggregator-provided active stage names. No normalization here."""
    def __init__(self, active_stages: list, width: float = 170 * mm):
        super().__init__()
        self.active = set(active_stages)   # already normalized by aggregator
        self.width, self.height = width, 16 * mm

    def draw(self):
        w_box = self.width / len(_KILL_CHAIN_ORDER)
        for i, (stage, label) in enumerate(zip(_KILL_CHAIN_ORDER, _KILL_CHAIN_LABELS)):
            active = stage in self.active
            self.canv.setFillColor(ACCENT if active else PRIMARY_LIGHT)
            self.canv.setStrokeColor(WHITE if active else BORDER)
            self.canv.rect(i * w_box, 6 * mm, w_box, 8 * mm, fill=1, stroke=1)
            self.canv.setFillColor(WHITE if active else TEXT_MUTED)
            self.canv.setFont("Helvetica-Bold" if active else "Helvetica", 5.5)
            self.canv.drawCentredString((i * w_box) + (w_box / 2), 8 * mm, label)
        self.canv.setFont("Helvetica", 7)
        self.canv.setFillColor(TEXT_MUTED)
        self.canv.drawString(0, 2 * mm, "Reconnaissance")
        self.canv.drawRightString(self.width, 2 * mm, "Actions on Objectives")


# ---------------------------------------------------------------------------
#  SECTION RENDERERS
# ---------------------------------------------------------------------------
def _render_section_1(data: dict, st: dict) -> list:
    """Raw input — verbatim passthrough, no modification."""
    return [SectionHeader(1, "Raw Input"), Spacer(1, 7 * mm),
            Paragraph(_safe(data.get("raw_input", "No raw input provided."), 2000), st["raw"])]
    
def _render_section_2(data: dict, st: dict) -> list:
    """Executive Summary — reads pre-shaped fields, emits flowables only."""
    s1    = data.get("section_1_summary_context", {})
    s2    = data.get("section_2_key_takeaways", {})
    s3    = data.get("section_3_severity", {})
    ex    = data.get("executive_summary", {})
    actor = s2.get("threat_actor", {})
    env   = s2.get("target_environment", {})

    items = [SectionHeader(2, "Executive Summary"), Spacer(1, 4 * mm),
             Paragraph("Incident Overview", st["subsection"]),
             Paragraph(_safe(s1.get("attack_narrative") or "No narrative available."), st["body"]),
             Paragraph("Key Takeaways", st["subsection"])]

    kv_rows = [
        [Paragraph("Threat Actor",   st["kv_key"]), Paragraph(f"{_safe(actor.get('name', 'Unidentified'))} (Conf: {actor.get('confidence', 0.0):.0%})", st["kv_val"])],
        [Paragraph("Initial Access", st["kv_key"]), Paragraph(_safe(s2.get("initial_access_vector", "Unknown")), st["kv_val"])],
        [Paragraph("Impact",         st["kv_key"]), Paragraph(_safe(s2.get("impact", "Unknown")), st["kv_val"])],
        [Paragraph("OS / Sector",    st["kv_key"]), Paragraph(f"{_safe(env.get('os', 'Unknown').title())} / {_safe(', '.join(env.get('sector', [])) or 'Unknown')}", st["kv_val"])],
        [Paragraph("Classification", st["kv_key"]), Paragraph(_safe(s1.get("threat_classification", "Unknown")), st["kv_val"])],
    ]
    kv = Table(kv_rows, colWidths=[48 * mm, 122 * mm])
    kv.setStyle(_table_style())
    items += [KeepTogether([kv]), Spacer(1, 4 * mm),
              Paragraph("Threat Severity", st["subsection"]),
              SeverityBadge(s3.get("global_level", "Unknown"), s3.get("global_score", 0.0)),
              Spacer(1, 4 * mm),
              Paragraph("Kill Chain Coverage", st["subsection"]),
              KillChainBar(ex.get("kill_chain_stages", []))]
    return items


def _render_section_3(data: dict, st: dict) -> list:
    """TTPs — aggregator pre-sorted by kill chain; procedure_text pre-built."""
    ttps  = data.get("section_4_ttps", [])
    items = [SectionHeader(3, "Tactics, Techniques & Procedures"), Spacer(1, 3 * mm)]

    if not ttps:
        return items + [Paragraph("No TTPs mapped for this threat.", st["body"])]

    rows = [[Paragraph(h, st["table_hdr"]) for h in ("Tactic", "Technique", "Sev.", "Procedure")]]
    prev_tactic = None
    for ttp in ttps:
        tactic  = _safe(ttp.get("tactic", "Unknown"))
        level   = ttp.get("severity_level", "Unknown")
        # FIX 1: hexval() returns "0xRRGGBB" — [2:] strips "0x", giving valid 6-char hex.
        # Original [1:] stripped only "0", leaving "xRRGGBB" which is an invalid color.
        hex_col = _sev_color(level).hexval()[2:]
        rows.append([
            Paragraph(tactic if tactic != prev_tactic else "", st["table_cell"]),
            Paragraph(f"<b>{_safe(ttp.get('technique_name', ''), 40)}</b><br/><font color='#0EA5E9'>{_safe(ttp.get('technique_id', ''))}</font>", st["table_cell"]),
            Paragraph(f"<font color='#{hex_col}'><b>{level[:4]}</b></font>", st["table_cell_c"]),
            Paragraph(_safe(ttp.get("procedure_text", "Behavior observed."), 300), st["procedure"]),
        ])
        prev_tactic = tactic

    tbl = Table(rows, colWidths=[30 * mm, 42 * mm, 14 * mm, 84 * mm], repeatRows=1)
    tbl.setStyle(_table_style())
    return items + [tbl, Spacer(1, 3 * mm),
                    Paragraph("Procedures extracted from source context.", st["caption"])]


def _render_section_4(data: dict, st: dict) -> list:
    """IOCs and CTI entities."""
    s5 = data.get("section_5_iocs_artifacts", {})
    items = [SectionHeader(4, "IOCs & Artifacts"), Spacer(1, 3 * mm)]

    tables = list(s5.get("ioc_tables", []))

    # Also render threat intelligence entities, not only raw network/file IOCs.
    ti = s5.get("threat_intelligence", {})
    label_map = {
        "malware_names": "Malware",
        "ransomware_groups": "Ransomware Group",
        "apt_groups": "APT Group",
        "tools_abused": "Tool",
        "cves": "CVE",
        "campaign_names": "Campaign",
    }

    ti_rows = []
    for key, label in label_map.items():
        for value in ti.get(key, []):
            if isinstance(value, str) and value.strip():
                ti_rows.append((label, value))

    existing_titles = {str(t.get("title", "")).lower() for t in tables}
    if ti_rows and "threat intelligence entities" not in existing_titles:
        tables.append({
            "title": "Threat Intelligence Entities",
            "rows": ti_rows,
        })

    if not tables:
        return items + [
            Paragraph(
                "No concrete network, file, crypto, CVE, malware, or actor indicators "
                "were extracted from this message.",
                st["body"],
            )
        ]

    for group in tables:
        raw_rows = group.get("rows", [])
        if not raw_rows:
            continue

        rows = [[Paragraph("Type", st["table_hdr"]), Paragraph("Indicator", st["table_hdr"])]]
        rows += [
            [
                Paragraph(_safe(k, 24), st["table_cell_c"]),
                Paragraph(_safe(v, 150), st["mono"]),
            ]
            for k, v in raw_rows
        ]

        tbl = Table(rows, colWidths=[35 * mm, 135 * mm], repeatRows=1)
        tbl.setStyle(_table_style())

        items += [
            Paragraph(_safe(group.get("title", "Indicators")), st["subsection"]),
            tbl,
            Spacer(1, 3 * mm),
        ]

    return items


def _render_section_5(data: dict, st: dict) -> list:
    """Mitigations."""
    s6 = data.get("section_6_mitigations", {})

    items = [
        SectionHeader(5, "Mitigation & Remediation"),
        Spacer(1, 3 * mm),
        Paragraph("Top Priority Actions", st["subsection"]),
    ]

    priority_actions = s6.get("priority_actions", [])

    if priority_actions:
        p_rows = [[Paragraph(h, st["table_hdr"]) for h in ("#", "Priority Action", "Standard Ref")]]

        for idx, action in enumerate(priority_actions[:5]):
            if isinstance(action, dict):
                text = action.get("text", "")
                refs = action.get("refs", "Best Practice")
            else:
                text = str(action)
                refs = "Best Practice"

            p_rows.append([
                Paragraph(str(idx + 1), st["table_cell_c"]),
                Paragraph(_safe(text, 300), st["table_cell"]),
                Paragraph(_safe(refs, 60), st["table_cell_c"]),
            ])

        p_tbl = Table(p_rows, colWidths=[8 * mm, 130 * mm, 32 * mm], repeatRows=1)
        p_tbl.setStyle(_table_style())
        items += [p_tbl, Spacer(1, 5 * mm)]
    else:
        items += [
            Paragraph(
                "No Critical/High priority actions were promoted. "
                "See technique-level mitigations below if available.",
                st["body"],
            ),
            Spacer(1, 3 * mm),
        ]

    by_technique = s6.get("by_technique", [])
    if by_technique:
        items += [Paragraph("Mitigations by Technique", st["subsection"])]

        rows = [[
            Paragraph("Technique", st["table_hdr"]),
            Paragraph("Severity", st["table_hdr"]),
            Paragraph("Priority", st["table_hdr"]),
            Paragraph("Recommended Mitigation", st["table_hdr"]),
        ]]

        for tm in by_technique[:8]:
            mitigations = tm.get("mitigations", [])
            mit_text = "; ".join(mitigations[:2]) if mitigations else "No mitigation text returned."

            rows.append([
                Paragraph(
                    f"<b>{_safe(tm.get('technique_id', ''))}</b><br/>{_safe(tm.get('technique_name', ''), 45)}",
                    st["table_cell"],
                ),
                Paragraph(_safe(tm.get("severity_level", "Unknown")), st["table_cell_c"]),
                Paragraph(_safe(tm.get("priority", "short_term")), st["table_cell_c"]),
                Paragraph(_safe(mit_text, 350), st["table_cell"]),
            ])

        tbl = Table(rows, colWidths=[42 * mm, 22 * mm, 26 * mm, 80 * mm], repeatRows=1)
        tbl.setStyle(_table_style())
        items += [tbl, Spacer(1, 5 * mm)]

    recs = s6.get("global_recommendations", [])
    if recs:
        items += [
            Paragraph("Strategic Recommendations", st["subsection"]),
            *[Paragraph(f"&#x2022; {_safe(r)}", st["bullet"]) for r in recs],
        ]

    if not priority_actions and not by_technique and not recs:
        items += [
            Paragraph(
                "No mitigations were generated. This usually means no ATT&CK techniques "
                "were mapped upstream.",
                st["body"],
            )
        ]

    return items




# ---------------------------------------------------------------------------
#  PAGE DECORATORS
# ---------------------------------------------------------------------------

def _on_page(canvas, doc):
    canvas.saveState()
    canvas.setStrokeColor(PRIMARY_DARK)
    canvas.line(15 * mm, A4[1] - 12 * mm, A4[0] - 15 * mm, A4[1] - 12 * mm)
    canvas.setFont("Helvetica-Bold", 9)
    canvas.setFillColor(PRIMARY_DARK)
    canvas.drawString(15 * mm, A4[1] - 10 * mm, "CTI Threat Intelligence Report")
    canvas.setStrokeColor(BORDER)
    canvas.line(15 * mm, 12 * mm, A4[0] - 15 * mm, 12 * mm)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(TEXT_MUTED)
    canvas.drawCentredString(A4[0] / 2, 8 * mm, f"Page {doc.page}  |  Strictly Confidential")
    canvas.restoreState()


# ---------------------------------------------------------------------------
#  PUBLIC ENTRY POINT
# ---------------------------------------------------------------------------
import copy
def generate_pdf_report(report_data: dict, output_filepath: str) -> None:
    doc = SimpleDocTemplate(output_filepath, pagesize=A4,
                            rightMargin=15 * mm, leftMargin=15 * mm,
                            topMargin=15 * mm, bottomMargin=20 * mm)

    st = _build_styles()
    story = []

    renderers = [
        ("1", _render_section_1),
        ("2", _render_section_2),
        ("3", _render_section_3),
        ("4", _render_section_4),
        ("5", _render_section_5),
        ("dash", _render_dashboard),
    ]

    seen = set()

    for label, r in renderers:
        if r in seen:
            raise RuntimeError(f"Duplicate renderer detected: {r.__name__}")
        seen.add(r)

        story.extend(r(report_data, st))
        story.append(Spacer(1, 10 * mm))

    doc.build(story, onFirstPage=_on_page, onLaterPages=_on_page)

Writing pdf_report.py


---
## Modules V7 ajoutés — Pipeline Complet

Les cellules ci-dessous complètent le pipeline avec :
- `web_search_agent.py` — collecte RSS/OTX (NODE 0)
- `notify_agent.py`     — notification email Brevo (NODE 6)
- `prioritizer.py`      — ranking actions par risk score
- `graph.py`            — orchestration LangGraph (START → END)
- `ragas_*.py`          — évaluation qualité RAG + agents


In [35]:
!pip install -q feedparser beautifulsoup4

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.8 MB/s eta 0:00:00


### `web_search_agent.py`

In [36]:
%%writefile web_search_agent.py
"""
==========================================
 WEB SEARCH AGENT (LangGraph compatible)
==========================================
Collects CTI threats from multiple open sources,
filters by embedding similarity (no keywords),
and returns raw textual messages for downstream
CTI analysis agents.
"""

import requests
import feedparser
import time
import random
import numpy as np
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer, util
from langsmith import traceable

# ─────────────────────────────────────
# CONFIG
# ─────────────────────────────────────
from kaggle_secrets import UserSecretsClient
OTX_API_KEY = UserSecretsClient().get_secret("OTX_API_KEY")

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Firefox/121.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) Chrome/119.0.0.0",
]

# ─────────────────────────────────────
# EMBEDDING THREAT FILTER
# ─────────────────────────────────────

# Modèle léger, multilingue, rapide
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

# Phrase de référence CTI — décrit ce qu'est une menace
CTI_REFERENCE = (
    "A cybersecurity threat, attack, vulnerability, malware, exploit, "
    "data breach, or malicious actor targeting systems or organizations."
)

# Embedding de référence (calculé une seule fois)
CTI_REFERENCE_EMB = EMBED_MODEL.encode(CTI_REFERENCE, convert_to_tensor=True)

# Seuil de similarité cosinus (0.0 → 1.0)
# 0.35 = bon équilibre précision/rappel pour du texte CTI court
THREAT_THRESHOLD = 0.35


def is_threat(item: dict) -> tuple[bool, float]:
    """
    Retourne (is_threat: bool, score: float).
    Compare title + summary à la phrase de référence CTI
    via similarité cosinus sur embeddings.
    """
    text = (item.get("title", "") + ". " + item.get("summary", "")).strip()
    if not text:
        return False, 0.0

    emb = EMBED_MODEL.encode(text, convert_to_tensor=True)
    score = float(util.cos_sim(emb, CTI_REFERENCE_EMB))
    return score >= THREAT_THRESHOLD, round(score, 4)


# ─────────────────────────────────────
# HELPERS
# ─────────────────────────────────────
def headers():
    return {"User-Agent": random.choice(USER_AGENTS)}

def wait():
    time.sleep(random.uniform(2, 4))


# ─────────────────────────────────────
# COLLECTORS
# ─────────────────────────────────────
def collect_rss():
    sources = {
        "TheHackerNews": "https://feeds.feedburner.com/TheHackersNews",
        "BleepingComputer": "https://www.bleepingcomputer.com/feed/"
    }
    data = []
    for src, url in sources.items():
        feed = feedparser.parse(url)
        for e in feed.entries[:5]:
            data.append({
                "source": src,
                "title": e.title,
                "summary": e.summary,
                "link": e.link
            })
        wait()
    return data


def collect_reddit():
    url = "https://www.reddit.com/r/netsec/hot.json?limit=5"
    try:
        r = requests.get(url, headers=headers(), timeout=30)
        if r.status_code != 200:
            return []
        posts = r.json()["data"]["children"]
        return [{
            "source": "Reddit r/netsec",
            "title": p["data"]["title"],
            "summary": p["data"]["selftext"][:300],
            "link": p["data"]["url"]
        } for p in posts]
    except Exception:
        return []


def collect_stackoverflow():
    url = "https://api.stackexchange.com/2.3/questions"
    params = {
        "order": "desc",
        "sort": "creation",
        "tagged": "security",
        "site": "stackoverflow",
        "pagesize": 5
    }
    r = requests.get(url, params=params, headers=headers())
    data = r.json().get("items", [])
    wait()
    return [{
        "source": "StackOverflow",
        "title": q["title"],
        "summary": str(q["tags"]),
        "link": q["link"]
    } for q in data]


def collect_otx():
    url = "https://otx.alienvault.com/api/v1/pulses/subscribed"
    headers_otx = {"X-OTX-API-KEY": OTX_API_KEY}
    try:
        r = requests.get(url, headers=headers_otx, timeout=30)
        pulses = r.json().get("results", [])[:5]
        return [{
            "source": "AlienVault OTX",
            "title": p["name"],
            "summary": p.get("description", "")[:300],
            "link": f"https://otx.alienvault.com/pulse/{p['id']}"
        } for p in pulses]
    except Exception:
        return []


# ─────────────────────────────────────
# UTILS
# ─────────────────────────────────────
def deduplicate(items):
    unique = []
    for item in items:
        title = item["title"].lower()
        if any(SequenceMatcher(None, title, u["title"].lower()).ratio() >= 0.7 for u in unique):
            continue
        unique.append(item)
    return unique


# ─────────────────────────────────────
# MAIN ENTRY (LangGraph)
# ─────────────────────────────────────
@traceable(name="SearchAgent — web_search_agent")
def main() -> list[str]:
    """
    LangGraph entry point.
    Returns list[str] of raw CTI messages (threats only).
    """

    print("🌐 Web Search Agent running...")

    collected = (
        collect_rss()
        + collect_reddit()
        + collect_stackoverflow()
        + collect_otx()
    )
    total_count = len(collected)


    # 1️⃣  Filtre par similarité d'embedding
    threats, non_threats = [], []
    for item in collected:
        threat, score = is_threat(item)
        item["threat_score"] = score
        if threat:
            threats.append(item)
        else:
            non_threats.append(item)

    print(f"🔍 Embedding filter : {len(threats)} threat(s) (score ≥ {THREAT_THRESHOLD}) "
          f"/ {len(non_threats)} non-threat(s) ignoré(s)")

    # 2️⃣  Déduplication
    unique = deduplicate(threats)
    threat_count = len(threats)
    non_threat_count = len(non_threats)
    # 3️⃣  Formatage — score inclus pour traçabilité
    messages = []
    for a in unique:
        messages.append(
            f"Source: {a['source']}\n"
            f"Title: {a['title']}\n"
            f"URL: {a['link']}\n"
            f"Threat Score: {a['threat_score']}\n"
            f"Summary: {a.get('summary','')}"
        )
    print("📊 ===== STATISTICS =====")
    print(f"🌐 Total collected items: {total_count}")
    print(f"🚨 Threats detected: {threat_count}")
    print(f"🧹 Non-threats filtered out: {non_threat_count}")
    print("=========================")

    print(f"✅ Web Search Agent returned {len(messages)} messages")
    return messages

Writing web_search_agent.py


### `notify_agent.py`

In [37]:
%%writefile notify_agent.py
"""
==========================================
 SINGLE NOTIFY AGENT (Kaggle + Brevo)
==========================================
"""
import os
import logging
import requests
import base64
from kaggle_secrets import UserSecretsClient
from langsmith import traceable

logging.basicConfig(level=logging.INFO)
from kaggle_secrets import UserSecretsClient
@traceable(name="NotifyAgent — notify_agent")
def main(pdf_path: str, executive_summary: dict):
    logging.info("📢 Notify Agent started")

    # 🔒 Lire les secrets ICI (jamais au niveau global)
    secrets = UserSecretsClient()
    try:
        BREVO_API_KEY = secrets.get_secret("BREVO_API_KEY")
        EMAIL_SENDER  = secrets.get_secret("EMAIL_SENDER")
    except Exception as e:
        logging.error(f"❌ Cannot read Kaggle secrets: {e}")
        return {"error": "secrets_not_found"}, 0

    logging.info(f"🔑 BREVO_API_KEY loaded: {bool(BREVO_API_KEY)}")
    logging.info(f"📧 EMAIL_SENDER loaded: {EMAIL_SENDER}")

    if not BREVO_API_KEY or not EMAIL_SENDER:
        logging.error("❌ Missing BREVO_API_KEY or EMAIL_SENDER")
        print("❌ Missing BREVO_API_KEY or EMAIL_SENDER")
        return {"error": "missing_secrets"}, 0

    # ───────────────────────────────────
    # Email content
    # ───────────────────────────────────
    subject = "Notification de sécurité – Rapport CTI automatique"
    html_content = f"""
    <p>Bonjour,</p>
    <p>Un rapport de cybersécurité a été analysé automatiquement.</p>
    <pre>{executive_summary}</pre>
    <p>Cordialement,<br><b>Notify Agent</b></p>
    """

    # ───────────────────────────────────
    # PDF attachment
    # ───────────────────────────────────
    attachments = []
    if pdf_path and os.path.exists(pdf_path):
        with open(pdf_path, "rb") as f:
            attachments.append({
                "content": base64.b64encode(f.read()).decode(),
                "name": os.path.basename(pdf_path),
            })

    # ───────────────────────────────────
    # Send via Brevo
    # ───────────────────────────────────
    response = requests.post(
        "https://api.brevo.com/v3/smtp/email",
        headers={
            "api-key": BREVO_API_KEY,
            "Content-Type": "application/json"
        },
        json={
            "sender": {"name": "Notify Agent", "email": EMAIL_SENDER},
            "to": [{"email": "emnaghorbel56@gmail.com"}],
            "subject": subject,
            "htmlContent": html_content,
            "attachment": attachments
        }
    )

    logging.info(f"✅ Brevo status: {response.status_code}")
    return {
        "subject": subject,
        "recipient": "emnaghorbel56@gmail.com",
        "pdf_path": pdf_path,
        "status_code": response.status_code
    }, 1 if response.status_code == 201 else 0

Writing notify_agent.py


### `prioritizer.py`

In [38]:
%%writefile prioritizer.py
"""
==========================================
 TOOL 5 — Prioritizer & Summary Engine
==========================================
Ranks actions based on risk and generates a 
concise summary for human stakeholders.
"""

def prioritize_actions(aggregated_report: dict) -> list[dict]:
    """
    Ranks techniques by severity score and assigns urgency levels.
    """
    techniques = aggregated_report.get("techniques", [])
    # Sort by severity score descending
    sorted_techs = sorted(techniques, key=lambda x: x.get("severity", 0), reverse=True)
    
    prioritized = []
    for tech in sorted_techs:
        score = tech.get("severity", 0)
        # Assign Urgency
        if score >= 4.0:
            urgency = "IMMEDIATE"
        elif score >= 3.0:
            urgency = "SHORT_TERM"
        else:
            urgency = "LONG_TERM"
            
        prioritized.append({
            "technique_id": tech.get("id"),
            "technique_name": tech.get("name"),
            "priority_score": score,
            "urgency": urgency,
            "recommended_actions": tech.get("mitigation", [])
        })
    return prioritized

def generate_executive_summary(aggregated_report: dict) -> dict:
    """
    Generates a high-level summary string and top 3 actions.
    """
    threats = ", ".join(aggregated_report.get("threats_identified", ["Unknown"]))
    severity = aggregated_report.get("global_severity", "Low")
    count = aggregated_report.get("statistics", {}).get("total_techniques", 0)
    
    summary_text = (
        f"The analysis identified activity associated with {threats}. "
        f"The overall threat landscape is currently rated as {severity} risk, "
        f"with {count} distinct MITRE ATT&CK techniques detected across the environment."
    )
    
    # Extract top 3 actions from the report for the "Quick Fix" section
    all_actions = []
    for tech in aggregated_report.get("techniques", []):
        all_actions.extend(tech.get("mitigation", []))
    
    # Deduplicate and take top 3
    top_3 = list(dict.fromkeys(all_actions))[:3]
    if not top_3:
        top_3 = ["Enable multi-factor authentication", "Review system logs", "Isolate affected hosts"]

    return {
        "executive_summary": summary_text,
        "risk_trend": "Increasing" if severity in ["High", "Critical"] else "Stable",
        "top_3_actions": top_3
    }

Writing prioritizer.py


In [39]:
%%writefile graph.py
"""
CTI PIPELINE — graph.py  PRODUCTION VERSION v2
================================================
Corrections vs v1 :
  FIX D-1 — dashboard_node : merge agrégat + premier rapport individuel
             Avant : dashboard recevait uniquement aggregated_report →
             generate_dashboard() ne trouvait pas section_4_ttps,
             behaviors, IOCs → "0 behaviors / 0 techniques / 0 IOCs"

  FIX D-2 — pdf_node : section_3_severity écrasée par les stats agrégées
             Avant : severity MEDIUM (1er rapport) au lieu de Critical (global)

CORRECTIONS héritées de v1 :
  FIX 1 — _source_message stocké dans cti_result (désynchronisation index)
  FIX 2 — pdf_node utilise l'agrégat
  FIX 3 — notify_node : bare except → except Exception
  FIX 4 — Checkpoint SQLite
  FIX 5 — thread_id configurable
  FIX 6 — sanitize_unicode sur executive_summary
  FIX 7 — aggregation_node : filtre global_score > 0 supprimé
"""

import re
import os
from typing import TypedDict, List, Optional
from datetime import datetime

from langgraph.graph import StateGraph, START, END

# ==============================
# LangSmith CONFIG
# ==============================
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["LANGSMITH_TRACING"]  = "true"
    os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGSMITH_API_KEY"]  = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGSMITH_PROJECT"]  = "cti_rag"
    os.environ["HF_HOME"]            = "/kaggle/working/hf_cache"
    print("✅ LangSmith enabled")
except Exception as e:
    print(f"⚠️ LangSmith disabled: {e}")

# ==============================
# IMPORTS
# ==============================
from cti_analysis_agent import analyze_message
from mitre_agent        import run_mitre_analysis
from aggregator         import aggregate_reports, build_report_metadata
from prioritizer        import prioritize_actions, generate_executive_summary
from dashboard          import generate_dashboard
from pdf_report         import generate_pdf_report
from web_search_agent   import main as main_websearch
from notify_agent       import main as main_notify
from ragas_worker       import start_worker, submit_for_evaluation

BASE = "/kaggle/working/"


# ==============================
# STATE
# ==============================
class PipelineState(TypedDict):
    messages:            List[str]
    cti_results:         List[dict]
    mitre_reports:       List[dict]
    aggregated_report:   dict
    prioritized_actions: List[dict]
    executive_summary:   dict
    dashboard_path:      str
    pdf_path:            str
    menaces_brutes:      List[dict]
    rapport_final:       dict
    alertes_envoyees:    int
    agent_logs:          List[dict]
    _thread_id:          Optional[str]


# ==============================
# UTILS
# ==============================
def sanitize_unicode(obj):
    """Remplace les tirets unicode (–, —, −) par des tirets ASCII pour FPDF."""
    if isinstance(obj, str):
        return re.sub(r"[\u2013\u2014\u2212]", "-", obj)
    if isinstance(obj, dict):
        return {k: sanitize_unicode(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_unicode(i) for i in obj]
    return obj


# ==============================
# NODE 0 — WEB SEARCH
# ==============================
def web_search_node(state: PipelineState) -> dict:
    print("\n🌐 [NODE 0] Web Search")
    try:
        fetched = main_websearch()
        if not isinstance(fetched, list):
            fetched = [str(fetched)]
    except Exception as e:
        print(f"⚠️ Web search error: {e}")
        fetched = []

    return {
        "messages":       state.get("messages", []) + fetched,
        "menaces_brutes": fetched,
    }


# ==============================
# NODE 1 — CTI ANALYSIS
# FIX 1 : stocker _source_message dans chaque résultat
# ==============================
def cti_analysis_node(state: PipelineState) -> dict:
    print("\n🧠 [NODE 1] CTI Analysis")

    messages  = state.get("messages", [])
    thread_id = state.get("_thread_id", "cti-run-001")
    results, logs = [], []

    for idx, msg in enumerate(messages):
        try:
            result = analyze_message(msg)

            # FIX 1 — conserver le message source dans le résultat
            result["_source_message"] = msg

            semantic = result.get("patterns", {}).get(
                "semantic_summary",
                str(result.get("threat_brief", ""))[:200]
            )

            logs.append({
                "agent":     "cti",
                "timestamp": datetime.now().isoformat(),
                "input":     msg,
                "output":    semantic,
                "tool_calls": [{"name": "analyze_message", "arguments": {"message": msg}}],
                "status":    "done",
            })

            results.append(result)

            # Soumettre pour évaluation RAGAS si thread ciblé
            submit_for_evaluation(
                job_id    = f"{thread_id}-msg-{idx}",
                cti_result = result,
                message   = msg,
                thread_id = thread_id,
            )

        except Exception as e:
            results.append({"error": str(e), "_source_message": msg})
            logs.append({
                "agent":     "cti",
                "timestamp": datetime.now().isoformat(),
                "input":     msg,
                "output":    str(e),
                "status":    "error",
            })

    return {
        "cti_results": results,
        "agent_logs":  state.get("agent_logs", []) + logs,
    }


# ==============================
# NODE 2 — MITRE
# FIX 1 : lire _source_message depuis le résultat, pas depuis messages[i]
# ==============================
def mitre_mapping_node(state: PipelineState) -> dict:
    print("\n🗺️ [NODE 2] MITRE Mapping")

    reports, logs = [], []
    valid = [r for r in state.get("cti_results", []) if "error" not in r]
    print(f"[MITRE NODE] {len(valid)} résultat(s) CTI valide(s)")

    for cti in valid:
        try:
            # FIX 1 — message source fiable, sans risque de désynchronisation
            msg_text = cti.get("_source_message", "")

            behaviors_raw = cti.get("behaviors", [])
            mitre_input   = cti.get("threat_brief", {}).get("mitre_input", {})
            print(f"[MITRE NODE] behaviors bruts  : {len(behaviors_raw)}")
            print(f"[MITRE NODE] behaviors fusion : {len(mitre_input.get('behaviors', []))}")

            mitre_res = run_mitre_analysis(cti)
            print(f"[MITRE NODE] status     : {mitre_res.get('status')}")
            print(f"[MITRE NODE] techniques : "
                  f"{mitre_res.get('techniques_mapped', {}).get('total_techniques', 0)}")

            rich = build_report_metadata(msg_text, cti, mitre_res)
            reports.append(rich)

            logs.append({
                "agent":     "mitre",
                "timestamp": datetime.now().isoformat(),
                "input":     str(cti)[:300],
                "output":    str(mitre_res.get("techniques_mapped", [])),
                "tool_calls": [{"name": "run_mitre_analysis", "arguments": {"cti": cti}}],
                "status":    "done",
            })

        except Exception as e:
            import traceback
            print(f"[MITRE NODE] ❌ {e}")
            print(traceback.format_exc())
            reports.append({"error": str(e)})

    return {
        "mitre_reports": reports,
        "agent_logs":    state.get("agent_logs", []) + logs,
    }


# ==============================
# NODE 3 — AGGREGATION
# FIX 7 : suppression du critère global_score > 0
# ==============================
def aggregation_node(state: PipelineState) -> dict:
    print("\n📊 [NODE 3] Aggregation")

    all_reports = state.get("mitre_reports", [])

    # FIX 7 — ne plus exclure les rapports avec global_score == 0
    valid = [
        r for r in all_reports
        if "error" not in r and r.get("report_metadata") is not None
    ]

    print(f"[AGG] {len(all_reports)} rapports reçus, {len(valid)} valides")

    if not valid:
        print("[AGG] ⚠️ Aucun rapport valide — vérifier MITRE mapping")
        return {
            "aggregated_report":   {"global_severity": "Unknown", "global_severity_score": 0.0},
            "prioritized_actions": [],
            "executive_summary":   {},
        }

    agg     = aggregate_reports(valid)
    actions = prioritize_actions(agg)
    summary = generate_executive_summary(agg)

    return {
        "aggregated_report":   agg,
        "prioritized_actions": actions,
        "executive_summary":   summary,
    }


# ==============================
# NODE 4 — DASHBOARD
# FIX D-1 : merger agrégat + premier rapport individuel
# ==============================
def dashboard_node(state: PipelineState) -> dict:
    print("\n📈 [NODE 4] Dashboard")

    agg     = state.get("aggregated_report", {})
    reports = state.get("mitre_reports", [])

    # FIX D-1 — le premier rapport individuel contient section_4_ttps,
    # behaviors, IOCs que generate_dashboard() cherche.
    # L'agrégat écrase les stats globales (severity, actor, etc.)
    first_valid = next((r for r in reports if "error" not in r), {})
    dashboard_data = {**first_valid, **agg}   # ← clé du fix

    result = generate_dashboard(
        dashboard_data,
        output=os.path.join(BASE, "dashboard.png"),
    )
    path = result.get("path", "") if isinstance(result, dict) else str(result)
    return {"dashboard_path": path}


# ==============================
# NODE 5 — PDF
# FIX 2 + FIX D-2 : rapport agrégé + section_3_severity corrigée
# ==============================
def pdf_node(state: PipelineState) -> dict:
    print("\n📄 [NODE 5] PDF")

    dash = state.get("dashboard_path", "")
    dash_path = dash.get("path", "") if isinstance(dash, dict) else dash

    agg     = state.get("aggregated_report", {})
    reports = state.get("mitre_reports", [])
    base    = next((r for r in reports if "error" not in r), {})

    # FIX D-2 — reconstruire section_3_severity depuis les stats globales
    # pour que le PDF affiche la severity agrégée (Critical) et non
    # la severity du 1er rapport individuel (Medium)
    agg_severity = {
        "global_score":        agg.get("global_severity_score", 0.0),
        "global_level":        agg.get("global_severity", "Unknown"),
        "chain_bonus_applied": 0.0,
        "overall_reasoning":   f"Aggregated from {len(reports)} report(s)",
        "distribution":        agg.get("statistics", {}).get("severity_distribution", {}),
        "environment_context": base.get("section_3_severity", {}).get(
                                   "environment_context", {}
                               ),
    }

    data = {
        **base,
        **{k: v for k, v in agg.items() if k not in ("report_metadata",)},
        "section_3_severity":  agg_severity,
        "dashboard_path":      dash_path,
        "executive_summary":   sanitize_unicode(state.get("executive_summary", {})),
    }

    output_path = os.path.join(BASE, "cti_report.pdf")
    generate_pdf_report(report_data=data, output_filepath=output_path)

    return {"pdf_path": output_path}


# ==============================
# NODE 6 — NOTIFY
# FIX 3 : bare except → except Exception
# ==============================
def notify_node(state: PipelineState) -> dict:
    print("\n📢 [NODE 6] Notify")

    try:
        rapport, alertes = main_notify(
            state.get("pdf_path", ""),
            state.get("executive_summary", {}),
        )
    except Exception as e:   # FIX 3 — ne plus capturer KeyboardInterrupt
        print(f"[NOTIFY] ❌ Erreur d'envoi : {e}")
        rapport, alertes = {"error": str(e)}, 0

    return {
        "rapport_final":    rapport,
        "alertes_envoyees": alertes,
    }


# ==============================
# GRAPH
# FIX 4 : checkpoint SQLite
# ==============================
def build_pipeline():
    g = StateGraph(PipelineState)

    g.add_node("web",    web_search_node)
    g.add_node("cti",    cti_analysis_node)
    g.add_node("mitre",  mitre_mapping_node)
    g.add_node("agg",    aggregation_node)
    g.add_node("dash",   dashboard_node)
    g.add_node("pdf",    pdf_node)
    g.add_node("notify", notify_node)

    g.add_edge(START,    "web")
    g.add_edge("web",    "cti")
    g.add_edge("cti",    "mitre")
    g.add_edge("mitre",  "agg")
    g.add_edge("agg",    "dash")
    g.add_edge("dash",   "pdf")
    g.add_edge("pdf",    "notify")
    g.add_edge("notify", END)

    # FIX 4 — checkpoint SQLite pour reprise automatique
    # SqliteSaver(conn) — API correcte pour LangGraph >= 0.2
    # from_conn_string() est dépréciée et ne crée pas les tables
    try:
        import sqlite3
        from langgraph.checkpoint.sqlite import SqliteSaver
        db_path      = os.path.join(BASE, "checkpoints.db")
        conn         = sqlite3.connect(db_path, check_same_thread=False)
        checkpointer = SqliteSaver(conn)
        print(f"✅ Checkpoint SQLite actif : {db_path}")
        return g.compile(checkpointer=checkpointer)
    except ImportError:
        print("⚠️ langgraph-checkpoint-sqlite manquant — sans checkpoint")
        print("   → pip install langgraph-checkpoint-sqlite")
        return g.compile()
    except Exception as e:
        print(f"⚠️ Checkpoint désactivé : {e}")
        return g.compile()


# ==============================
# RUN
# FIX 5 : thread_id configurable
# ==============================
def run_pipeline(messages=None, thread_id: str = "cti-run-001"):
    """
    Lance ou reprend le pipeline CTI.

    Args:
        messages  : liste de messages CTI. Si None, web_search_node
                    les récupère automatiquement.
        thread_id : identifiant pour le checkpointer SQLite ET pour
                    le filtrage RAGAS (doit correspondre à
                    RAGAS_TARGET_THREAD_ID dans ragas_worker.py).
    """
    pipeline = build_pipeline()
    start_worker()

    initial_state: PipelineState = {
        "messages":            messages or [],
        "cti_results":         [],
        "mitre_reports":       [],
        "aggregated_report":   {},
        "prioritized_actions": [],
        "executive_summary":   {},
        "dashboard_path":      "",
        "pdf_path":            "",
        "menaces_brutes":      [],
        "rapport_final":       {},
        "alertes_envoyees":    0,
        "agent_logs":          [],
        "_thread_id":          thread_id,
    }

    # FIX 5 — thread_id transmis au checkpointer
    config = {"configurable": {"thread_id": thread_id}}

    return pipeline.invoke(initial_state, config=config)

Writing graph.py


### `ragas_evaluator.py`

In [112]:
%%writefile /kaggle/working/ragas_evaluator.py
import json
import math
import asyncio
from pathlib import Path
from typing import Any, List, Optional

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from langchain_core.language_models import BaseLLM
from langchain_core.outputs import LLMResult, Generation
# AJOUTER ces imports
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from kaggle_secrets import UserSecretsClient
SCORES_FILE = Path("/kaggle/working/ragas_scores.jsonl")


class AsyncSafePipeline(BaseLLM):
    pipeline: Any
    model_name: str = "qwen-cti"

    class Config:
        arbitrary_types_allowed = True

    @property
    def _llm_type(self) -> str:
        return "async_safe_hf_pipeline"

    def _generate(self, prompts, stop=None, **kwargs):
        generations = []
        for prompt in prompts:
            try:
                output = self.pipeline(prompt, max_new_tokens=512, do_sample=False)
                if isinstance(output, list) and output:
                    text = output[0].get("generated_text", "")
                    if text.startswith(prompt):
                        text = text[len(prompt):].strip()
                else:
                    text = str(output)
            except Exception as e:
                text = f'{{"error": "{str(e)}"}}'
            generations.append([Generation(text=text)])
        return LLMResult(generations=generations)

    async def _agenerate(self, prompts, stop=None, **kwargs):
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(
            None, lambda: self._generate(prompts, stop, **kwargs)
        )

    def _call(self, prompt, stop=None, **kwargs):
        return self._generate([prompt], stop, **kwargs).generations[0][0].text


def build_sample(answer: dict, question: str, tool_calls=None) -> SingleTurnSample:
    if not isinstance(answer, dict):
        raise ValueError(f"build_sample expects dict, got {type(answer)}")

    response_text = (
        answer.get("patterns", {}).get("semantic_summary")
        or answer.get("threat_brief", {}).get("attack_narrative")
        or answer.get("attack_narrative", "")
        or str(answer)[:500]
    )

    rag_results = answer.get("rag_context", [])
    if isinstance(rag_results, dict):
        rag_results = rag_results.get("results", [])
    retrieved_contexts = [str(r) for r in rag_results if r] if rag_results else []
    if not retrieved_contexts:
        retrieved_contexts = ["No RAG context available"]

    brief    = answer.get("threat_brief", {})
    patterns = answer.get("patterns", {})
    entities = answer.get("entities", {})

    threat_class  = patterns.get("threat_classification", "UNKNOWN")
    is_malicious  = patterns.get("is_malicious", False)
    kill_chain    = brief.get("kill_chain_stages", [])
    malware       = entities.get("malware_names",     [])
    ransomware    = entities.get("ransomware_groups", [])
    apt           = entities.get("apt_groups",        [])
    likely_actor  = brief.get("actor_profile", {}).get("likely_actor", "unknown")
    sectors       = brief.get("environment_context", {}).get("target_sector", [])

    ref_parts = [f"Threat classification: {threat_class}.", f"Malicious: {is_malicious}."]
    if malware:                   ref_parts.append(f"Malware: {', '.join(malware)}.")
    if ransomware:                ref_parts.append(f"Ransomware: {', '.join(ransomware)}.")
    if apt:                       ref_parts.append(f"APT: {', '.join(apt)}.")
    if likely_actor != "unknown": ref_parts.append(f"Actor: {likely_actor}.")
    if sectors:                   ref_parts.append(f"Sectors: {', '.join(sectors)}.")
    if kill_chain:                ref_parts.append(f"Kill chain: {', '.join(kill_chain)}.")

    return SingleTurnSample(
        user_input         = question,
        response           = response_text,
        retrieved_contexts = retrieved_contexts,
        reference          = " ".join(ref_parts),
    )


_METRICS = [faithfulness, answer_relevancy, context_precision, context_recall]
_WEIGHTS = {"faithfulness": 0.35, "answer_relevancy": 0.30,
            "context_precision": 0.20, "context_recall": 0.15}


def _to_float(val) -> float:
    if isinstance(val, list):
        valid = []
        for v in val:
            try:
                f = float(v)
                if not math.isnan(f): valid.append(f)
            except (TypeError, ValueError): pass
        return round(sum(valid) / len(valid), 3) if valid else 0.0
    if val is None: return 0.0
    try:
        f = float(val)
        return round(f, 3) if not math.isnan(f) else 0.0
    except (TypeError, ValueError): return 0.0


def run_rag_evaluation(samples, llm, embeddings) -> dict:
    dataset = EvaluationDataset(samples=samples)

    # ── Charger les clés ──────────────────────────────────────────
    from kaggle_secrets import UserSecretsClient
    from langchain_groq import ChatGroq
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    secrets        = UserSecretsClient()
    GOOGLE_API_KEY_2 = secrets.get_secret("GOOGLE_API_KEY_2")
    GROQ_API_KEY     = secrets.get_secret("GROQ_API_KEY")
    
    # Gemini pour Faithfulness
    gemini_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
        model="gemini-flash-latest",
        google_api_key=GOOGLE_API_KEY_2,
        temperature=0
    ))
    
    # Groq pour les autres
    groq_llm = LangchainLLMWrapper(ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=GROQ_API_KEY,
        temperature=0
    ))
    # ── Groq LLM ─────────────────────────────────────────────────
    groq_llm = LangchainLLMWrapper(ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=GROQ_API_KEY,
        temperature=0
    ))
    
    # ── Embeddings Gemini ─────────────────────────────────────────  ← MANQUANT
    gemini_embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY_2
    ))
    
    faithfulness.llm            = gemini_llm   # ← Gemini
    faithfulness.strictness     = 1
    answer_relevancy.llm        = groq_llm     # ← Groq
    answer_relevancy.strictness = 1
    context_precision.llm       = groq_llm
    context_recall.llm          = groq_llm
    
    

    # ── Assigner embeddings à chaque métrique ────────────────────
    faithfulness.embeddings      = gemini_embeddings
    answer_relevancy.embeddings  = gemini_embeddings
    context_precision.embeddings = gemini_embeddings
    context_recall.embeddings    = gemini_embeddings

    run_config = RunConfig(max_retries=2, max_wait=60, timeout=120, max_workers=1)
    try:
        result = evaluate(dataset=dataset, metrics=_METRICS, run_config=run_config)
    except Exception as e:
        print(f"[RAGAS] ❌ evaluate() failed: {e}")
        return {"faithfulness": 0.0, "answer_relevancy": 0.0,
                "context_precision": 0.0, "context_recall": 0.0,
                "reliability_score": 0.0, "flags": ["EVALUATION_FAILED"], "error": str(e)}

    row = result.scores[0] if result.scores else {}
    f  = _to_float(row.get("faithfulness"))
    ar = _to_float(row.get("answer_relevancy"))
    cp = _to_float(row.get("context_precision"))
    cr = _to_float(row.get("context_recall"))

    reliability = (_WEIGHTS["faithfulness"] * f + _WEIGHTS["answer_relevancy"] * ar +
                   _WEIGHTS["context_precision"] * cp + _WEIGHTS["context_recall"] * cr)

    flags = []
    if f  < 0.50: flags.append("HIGH_HALLUCINATION_RISK")
    if ar < 0.50: flags.append("LOW_RELEVANCY")
    if cp < 0.50: flags.append("NOISY_RETRIEVAL")
    if cr < 0.50: flags.append("MISSING_CONTEXT")
    if reliability < 0.50: flags.append("UNRELIABLE_PIPELINE")

    return {"faithfulness": f, "answer_relevancy": ar, "context_precision": cp,
            "context_recall": cr, "reliability_score": round(reliability, 3), "flags": flags}

def save_scores(scores: dict) -> None:
    from datetime import datetime
    scores["timestamp"] = datetime.now().isoformat()
    with open(SCORES_FILE, "a") as f:
        f.write(json.dumps(scores) + "\n")


def load_scores() -> list:
    if not SCORES_FILE.exists():
        return []
    scores = []
    with open(SCORES_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                try:    scores.append(json.loads(line))
                except json.JSONDecodeError: pass
    return scores

Overwriting /kaggle/working/ragas_evaluator.py


In [61]:
!pip install langchain-google-genai --quiet

In [99]:
%%writefile /kaggle/working/ragas_worker.py
# # %%writefile /kaggle/working/ragas_worker.py

# # import threading
# # import queue
# # import json

# # from langchain_huggingface import HuggingFacePipeline
# # from langchain_community.embeddings import HuggingFaceEmbeddings

# # from llm_helper import get_llm
# # import threading

# # _eval_counter = 0
# # _eval_lock = threading.Lock()
# # try:
# #     from ragas_evaluator import build_sample, run_rag_evaluation, save_scores
# #     _ragas_available = True
# # except ImportError as e:
# #     print(f"[RAGAS] ⚠️ ragas_evaluator unavailable: {e}")
# #     _ragas_available = False
# # EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# # _eval_queue = queue.Queue()
# # _worker_started = False


# # # ================================================================
# # # Utilities
# # # ================================================================

# # def sanitize_json_payload(payload):
# #     """
# #     Nettoie toute sortie avant envoi à RAGAS :
# #     - supprime ```json ``` et ```
# #     - supprime le texte hors { ... }
# #     - safe : ne lève jamais
# #     """
# #     if not isinstance(payload, str):
# #         return payload

# #     text = payload.strip()

# #     # Supprimer markdown ```json
# #     if text.startswith("```"):
# #         parts = text.split("```")
# #         if len(parts) >= 2:
# #             text = parts[1].strip()

# #     # Extraire contenu JSON
# #     try:
# #         start = text.find("{")
# #         end = text.rfind("}") + 1
# #         if start != -1 and end != -1:
# #             return text[start:end]
# #     except Exception:
# #         pass

# #     return text


# # def extract_tool_calls(cti_result):
# #     """
# #     Extrait les tool_calls s'ils existent dans le résultat CTI.
# #     Compatible avec les logs LangGraph.
# #     """
# #     tool_calls = []

# #     # Cas direct
# #     if isinstance(cti_result, dict) and "tool_calls" in cti_result:
# #         return cti_result["tool_calls"]

# #     # Cas imbriqué (agent_logs)
# #     if isinstance(cti_result, dict):
# #         logs = cti_result.get("agent_logs", [])
# #         for log in logs:
# #             if "tool_calls" in log:
# #                 tool_calls.extend(log["tool_calls"])

# #     return tool_calls


# # # ================================================================
# # # Worker loop
# # # ================================================================

# # def _worker_loop():
# #     if not _ragas_available:
# #         print("[RAGAS] ⚠️ Worker stopped — ragas_evaluator not available")
# #         return

# #     pipe, _ = get_llm()
# #     llm = HuggingFacePipeline(pipeline=pipe)
# #     embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# #     while True:
# #         job = _eval_queue.get()
# #         if job is None:
# #             break

# #         try:
# #             # ✅ Nettoyage strict
# #             clean_message = sanitize_json_payload(job["message"])
# #             raw_result = job["cti_result"]

# #             # ✅ Forcer JSON → dict Python
# #             if isinstance(raw_result, str):
# #                 cleaned = sanitize_json_payload(raw_result)
# #                 clean_result = json.loads(cleaned)
# #             elif isinstance(raw_result, dict):
# #                 clean_result = raw_result
# #             else:
# #                 raise TypeError("cti_result must be dict or JSON string")

# #             # ✅ Extraction tool calls
# #             tool_calls = extract_tool_calls(clean_result)

# #             # ✅ Build RAGAS sample enrichi
# #             sample = build_sample(
# #                 answer=clean_result,
# #                 question=clean_message,
# #                 tool_calls=tool_calls,   # 🔑 clé manquante avant
# #             )

# #             scores = run_rag_evaluation([sample], llm, embeddings)
# #             scores["job_id"] = job["job_id"]

# #             save_scores(scores)

# #             print(f"\n[RAGAS] ✅ Job {job['job_id']}")
# #             print(f"  Tool accuracy : {scores.get('tool_call_accuracy')}")
# #             print(f"  Faithfulness  : {scores.get('faithfulness')}")
# #             print(f"  Relevancy     : {scores.get('answer_relevancy')}")
# #             print(f"  Ctx Precision : {scores.get('context_precision')}")
# #             print(f"  Ctx Recall    : {scores.get('context_recall')}")

# #         except Exception as e:
# #             print(f"[RAGAS] ❌ Job {job['job_id']} failed: {e}")

# #         finally:
# #             _eval_queue.task_done()


# # # ================================================================
# # # API
# # # ================================================================

# # def start_worker():
# #     global _worker_started
# #     if not _worker_started:
# #         t = threading.Thread(target=_worker_loop, daemon=True)
# #         t.start()
# #         _worker_started = True
# #         print("[RAGAS] ✅ Worker started")


# # def submit_for_evaluation(job_id: str, cti_result: dict, message: str):
# #     global _eval_counter

# #     is_critical = cti_result.get("is_malicious", False)

# #     with _eval_lock:
# #         _eval_counter += 1
# #         # Évaluer si : malveillant, OU 1er message, OU tous les 3
# #         should_eval = is_critical or _eval_counter == 1 or (_eval_counter % 3 == 0)

# #     if not should_eval:
# #         return

# #     _eval_queue.put({
# #         "job_id":     job_id,
# #         "cti_result": cti_result,
# #         "message":    message,
# #     })


# import queue
# import json

# from langchain_community.embeddings import HuggingFaceEmbeddings
# from llm_helper import get_llm

# try:
#     from ragas_evaluator import (
#         AsyncSafePipeline,
#         build_sample,
#         run_rag_evaluation,
#         save_scores,
#     )
#     _ragas_available = True
# except ImportError as e:
#     print(f"[RAGAS] ragas_evaluator unavailable: {e}")
#     _ragas_available = False

# EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# # ← Changer ici pour cibler un autre thread
# RAGAS_TARGET_THREAD_ID = "cti-run-001"

# # Queue de taille 1 : 1 seul job à la fois, les autres ignorés
# _eval_queue = queue.Queue(maxsize=1)

# _llm_singleton        = None
# _embeddings_singleton = None


# def _sanitize_payload(payload):
#     if not isinstance(payload, str):
#         return payload
#     text = payload.strip()
#     if text.startswith("```"):
#         parts = text.split("```")
#         if len(parts) >= 2:
#             text = parts[1].strip()
#     try:
#         start = text.find("{")
#         end   = text.rfind("}")
#         if start != -1 and end != -1:
#             return text[start:end + 1]
#     except Exception:
#         pass
#     return text


# def _extract_tool_calls(cti_result):
#     tool_calls = []
#     if isinstance(cti_result, dict) and "tool_calls" in cti_result:
#         return cti_result["tool_calls"]
#     if isinstance(cti_result, dict):
#         for log in cti_result.get("agent_logs", []):
#             if "tool_calls" in log:
#                 tool_calls.extend(log["tool_calls"])
#     return tool_calls


# def _get_models():
#     global _llm_singleton, _embeddings_singleton
#     if _llm_singleton is None:
#         pipe, _ = get_llm()
#         _llm_singleton        = AsyncSafePipeline(pipeline=pipe)
#         _embeddings_singleton = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
#         print("[RAGAS] Models loaded.")
#     return _llm_singleton, _embeddings_singleton


# def submit_for_evaluation(job_id: str, cti_result: dict, message: str,
#                           thread_id: str = ""):
#     """
#     Dépose dans la queue UNIQUEMENT si thread_id == RAGAS_TARGET_THREAD_ID.
#     Non bloquant — les autres threads CTI/MITRE ne sont jamais bloqués.
#     """
#     if not _ragas_available:
#         return
#     if thread_id != RAGAS_TARGET_THREAD_ID:
#         return  # thread non ciblé → ignoré silencieusement
#     try:
#         _eval_queue.put_nowait({
#             "job_id":     job_id,
#             "cti_result": cti_result,
#             "message":    message,
#             "thread_id":  thread_id,
#         })
#         print(f"[RAGAS] Job {job_id} queued for thread '{thread_id}'")
#     except queue.Full:
#         print(f"[RAGAS] Queue full — job {job_id} skipped")


# def evaluate_one_now() -> dict:
#     """
#     Consomme 1 seul job et l'évalue directement dans le thread notebook.
#     Appelé manuellement APRÈS run_pipeline() → pas de conflit asyncio.
#     """
#     if not _ragas_available:
#         return {"error": "ragas_not_available"}

#     print(f"[RAGAS] Waiting for job (target: '{RAGAS_TARGET_THREAD_ID}')...")
#     job = _eval_queue.get()  # bloquant jusqu'au job disponible

#     try:
#         raw = job["cti_result"]
#         cti_result = json.loads(_sanitize_payload(raw)) if isinstance(raw, str) else raw
#         message    = _sanitize_payload(job["message"])
#         tool_calls = _extract_tool_calls(cti_result)

#         print(f"[RAGAS] Evaluating job '{job['job_id']}' thread '{job.get('thread_id')}'...")

#         sample = build_sample(answer=cti_result, question=message, tool_calls=tool_calls)
#         llm, embeddings = _get_models()

#         scores = run_rag_evaluation([sample], llm, embeddings)
#         scores["job_id"]    = job["job_id"]
#         scores["thread_id"] = job.get("thread_id", "")
#         save_scores(scores)

#         print(f"[RAGAS] ✅ Done — job '{job['job_id']}'")
#         return scores

#     except Exception as e:
#         print(f"[RAGAS] ❌ Failed: {e}")
#         return {"error": str(e), "job_id": job.get("job_id", "?")}
#     finally:
#         _eval_queue.task_done()
"""
ragas_worker.py — PRODUCTION VERSION
======================================
CORRECTIONS vs version commentée :
  FIX W-1 : Import AsyncSafePipeline depuis ragas_evaluator (était manquant
             → ImportError → _ragas_available = False → "ragas_not_available")

  FIX W-2 : evaluate_one_now() est appelé SYNCHRONEMENT dans le thread
             notebook, après run_pipeline(). Pas de thread daemon asyncio —
             élimine IndexError: pop from an empty deque.

  FIX W-3 : submit_for_evaluation() filtre par thread_id == RAGAS_TARGET_THREAD_ID.
             Avec 1 seul message, _eval_counter==1, 1%3!=0 → le job était
             silencieusement ignoré. Désormais tout job du thread cible passe.

  FIX W-4 : Queue maxsize=1 — si le pipeline produit plusieurs CTI results,
             seul le dernier est conservé pour évaluation.
"""

import queue
import json

from langchain_community.embeddings import HuggingFaceEmbeddings
from llm_helper import get_llm

try:
    from ragas_evaluator import (
        AsyncSafePipeline,   # FIX W-1 — était manquant → ImportError
        build_sample,
        run_rag_evaluation,
        save_scores,
    )
    _ragas_available = True
    print("[RAGAS] ✅ ragas_evaluator imported successfully")
except ImportError as e:
    print(f"[RAGAS] ⚠️ ragas_evaluator unavailable: {e}")
    _ragas_available = False

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# Thread cible : seuls les jobs de ce thread sont évalués
RAGAS_TARGET_THREAD_ID = "cti-run-001"

# FIX W-4 — Queue de taille 1 : 1 seul job à la fois
_eval_queue = queue.Queue(maxsize=1)

# Singletons LLM/embeddings — chargés une seule fois au premier evaluate_one_now()
_llm_singleton        = None
_embeddings_singleton = None


# =============================================================================
# UTILITIES
# =============================================================================

def _sanitize_payload(payload):
    """Nettoie les backticks markdown que Qwen ajoute parfois en sortie."""
    if not isinstance(payload, str):
        return payload
    text = payload.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1].strip()
            # Enlever le label de langage (```json → "json\n{...")
            if "\n" in text:
                first_line = text.split("\n")[0].strip()
                if first_line in ("json", "python", ""):
                    text = text[len(first_line):].strip()
    try:
        start = text.find("{")
        end   = text.rfind("}")
        if start != -1 and end != -1:
            return text[start:end + 1]
    except Exception:
        pass
    return text


def _extract_tool_calls(cti_result: dict) -> list:
    """Extrait les tool_calls depuis un résultat CTI ou ses agent_logs."""
    tool_calls = []
    if isinstance(cti_result, dict) and "tool_calls" in cti_result:
        return cti_result["tool_calls"]
    if isinstance(cti_result, dict):
        for log in cti_result.get("agent_logs", []):
            if "tool_calls" in log:
                tool_calls.extend(log["tool_calls"])
    return tool_calls

# def _get_models():
#     global _llm_singleton, _embeddings_singleton
#     if _llm_singleton is None:
#         print("[RAGAS] Loading models (first call)...")

#         from kaggle_secrets import UserSecretsClient
#         from langchain_groq import ChatGroq
    #     from langchain_google_genai import GoogleGenerativeAIEmbeddings
    #     from ragas.llms import LangchainLLMWrapper
    #     from ragas.embeddings import LangchainEmbeddingsWrapper

    #     secrets = UserSecretsClient()
    #     GROQ_API_KEY   = secrets.get_secret("GROQ_API_KEY")
    #     GOOGLE_API_KEY = secrets.get_secret("GOOGLE_API_KEY")

    #     # LLM juge — Groq supporte multiple candidates
    #     _llm_singleton = LangchainLLMWrapper(ChatGroq(
    #         model="llama-3.3-70b-versatile",
    #         api_key=GROQ_API_KEY,
    #         temperature=0
    #     ))

    #     # Embeddings — Gemini (ta clé fonctionne pour les embeddings)
    #     _embeddings_singleton = LangchainEmbeddingsWrapper(
    #         GoogleGenerativeAIEmbeddings(
    #             model="models/gemini-embedding-001",
    #             google_api_key=GOOGLE_API_KEY
    #         )
    #     )

    #     print("[RAGAS] ✅ Models loaded.")
    # return _llm_singleton, _embeddings_singleton

def _get_models():
    global _llm_singleton, _embeddings_singleton
    if _llm_singleton is None:
        print("[RAGAS] Loading models (first call)...")
        # LLM et embeddings gérés directement dans run_rag_evaluation()
        _llm_singleton        = "gemini+groq"  # placeholder
        _embeddings_singleton = "gemini"        # placeholder
        print("[RAGAS] ✅ Models loaded.")
    return _llm_singleton, _embeddings_singleton
# =============================================================================
# FIX W-3 — submit_for_evaluation : filtre par thread_id, pas par compteur
# =============================================================================

def submit_for_evaluation(
    job_id:     str,
    cti_result: dict,
    message:    str,
    thread_id:  str = "",
) -> None:
    """
    Dépose un job dans la queue UNIQUEMENT si :
      - RAGAS est disponible
      - thread_id == RAGAS_TARGET_THREAD_ID

    Non bloquant — put_nowait() ignore si la queue est pleine.
    """
    if not _ragas_available:
        return

    if thread_id != RAGAS_TARGET_THREAD_ID:
        # Thread non ciblé — ignorer silencieusement
        return

    try:
        _eval_queue.put_nowait({
            "job_id":     job_id,
            "cti_result": cti_result,
            "message":    message,
            "thread_id":  thread_id,
        })
        print(f"[RAGAS] Job '{job_id}' queued (thread: '{thread_id}')")
    except queue.Full:
        # Remplacer l'ancien job par le nouveau
        try:
            _eval_queue.get_nowait()
        except queue.Empty:
            pass
        try:
            _eval_queue.put_nowait({
                "job_id":     job_id,
                "cti_result": cti_result,
                "message":    message,
                "thread_id":  thread_id,
            })
            print(f"[RAGAS] Job '{job_id}' replaced previous job in queue")
        except queue.Full:
            print(f"[RAGAS] ⚠️ Queue still full — job '{job_id}' skipped")


# =============================================================================
# FIX W-2 — evaluate_one_now : synchrone, appelé dans le thread notebook
# Pas de thread daemon → pas de conflit asyncio → pas de IndexError deque
# =============================================================================

def evaluate_one_now(timeout: float = 300.0) -> dict:
    """
    Consomme et évalue 1 job directement dans le thread courant.

    USAGE dans le notebook :
        state  = run_pipeline([...], thread_id="cti-run-001")
        scores = evaluate_one_now()   # appelé APRÈS run_pipeline()

    Args:
        timeout: secondes à attendre qu'un job soit disponible (défaut 5min)

    Returns:
        dict avec faithfulness, answer_relevancy, context_precision,
        context_recall, reliability_score, flags, job_id, thread_id
    """
    if not _ragas_available:
        return {"error": "ragas_not_available"}

    print(f"[RAGAS] Waiting for job (target thread: '{RAGAS_TARGET_THREAD_ID}')...")

    try:
        # Attente bloquante avec timeout
        job = _eval_queue.get(timeout=timeout)
    except queue.Empty:
        return {
            "error":    "timeout",
            "message":  f"No job received within {timeout}s. "
                        f"Check that submit_for_evaluation() was called with "
                        f"thread_id='{RAGAS_TARGET_THREAD_ID}'.",
        }

    try:
        # Nettoyage et désérialisation du résultat CTI
        raw = job["cti_result"]
        if isinstance(raw, str):
            cti_result = json.loads(_sanitize_payload(raw))
        elif isinstance(raw, dict):
            cti_result = raw
        else:
            raise TypeError(f"cti_result must be dict or JSON string, got {type(raw)}")

        message    = _sanitize_payload(str(job["message"]))
        tool_calls = _extract_tool_calls(cti_result)

        print(f"[RAGAS] Evaluating job '{job['job_id']}' (thread: '{job.get('thread_id')}')...")

        sample = build_sample(
            answer     = cti_result,
            question   = message,
            tool_calls = tool_calls,
        )

        llm, embeddings = _get_models()
        scores = run_rag_evaluation([sample], llm, embeddings)

        scores["job_id"]    = job["job_id"]
        scores["thread_id"] = job.get("thread_id", "")

        save_scores(scores)

        print(f"[RAGAS] ✅ Evaluation done — job '{job['job_id']}'")
        print(f"  Reliability   : {scores.get('reliability_score', 0):.3f}")
        print(f"  Faithfulness  : {scores.get('faithfulness', 0):.3f}")
        print(f"  Relevancy     : {scores.get('answer_relevancy', 0):.3f}")
        print(f"  Ctx Precision : {scores.get('context_precision', 0):.3f}")
        print(f"  Ctx Recall    : {scores.get('context_recall', 0):.3f}")
        if scores.get("flags"):
            print(f"  ⚠️  Flags      : {', '.join(scores['flags'])}")

        return scores

    except Exception as e:
        import traceback
        print(f"[RAGAS] ❌ Evaluation failed: {e}")
        print(traceback.format_exc())
        return {"error": str(e), "job_id": job.get("job_id", "?")}

    finally:
        _eval_queue.task_done()


# =============================================================================
# start_worker — stub pour compatibilité avec graph.py
# (le worker thread est désactivé — evaluate_one_now() remplace)
# =============================================================================

def start_worker() -> None:
    """
    Stub de compatibilité — ne démarre plus de thread daemon.
    L'évaluation RAGAS est maintenant synchrone via evaluate_one_now().
    Gardé pour éviter ImportError dans graph.py si appelé.
    """
    if not _ragas_available:
        print("[RAGAS] ⚠️ RAGAS not available — worker disabled")
        return
    print("[RAGAS] ✅ Worker ready (synchronous mode — call evaluate_one_now() after pipeline)")

Overwriting /kaggle/working/ragas_worker.py


In [86]:
!pip install langchain-groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.0 MB/s eta 0:00:00


### `ragas_agent_evaluator.py`

In [58]:
%%writefile /kaggle/working/ragas_agent_evaluator.py
from ragas.metrics import AgentGoalAccuracyWithoutReference, ToolCallAccuracy
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset
from ragas.messages import HumanMessage, AIMessage, ToolMessage, ToolCall
from ragas import evaluate
def _clean_llm_output(text: str) -> str:
    """Supprime les backticks markdown que Qwen ajoute parfois."""
    text = text.strip()
    if text.startswith("```"):
        # Supprimer ```json ... ``` ou ``` ... ```
        lines = text.split("\n")
        # Enlever première ligne (```json) et dernière (```)
        inner = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(inner).strip()
    return text
def evaluate_agent_run(state: dict, llm) -> dict:
    turns = []
    reference_tool_calls = []

    for log in state.get("agent_logs", []):
        if log.get("input"):
            turns.append(HumanMessage(content=log["input"]))

        if log.get("tool_name") and log.get("tool_args"):
            tool_call = ToolCall(
                name=log["tool_name"],
                args=log["tool_args"],
            )
            reference_tool_calls.append(tool_call)
            turns.append(AIMessage(
                content=f"Calling tool: {log['tool_name']}",
                tool_calls=[tool_call],
            ))

        if log.get("tool_output"):
            turns.append(ToolMessage(content=str(log["tool_output"])))

        if log.get("output"):
            # CORRECTION — nettoyer la sortie LLM avant de la passer à RAGAS
            clean_output = _clean_llm_output(str(log["output"]))
            turns.append(AIMessage(content=clean_output))

    if not turns:
        print("[RAGAS AGENT] No agent_logs found in state")
        return {}

    if not isinstance(turns[0], HumanMessage):
        turns.insert(0, HumanMessage(content="Analyze the following CTI message"))

    sample = MultiTurnSample(
        user_input=turns,
        reference=str(
            state.get("aggregated_report", {}).get("global_severity", "unknown")
        ),
        reference_tool_calls=reference_tool_calls if reference_tool_calls else None,
    )

    dataset = EvaluationDataset(samples=[sample])
    metrics = [AgentGoalAccuracyWithoutReference(llm=llm)]
    if reference_tool_calls:
        metrics.append(ToolCallAccuracy())

    try:
        results = evaluate(
        dataset=ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=evaluator_llm,
        run_config=RunConfig(max_workers=1)  # évite le rate limit Gemini
    )
    except Exception as e:
        print(f"[RAGAS AGENT] ❌ Evaluation failed: {e}")
        return {}

    scores = {}
    if result.scores and "agent_goal_accuracy_without_reference" in result.scores[0]:
        scores["agent_goal_accuracy"] = round(
            result.scores[0]["agent_goal_accuracy_without_reference"], 3
        )
    if result.scores and "tool_call_accuracy" in result.scores[0]:
        scores["tool_call_accuracy"] = round(
            result.scores[0]["tool_call_accuracy"], 3
        )

    print(f"\n[RAGAS AGENT] Results")
    for k, v in scores.items():
        print(f"  {k}: {v}")

    return scores


Overwriting /kaggle/working/ragas_agent_evaluator.py


### Installation complète des dépendances

In [43]:
# ── Cell 1 — run this EVERY session before anything else ─────────────────
import subprocess, sys

packages = [
    "feedparser",
    "beautifulsoup4",
    "fpdf2",
    "slowapi",
    "fastapi",
    "uvicorn[standard]",
    "pyngrok",
    "ragas",
    "datasets",
    "langchain-huggingface",
    "langsmith",
    "langchain",
    "langgraph",
    "langchain-community",
    "sentence-transformers",
    "faiss-cpu",
    "transformers",
    "httpx",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
✅ All packages installed


### Lancer le pipeline complet

In [52]:
# ── Forcer le rechargement des modules corrigés ──────────────────────────
import importlib, sys

# Supprimer le cache des anciens modules
for mod in ["ragas_evaluator", "ragas_worker", "graph"]:
    if mod in sys.modules:
        del sys.modules[mod]

import ragas_evaluator, ragas_worker, graph
importlib.reload(ragas_evaluator)
importlib.reload(ragas_worker)
importlib.reload(graph)

# ── Imports après reload ──────────────────────────────────────────────────
from graph           import run_pipeline
from ragas_evaluator import load_scores
from ragas_worker    import evaluate_one_now, RAGAS_TARGET_THREAD_ID

# ── 1. Pipeline ───────────────────────────────────────────────────────────
TARGET = RAGAS_TARGET_THREAD_ID  # "cti-run-001"

state = run_pipeline(
    messages  = ["LockBit 3.0 ransomware targeting European banks."],
    thread_id = TARGET,
)
print(f"\n✅ Pipeline done — severity: {state.get('aggregated_report', {}).get('global_severity', 'Unknown')}")

# ── 2. Vérifier que le job est bien en queue avant d'évaluer ─────────────
from ragas_worker import _eval_queue
print(f"\n[RAGAS] Jobs en queue : {_eval_queue.qsize()}")

if _eval_queue.empty():
    print("[RAGAS] ⚠️ Queue vide — resoumission manuelle depuis state...")
    from ragas_worker import submit_for_evaluation
    cti_results = state.get("cti_results", [])
    messages    = state.get("messages",    [])
    valid = [(r, m) for r, m in zip(cti_results, messages) if "error" not in r]
    if valid:
        cti_result, message = valid[0]
        submit_for_evaluation(
            job_id     = f"{TARGET}-manual",
            cti_result = cti_result,
            message    = message,
            thread_id  = TARGET,
        )
        print(f"[RAGAS] Jobs en queue après resoumission : {_eval_queue.qsize()}")
    else:
        print("[RAGAS] ❌ Aucun résultat CTI valide dans state")

# ── 3. Évaluation RAGAS ───────────────────────────────────────────────────
print(f"\n[RAGAS] Starting evaluation for thread '{TARGET}'...")
scores = evaluate_one_now(timeout=300)

# ── 4. Résultats ──────────────────────────────────────────────────────────
print("\n" + "="*48)
print("         RAGAS EVALUATION RESULTS")
print("="*48)
if "error" in scores:
    print(f"  ❌ Error : {scores['error']}")
    if scores.get("message"):
        print(f"  ℹ️  Info  : {scores['message']}")
else:
    print(f"  Thread        : {scores.get('thread_id', TARGET)}")
    print(f"  Job ID        : {scores.get('job_id', '?')}")
    print(f"  🎯 Reliability   : {scores.get('reliability_score', 0):.3f}")
    print(f"  ✅ Faithfulness  : {scores.get('faithfulness',      0):.3f}")
    print(f"  ✅ Relevancy     : {scores.get('answer_relevancy',  0):.3f}")
    print(f"  ✅ Ctx Precision : {scores.get('context_precision', 0):.3f}")
    print(f"  ✅ Ctx Recall    : {scores.get('context_recall',    0):.3f}")
    if scores.get("flags"):
        print(f"\n  ⚠️  FLAGS: {', '.join(scores['flags'])}")
print("="*48)

all_scores = load_scores()
print(f"\n[RAGAS] Total evaluations saved: {len(all_scores)}")

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


[RAGAS] ✅ ragas_evaluator imported successfully
✅ LangSmith enabled
[RAGAS] ✅ ragas_evaluator imported successfully
✅ LangSmith enabled
⚠️ langgraph-checkpoint-sqlite manquant — sans checkpoint
   → pip install langgraph-checkpoint-sqlite
[RAGAS] ✅ Worker ready (synchronous mode — call evaluate_one_now() after pipeline)

🌐 [NODE 0] Web Search
🌐 Web Search Agent running...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/tokenizer_config.js

[RAGAS] ✅ Models loaded.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
ERROR:ragas.executor:Exception raised in Job[0]: ValidationError(1 validation error for StatementGeneratorOutput
  Invalid JSON: trailing characters at line 6 column 3 [type=json_invalid, input_value=' {\n    "statements": [\...escaped when necessary.', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the doc

[RAGAS] ✅ Evaluation done — job 'cti-run-001-msg-6'
  Reliability   : 0.147
  Faithfulness  : 0.000
  Relevancy     : 0.490
  Ctx Precision : 0.000
  Ctx Recall    : 0.000
  ⚠️  Flags      : HIGH_HALLUCINATION_RISK, LOW_RELEVANCY, NOISY_RETRIEVAL, MISSING_CONTEXT, UNRELIABLE_PIPELINE

         RAGAS EVALUATION RESULTS
  Thread        : cti-run-001
  Job ID        : cti-run-001-msg-6
  🎯 Reliability   : 0.147
  ✅ Faithfulness  : 0.000
  ✅ Relevancy     : 0.490
  ✅ Ctx Precision : 0.000
  ✅ Ctx Recall    : 0.000

  ⚠️  FLAGS: HIGH_HALLUCINATION_RISK, LOW_RELEVANCY, NOISY_RETRIEVAL, MISSING_CONTEXT, UNRELIABLE_PIPELINE

[RAGAS] Total evaluations saved: 1


In [53]:
!grep -n "RAG_SCORE_THRESHOLD" /kaggle/working/threat_brief.py

47:RAG_SCORE_THRESHOLD = 0.8
159:        if score > RAG_SCORE_THRESHOLD:


In [113]:
# ── RAGAS uniquement — sans relancer le pipeline ─────────────────────────
import sys
for mod in ["ragas_evaluator", "ragas_worker"]:
    sys.modules.pop(mod, None)

from ragas_worker    import submit_for_evaluation, evaluate_one_now, RAGAS_TARGET_THREAD_ID, _eval_queue
from ragas_evaluator import load_scores

TARGET = RAGAS_TARGET_THREAD_ID

# Resoumission depuis state existant
cti_results = state.get("cti_results", [])
messages    = state.get("messages",    [])
valid = [(r, m) for r, m in zip(cti_results, messages) if "error" not in r]

print(f"[RAGAS] {len(valid)} résultat(s) CTI disponible(s) dans state")

if valid:
    cti_result, message = valid[0]
    submit_for_evaluation(
        job_id     = "ragas-only-test",
        cti_result = cti_result,
        message    = message,
        thread_id  = TARGET,
    )
    print(f"[RAGAS] Jobs en queue : {_eval_queue.qsize()}")

    scores = evaluate_one_now(timeout=300)

    print("\n" + "="*48)
    print("         RAGAS EVALUATION RESULTS")
    print("="*48)
    if "error" in scores:
        print(f"  ❌ Error : {scores['error']}")
    else:
        print(f"  🎯 Reliability   : {scores.get('reliability_score', 0):.3f}")
        print(f"  ✅ Faithfulness  : {scores.get('faithfulness',      0):.3f}")
        print(f"  ✅ Relevancy     : {scores.get('answer_relevancy',  0):.3f}")
        print(f"  ✅ Ctx Precision : {scores.get('context_precision', 0):.3f}")
        print(f"  ✅ Ctx Recall    : {scores.get('context_recall',    0):.3f}")
        if scores.get("flags"):
            print(f"\n  ⚠️  FLAGS: {', '.join(scores['flags'])}")
    print("="*48)

else:
    print("❌ state vide ou expiré — relancer run_pipeline() d'abord")

[RAGAS] ✅ ragas_evaluator imported successfully
[RAGAS] 7 résultat(s) CTI disponible(s) dans state
[RAGAS] Job 'ragas-only-test' queued (thread: 'cti-run-001')
[RAGAS] Jobs en queue : 1
[RAGAS] Waiting for job (target thread: 'cti-run-001')...
[RAGAS] Evaluating job 'ragas-only-test' (thread: 'cti-run-001')...
[RAGAS] Loading models (first call)...
[RAGAS] ✅ Models loaded.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[RAGAS] ✅ Evaluation done — job 'ragas-only-test'
  Reliability   : 0.633
  Faithfulness  : 0.500
  Relevancy     : 0.776
  Ctx Precision : 1.000
  Ctx Recall    : 0.167
  ⚠️  Flags      : MISSING_CONTEXT

         RAGAS EVALUATION RESULTS
  🎯 Reliability   : 0.633
  ✅ Faithfulness  : 0.500
  ✅ Relevancy     : 0.776
  ✅ Ctx Precision : 1.000
  ✅ Ctx Recall    : 0.167

  ⚠️  FLAGS: MISSING_CONTEXT


In [ ]:
!pip install langgraph-checkpoint-sqlite